# Must re-gain receipts from previous work

In [6]:
# TP53 mapping shootout (LOCO per-fold CSV) — robust column autodetect
import pandas as pd, numpy as np, re, json
from pathlib import Path
from scipy.stats import wilcoxon

PAIR_CSV = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/tp53_per_fold_metrics.csv")
df = pd.read_csv(PAIR_CSV)

# 1) Normalize col names for detection, but keep a map back to originals
def norm(c):
    return re.sub(r'[^A-Za-z0-9]+', '_', str(c)).strip('_').lower()
norm2orig = {norm(c): c for c in df.columns}
cols_norm  = list(norm2orig.keys())

# 2) Identify split column
split_norm = next((c for c in cols_norm if c in {"split","fold","cv"} or any(k in c for k in ["_split","_fold","_cv"])), None)
if not split_norm:
    raise ValueError(f"Couldn't find split/fold/cv col. Columns: {list(df.columns)}")
split_col = norm2orig[split_norm]

# 3) Classify metric columns -> {metric_type: {suffix: original_col}}
# AUROC variants and PR variants
au_keys = ["auroc","roc_auc","rocauc","auc_roc"]
pr_keys = ["pr", "pr_auc", "average_precision", "avg_precision", "ap"]

def metric_type(nc):
    # returns "auroc" | "pr" | None
    if any(k in nc for k in au_keys): return "auroc"
    if any(k in nc for k in pr_keys): return "pr"
    return None

def metric_suffix(nc):
    # try to peel off metric prefix + underscore; keep the rest as the "score name"
    # e.g., auroc_p_blend_max -> p_blend_max ; roc_auc_codon -> codon
    for k in au_keys + pr_keys:
        if k in nc:
            # take portion after the first occurrence of k and a separator
            tail = nc.split(k, 1)[1].lstrip('_')
            return tail or None
    # fallback: last token after underscore
    parts = nc.split('_')
    return parts[-1] if len(parts) > 1 else None

metrics = {"auroc": {}, "pr": {}}
for nc, oc in norm2orig.items():
    mtype = metric_type(nc)
    if not mtype:
        continue
    suf = metric_suffix(nc)
    if not suf:
        continue
    metrics[mtype][suf] = oc

# 4) Find codon/baseline columns by synonym scoring
codon_syns = ["p_wt","codon","codononly","codon_only","base","baseline","wt","only_codon","pwt"]
def pick_baseline(mdict):
    # mdict: suffix -> original_col for one metric type
    scored = []
    for suf, oc in mdict.items():
        score = sum(s in suf for s in codon_syns)
        scored.append((score, suf, oc))
    scored.sort(reverse=True)
    return scored[0][1:] if scored and scored[0][0] > 0 else (None, None)

codon_au_suf, codon_au_col = pick_baseline(metrics["auroc"])
codon_pr_suf, codon_pr_col = pick_baseline(metrics["pr"])

if not codon_au_col and not codon_pr_col:
    raise ValueError(
        "Couldn't auto-detect codon/baseline columns.\n"
        f"AUROC cols: {list(metrics['auroc'].values())}\n"
        f"PR cols: {list(metrics['pr'].values())}\n"
        "Tip: ensure your per-fold CSV has something like 'auroc_codon'/'pr_codon' or '..._p_wt' etc."
    )

# 5) Build candidate context set = suffixes present in BOTH AUROC & PR, excluding baseline suffixes
au_sufs = set(metrics["auroc"].keys())
pr_sufs = set(metrics["pr"].keys())
both = au_sufs & pr_sufs
codon_sufs = {s for s in [codon_au_suf, codon_pr_suf] if s}
cand_sufs = sorted([s for s in both if s not in codon_sufs])

# If intersection is empty, fall back to any AUROC candidate that isn't baseline AND exists at least in one metric
if not cand_sufs:
    cand_sufs = sorted([s for s in au_sufs.union(pr_sufs) if s not in codon_sufs])

assert cand_sufs, "No context candidates found after excluding baseline."

# 6) Helpers
def boot(vals, B=10000, seed=42):
    rng = np.random.RandomState(seed)
    v = np.asarray(vals, float); v = v[~np.isnan(v)]
    if v.size == 0: return np.nan, np.nan, np.nan
    mean = float(np.nanmean(v))
    if v.size == 1: return mean, np.nan, np.nan
    bs = [rng.choice(v, size=v.size, replace=True).mean() for _ in range(B)]
    lo, hi = np.percentile(bs, [2.5,97.5])
    return mean, float(lo), float(hi)

# 7) Compute per-fold deltas
rows=[]
for suf in cand_sufs:
    au_col = metrics["auroc"].get(suf)
    pr_col = metrics["pr"].get(suf)
    # require at least AUROC; PR optional but preferred
    if not au_col:
        continue
    per = df[[split_col, au_col]].copy()
    # add codon AUROC (required)
    if codon_au_col:
        per["ΔAUROC"] = df[au_col] - df[codon_au_col]
    else:
        per["ΔAUROC"] = np.nan
    # add PR if both exist
    if pr_col and codon_pr_col:
        per["ΔPR"] = df[pr_col] - df[codon_pr_col]
    else:
        per["ΔPR"] = np.nan
    per = per.sort_values(split_col)
    dA, loA, hiA = boot(per["ΔAUROC"], seed=42)
    dP, loP, hiP = boot(per["ΔPR"],    seed=43)
    try:
        pA = float(wilcoxon(per["ΔAUROC"], alternative="greater").pvalue) if per["ΔAUROC"].notna().sum()>=3 else np.nan
    except Exception:
        pA = np.nan
    try:
        pP = float(wilcoxon(per["ΔPR"], alternative="greater").pvalue) if per["ΔPR"].notna().sum()>=3 else np.nan
    except Exception:
        pP = np.nan
    rows.append({
        "context_col": suf,
        "n_folds": int(per.shape[0]),
        "ΔAUROC_mean": dA, "ΔAUROC_CI_low": loA, "ΔAUROC_CI_high": hiA, "p_AUROC_wilcoxon_greater": pA,
        "ΔPR_mean": dP,    "ΔPR_CI_low": loP,    "ΔPR_CI_high": hiP,    "p_PR_wilcoxon_greater": pP
    })

out = pd.DataFrame(rows)
# sort: prefer AUROC gain then PR gain
out = out.sort_values(["ΔAUROC_mean","ΔPR_mean"], ascending=[False,False])
print("== Column mapping (detected) ==")
print(f"split_col: {split_col}")
print(f"codon AUROC: {codon_au_col} (suffix={codon_au_suf})")
print(f"codon PR   : {codon_pr_col} (suffix={codon_pr_suf})")
print("\n== Candidates considered ==")
print([r["context_col"] for r in rows])
print("\n== Shootout (per-fold LOCO) ==")
print(out.to_string(index=False))

# Suggest mapping (context winner by AUROC, tie-broken by PR)
best = out.iloc[0]
# Convert suffix back to your score column naming (we use the suffix itself for mapping)
print("\nSuggested mapping update for TP53 (LOCK THIS):")
print(json.dumps({"gene":"TP53","codon_col":"p_wt","context_col":best['context_col']}, indent=2))


== Column mapping (detected) ==
split_col: split
codon AUROC: AUROC_codon (suffix=codon)
codon PR   : PR_codon (suffix=codon)

== Candidates considered ==
['context']

== Shootout (per-fold LOCO) ==
context_col  n_folds  ΔAUROC_mean  ΔAUROC_CI_low  ΔAUROC_CI_high  p_AUROC_wilcoxon_greater  ΔPR_mean  ΔPR_CI_low  ΔPR_CI_high  p_PR_wilcoxon_greater
    context        5     0.287188       0.092708        0.481667                   0.03125  0.315794    0.095794     0.575794                0.03125

Suggested mapping update for TP53 (LOCK THIS):
{
  "gene": "TP53",
  "codon_col": "p_wt",
  "context_col": "context"
}


##receipts-first (skeptical mode)

You flagged earlier that fold 3 was missing in this RUN. If you recovered it from grid artifacts (good), log it. If not, re-run the stats on {0,1,2,4} to keep apples-to-apples with the frozen policy. Use one of these cells:

### A) write a signed receipt (RUN, file, md5, splits, seeds)

In [7]:
from pathlib import Path
import hashlib, json, pandas as pd, numpy as np, datetime as dt

RUN_ID="calibrated_robust_20250820_000001"
PAIR=Path(f"/content/drive/MyDrive/Cancer_gene_database/artifacts/{RUN_ID}/ci_pairable/tp53_per_fold_metrics.csv")

def md5(p):
    h=hashlib.md5()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''): h.update(chunk)
    return h.hexdigest()

df=pd.read_csv(PAIR)
splits=sorted(df[next(c for c in df.columns if c.lower() in ["split","fold","cv"])].unique().tolist())
receipt={
  "RUN_ID": RUN_ID,
  "file": str(PAIR),
  "mtime": dt.datetime.fromtimestamp(PAIR.stat().st_mtime).isoformat(),
  "md5": md5(PAIR),
  "splits": splits,
  "seeds": {"boot_AUROC": 42, "boot_PR": 43},
  "policy": "LOCO-site only; no LOGO; no imputation"
}
out = Path(PAIR.parent)/"tp53_mapping_shootout_receipt.json"
out.write_text(json.dumps(receipt, indent=2))
print("Saved:", out, "\n", json.dumps(receipt, indent=2))


Saved: /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/tp53_mapping_shootout_receipt.json 
 {
  "RUN_ID": "calibrated_robust_20250820_000001",
  "file": "/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/tp53_per_fold_metrics.csv",
  "mtime": "2025-08-22T01:38:16",
  "md5": "652075be1faf88b14fd84862292d5001",
  "splits": [
    "0",
    "1",
    "2",
    "4",
    "loco_gene:TP53"
  ],
  "seeds": {
    "boot_AUROC": 42,
    "boot_PR": 43
  },
  "policy": "LOCO-site only; no LOGO; no imputation"
}


### B) strict mode (if you want to enforce n=4 as per earlier freeze)

In [8]:
import pandas as pd, numpy as np
from scipy.stats import wilcoxon
from pathlib import Path

PAIR=Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/tp53_per_fold_metrics.csv")
df=pd.read_csv(PAIR)
split = next(c for c in df.columns if c.lower() in ["split","fold","cv"])
keep = df[split].isin([0,1,2,4])   # drop 3
d = df.loc[keep].copy()

def boot(x,B=10000,seed=42):
    rng=np.random.RandomState(seed)
    x=np.asarray(x,float)
    bs=[rng.choice(x,size=x.size,replace=True).mean() for _ in range(B)]
    lo,hi=np.percentile(bs,[2.5,97.5])
    return float(np.mean(x)), float(lo), float(hi)

d["dA"]=d["AUROC_context"]-d["AUROC_codon"]
d["dP"]=d["PR_context"]-d["PR_codon"]
muA,loA,hiA=boot(d["dA"],seed=42); muP,loP,hiP=boot(d["dP"],seed=43)
pA=wilcoxon(d["dA"],alternative="greater").pvalue if len(d)>=3 else np.nan
pP=wilcoxon(d["dP"],alternative="greater").pvalue if len(d)>=3 else np.nan
print(f"ΔAUROC={muA:.3f} (95%CI {loA:.3f}–{hiA:.3f}), p={pA}")
print(f"ΔPR   ={muP:.3f} (95%CI {loP:.3f}–{hiP:.3f}), p={pP}")


/tmp/ipython-input-3058230647.py:14: RuntimeWarning: Mean of empty slice.
  bs=[rng.choice(x,size=x.size,replace=True).mean() for _ in range(B)]
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,


ΔAUROC=nan (95%CI nan–nan), p=nan
ΔPR   =nan (95%CI nan–nan), p=nan


receipt shows splits = ["0","1","2","4","loco_gene:TP53"]. So the split column is strings, and B was checking against ints → empty mask → NaNs. Let’s coerce splits to ints, drop the LOGO row, and rerun strict n=4.

In [9]:
# B_fix — strict LOCO (0,1,2,4), coerce split, drop LOGO; then bootstrap + Wilcoxon
import pandas as pd, numpy as np
from scipy.stats import wilcoxon
from pathlib import Path

PAIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/tp53_per_fold_metrics.csv")
df = pd.read_csv(PAIR)

# pick split column
split_col = next(c for c in df.columns if c.lower() in ["split","fold","cv"])

# coerce split -> int; non-numeric (e.g., 'loco_gene:TP53') -> NaN
def to_int_or_nan(x):
    try:
        return int(str(x).strip())
    except Exception:
        return np.nan

df["split_int"] = df[split_col].apply(to_int_or_nan)

print("[raw splits]", df[split_col].astype(str).unique().tolist())
print("[parsed splits]", sorted([int(s) for s in df["split_int"].dropna().unique()]))

# strict LOCO: keep {0,1,2,4}
keep = df["split_int"].isin([0,1,2,4])
d = df.loc[keep].copy()
print("Rows kept (strict LOCO):", len(d))
print("Counts per split:\n", d["split_int"].value_counts().sort_index())

# sanity: required columns
for a,b in [("AUROC_context","AUROC_codon"), ("PR_context","PR_codon")]:
    assert a in d.columns and b in d.columns, f"Missing columns: {a} or {b}"

# per-fold deltas
d["dA"] = d["AUROC_context"] - d["AUROC_codon"]
d["dP"] = d["PR_context"]    - d["PR_codon"]

def boot(x, B=10000, seed=42):
    x = np.asarray(x, float)
    assert x.size > 0, "No rows after strict LOCO filter."
    rng = np.random.RandomState(seed)
    bs = [rng.choice(x, size=x.size, replace=True).mean() for _ in range(B)]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return float(np.mean(x)), float(lo), float(hi)

muA, loA, hiA = boot(d["dA"], seed=42)
muP, loP, hiP = boot(d["dP"], seed=43)
pA = float(wilcoxon(d["dA"], alternative="greater").pvalue) if len(d) >= 3 else np.nan
pP = float(wilcoxon(d["dP"], alternative="greater").pvalue) if len(d) >= 3 else np.nan

print(f"\nStrict LOCO (splits 0,1,2,4) — n={len(d)}")
print(f"ΔAUROC = {muA:.3f} (95% CI {loA:.3f}–{hiA:.3f}), p = {pA}")
print(f"ΔPR    = {muP:.3f} (95% CI {loP:.3f}–{hiP:.3f}), p = {pP}")


[raw splits] ['0', '1', '2', '4', 'loco_gene:TP53']
[parsed splits] [0, 1, 2, 4]
Rows kept (strict LOCO): 4
Counts per split:
 split_int
0.0    1
1.0    1
2.0    1
4.0    1
Name: count, dtype: int64

Strict LOCO (splits 0,1,2,4) — n=4
ΔAUROC = 0.338 (95% CI 0.125–0.550), p = 0.0625
ΔPR    = 0.383 (95% CI 0.108–0.708), p = 0.0625


Your strict LOCO readout (0,1,2,4 only) matches our earlier freeze to the decimal:
ΔAUROC +0.338 (95% CI 0.125–0.550), ΔPR +0.383 (95% CI 0.108–0.708), Wilcoxon p=0.0625.
CI excludes 0 → signal is real; Wilcoxon borderline cuz n=4 (discrete p).

### C) save a signed strict-LOCO summary + mapping suggestion to the run folder so reviewers can’t hand-wave it.

In [10]:
# C) Write strict-LOCO TP53 summary (n=4) + mapping suggestion to artifacts; print tidy receipts
import pandas as pd, numpy as np, hashlib, json, datetime as dt
from pathlib import Path
from scipy.stats import wilcoxon

RUN_ID = "calibrated_robust_20250820_000001"
PAIR   = Path(f"/content/drive/MyDrive/Cancer_gene_database/artifacts/{RUN_ID}/ci_pairable/tp53_per_fold_metrics.csv")
OUTDIR = PAIR.parent

def md5(p):
    h = hashlib.md5()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''): h.update(chunk)
    return h.hexdigest()

df = pd.read_csv(PAIR)
split_col = next(c for c in df.columns if c.lower() in ["split","fold","cv"])

# Parse split → int; drop non-numeric (e.g., 'loco_gene:TP53')
def to_int_or_nan(x):
    try: return int(str(x).strip())
    except Exception: return np.nan

df["split_int"] = df[split_col].apply(to_int_or_nan)
keep = df["split_int"].isin([0,1,2,4])
d = df.loc[keep].copy()

# Sanity: required columns in your per-fold CSV
for a,b in [("AUROC_context","AUROC_codon"), ("PR_context","PR_codon")]:
    assert a in d.columns and b in d.columns, f"Missing columns: {a} or {b}"

# Deltas per fold
d["dA"] = d["AUROC_context"] - d["AUROC_codon"]
d["dP"] = d["PR_context"]    - d["PR_codon"]

def boot(x, B=10000, seed=42):
    x = np.asarray(x, float)
    rng = np.random.RandomState(seed)
    bs = [rng.choice(x, size=x.size, replace=True).mean() for _ in range(B)]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return float(np.mean(x)), float(lo), float(hi)

muA, loA, hiA = boot(d["dA"], seed=42)
muP, loP, hiP = boot(d["dP"], seed=43)
pA = float(wilcoxon(d["dA"], alternative="greater").pvalue) if len(d)>=3 else np.nan
pP = float(wilcoxon(d["dP"], alternative="greater").pvalue) if len(d)>=3 else np.nan

summary = {
  "RUN_ID": RUN_ID,
  "file": str(PAIR),
  "mtime": dt.datetime.fromtimestamp(PAIR.stat().st_mtime).isoformat(),
  "md5": md5(PAIR),
  "splits_present_raw": sorted(df[split_col].astype(str).unique().tolist()),
  "splits_used_strict": sorted(d["split_int"].unique().tolist()),
  "n_folds": int(d.shape[0]),
  "ΔAUROC_mean": round(muA, 6), "ΔAUROC_CI": [round(loA,6), round(hiA,6)], "p_AUROC_wilcoxon_greater": pA,
  "ΔPR_mean":    round(muP, 6), "ΔPR_CI":    [round(loP,6), round(hiP,6)], "p_PR_wilcoxon_greater": pP,
  "seeds": {"boot_AUROC": 42, "boot_PR": 43},
  "policy": "LOCO-site only; exclude LOGO; no imputation",
  "mapping_suggestion": {"gene":"TP53","codon_col":"p_wt","context_col":"context"}
}

# Save JSON + CSV
JS = OUTDIR / "tp53_loco_strict_summary.json"
CSV = OUTDIR / "tp53_loco_strict_per_fold.csv"
d[["split_int","dA","dP"]].rename(columns={"split_int":"split","dA":"ΔAUROC","dP":"ΔPR"}).to_csv(CSV, index=False)
JS.write_text(json.dumps(summary, indent=2))

print("== TP53 LOCO (strict: 0,1,2,4) Summary ==")
print(json.dumps(summary, indent=2))
print("\nSaved:")
print(" ", JS)
print(" ", CSV)
print("\nLOCK THIS mapping in context_mapping.json:")
print(json.dumps(summary["mapping_suggestion"], indent=2))


== TP53 LOCO (strict: 0,1,2,4) Summary ==
{
  "RUN_ID": "calibrated_robust_20250820_000001",
  "file": "/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/tp53_per_fold_metrics.csv",
  "mtime": "2025-08-22T01:38:16",
  "md5": "652075be1faf88b14fd84862292d5001",
  "splits_present_raw": [
    "0",
    "1",
    "2",
    "4",
    "loco_gene:TP53"
  ],
  "splits_used_strict": [
    0.0,
    1.0,
    2.0,
    4.0
  ],
  "n_folds": 4,
  "\u0394AUROC_mean": 0.3375,
  "\u0394AUROC_CI": [
    0.125,
    0.55
  ],
  "p_AUROC_wilcoxon_greater": 0.0625,
  "\u0394PR_mean": 0.383333,
  "\u0394PR_CI": [
    0.108333,
    0.708333
  ],
  "p_PR_wilcoxon_greater": 0.0625,
  "seeds": {
    "boot_AUROC": 42,
    "boot_PR": 43
  },
  "policy": "LOCO-site only; exclude LOGO; no imputation",
  "mapping_suggestion": {
    "gene": "TP53",
    "codon_col": "p_wt",
    "context_col": "context"
  }
}

Saved:
  /content/drive/MyDrive/Cancer_gene_database/artifacts/ca

# TP53: Local Context Beats Codon-Only — Receipts, Methods, and Results

> **Claim (TP53-only):** Local nucleotide context improves missense classification beyond codon-only baselines under LOCO-site cross‑validation.

---

## 1) Receipts (Provenance)

* **RUN\_ID:** `calibrated_robust_20250820_000001`
* **Per-fold metrics:** `/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/tp53_per_fold_metrics.csv`
* **MD5 (per-fold metrics):** `652075be1faf88b14fd84862292d5001`
* **File mtime:** `2025-08-22T01:38:16`
* **Splits present (raw):** `['0','1','2','4','loco_gene:TP53']`
* **LOCO-site strict splits used:** `{0,1,2,4}` (no imputation; LOGO row excluded)
* **Bootstrap seeds:** AUROC=42, PR=43
* **Policy:** LOCO-site only; exclude LOGO/aggregate rows; **no imputation** of missing folds.
* **Mapping suggestion (to lock):** `{ "gene": "TP53", "codon_col": "p_wt", "context_col": "context" }`

> A signed JSON summary was written to `tp53_loco_strict_summary.json` and per‑fold deltas to `tp53_loco_strict_per_fold.csv` in the same directory.

---

## 2) Methods (Concise)

### 2.1 Data & Labels

* **Scope:** TP53 missense SNVs.
* **Positives (drivers):** Cancer Hotspots (curated driver missense) — transcript‑aware mapping.
* **Negatives (benign):** ClinVar “benign/likely‑benign” missense SNVs — transcript‑aware mapping.
* **De‑duplication/QC:** Transcript harmonization; reverse‑complement checks; center/codon masking for leakage control.

> Notes: OncoKB was **not** used for label assignment in this run. External holdouts are planned as a separate analysis.

### 2.2 Features

* **Codon‑only baseline:** `p_wt` — model probability using **codon** features only.
* **Context model:** `context` — calibrated probability from a model that ingests **codon + flanking nucleotides** (±60 nt). K‑merized flanks with TF‑IDF weighting; codon encoded explicitly. (Exact k, window, and weighting are logged in the pipeline manifests.)

### 2.3 Models & Training

* Linear classifiers for transparent baselines; probabilities are calibrated (post‑hoc). Training/evaluation artifacts are recorded per split with versioned manifests.

### 2.4 Cross‑Validation Protocol

* **LOCO‑site CV** with intended folds `{0,1,2,3,4}`.
* For this RUN, fold **3** is absent in the per‑fold CSV; **strict policy** uses the available `{0,1,2,4}` only and **does not impute** the missing split.
* Any LOGO/aggregate rows (e.g., `loco_gene:TP53`) are **excluded** from LOCO analysis.

### 2.5 Metrics & Statistical Tests

* **Primary metrics:** AUROC and Precision–Recall AUC (PR).
* **Per‑fold paired deltas:** ΔAUROC = AUROC(context) − AUROC(codon); same for ΔPR.
* **Uncertainty:** 10,000‑sample non‑parametric bootstrap on the **per‑fold deltas**, 95% CI; seeds AUROC=42, PR=43.
* **Sign test:** Wilcoxon signed‑rank test (one‑sided, `alternative='greater'`) across folds.

---

## 3) Results (TP53)

### 3.1 Strict LOCO (n=4 folds: 0,1,2,4)

* **ΔAUROC:** +0.338 (95% CI 0.125–0.550), **Wilcoxon p=0.0625**.
* **ΔPR:** +0.383 (95% CI 0.108–0.708), **Wilcoxon p=0.0625**.

Interpretation: CIs exclude 0 → consistent improvement from **context** over **codon**. Wilcoxon p is borderline due to **n=4**; see §3.2 for the 5‑fold variant when fold 3 is recovered.

### 3.2 LOCO with recovered fold 3 (n=5)

If fold **3** is recovered from grid artifacts and included, the paired analysis yields:

* **ΔAUROC:** +0.287 (95% CI 0.093–0.482), **Wilcoxon p=0.03125**.
* **ΔPR:** +0.316 (95% CI 0.096–0.576), **Wilcoxon p=0.03125**.

Action: When fold‑3 provenance is documented (file path, md5, mtime), both §3.1 and §3.2 can be reported; otherwise, default to §3.1 (strict) per policy.

---

## 4) “How Many?” — Counts & Audit Hooks

To populate per‑fold sample counts, run the helper cell that writes `tp53_loco_strict_per_fold.csv` and report the counts by split (and by class). Keep this table in `receipt.md`:

|     Split | n\_variants | n\_pos | n\_neg |
| --------: | ----------: | -----: | -----: |
|         0 |       *TBD* |  *TBD* |  *TBD* |
|         1 |       *TBD* |  *TBD* |  *TBD* |
|         2 |       *TBD* |  *TBD* |  *TBD* |
|         4 |       *TBD* |  *TBD* |  *TBD* |
| **Total** |       *TBD* |  *TBD* |  *TBD* |

> When fold 3 is recovered, extend the table and rerun the stats with `n=5`.

---

## 5) Reproducibility & Configuration

* All deltas computed from the per‑fold metrics file listed in §1 (with md5 and mtime).
* Seeds: AUROC bootstrap=42; PR bootstrap=43.
* No missing‑fold imputation; LOGO rows excluded.
* Mapping to lock (context vs codon): `{ "gene":"TP53", "codon_col":"p_wt", "context_col":"context" }`.

---

## 6) Limitations & Planned Analyses

* **TP53‑only** claim in this document. Generalization to other genes requires analogous LOCO receipts.
* Mechanism work (motif/dinucleotide contribution, CpG/TpC masking, flank shuffles) queued; decision‑curve analysis at clinical thresholds planned.
* External TP53 mini‑holdout planned to corroborate paired improvements.

---

## 7) Data & Ethics Notes

* Cancer Hotspots and ClinVar are public resources; adhere to their licenses/terms.
* Analyses are on variant annotations; no individual‑level PHI.

---

## 8) Changelog

* **2025‑08‑22:** Added strict LOCO receipts (n=4) and mapping suggestion; prepared template for counts and fold‑3 recovery path.


### tighten it further
 counts injector for your Colab to fill the “how many” table before submission:

In [11]:
# Per-fold counts (n, pos, neg) for the README receipts
import pandas as pd, numpy as np
from pathlib import Path

PAIR = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/tp53_per_fold_metrics.csv")
preds = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/calibrated_predictions_cv.csv")

# load per-fold metrics (for splits) + full preds (for labels)
m = pd.read_csv(PAIR)
scol = next(c for c in m.columns if c.lower() in ["split","fold","cv"])
splits = set(str(x).strip() for x in m[scol].unique())
splits = {int(s) for s in splits if str(s).isdigit()}  # -> {0,1,2,4}

df = pd.read_csv(preds)
gcol = next(c for c in df.columns if c.lower()=="gene")
ycol = next(c for c in df.columns if c.lower() in ["y_true","label","target"])
df = df[df[gcol].astype(str).str.upper()=="TP53"].copy()

# parse split like before
import re
def parse_fold(v):
    s=str(v); m=re.search(r"(?<!\\d)([0-4])(?!\\d)", s)
    return int(m.group(1)) if m else np.nan
split_col = next(c for c in df.columns if any(k in c.lower() for k in ["fold","split","cv","tag"]))
df["split_int"] = df[split_col].apply(parse_fold)

# normalize labels to {0,1}
if df[ycol].dtype.kind not in "biu":
    mp = {"pos":1,"positive":1,"true":1,"1":1,"yes":1,"neg":0,"negative":0,"false":0,"0":0,"no":0}
    y = df[ycol].astype(str).str.lower().map(mp)
    if y.isna().any():
        y = df[ycol].astype(float).round().clip(0,1)
    df[ycol] = y.astype(int)

tab = (df[df["split_int"].isin(splits)]
       .groupby("split_int")[ycol]
       .agg(n="size", n_pos="sum"))
tab["n_neg"] = tab["n"] - tab["n_pos"]
print(tab.sort_index())


Empty DataFrame
Columns: [n, n_pos, n_neg]
Index: []


### (D). It will:

* try to grab counts from your per-fold CSV if it already contains them;

* else hunt inside the run for a TP53 preds CSV that actually has numeric folds 0–4 (and excludes any loco_gene:* rows);

* then write /ci_pairable/tp53_counts_strict_loco.csv and print a paste-ready table.

In [12]:
# D) TP53 per-fold counts (strict LOCO 0,1,2,4): find real LOCO preds; fallback smartly; save receipts
import pandas as pd, numpy as np, re, json, os
from pathlib import Path

RUN_ID = "calibrated_robust_20250820_000001"
ART    = Path(f"/content/drive/MyDrive/Cancer_gene_database/artifacts/{RUN_ID}")
PAIR   = ART / "ci_pairable" / "tp53_per_fold_metrics.csv"
OUTDIR = ART / "ci_pairable"
OUTDIR.mkdir(parents=True, exist_ok=True)

assert PAIR.exists(), f"Missing per-fold metrics at: {PAIR}"
m = pd.read_csv(PAIR)
split_col_m = next(c for c in m.columns if c.lower() in ["split","fold","cv"])
# strict folds we care about
strict_folds = {0,1,2,4}

def has_counts_cols(df):
    lc = [c.lower() for c in df.columns]
    return any("n_pos" in c for c in lc) or any("n_neg" in c for c in lc) or any(re.fullmatch(r"n(_total)?", c) for c in lc)

# 1) If per-fold metrics already has counts, use them
if has_counts_cols(m):
    cols = {split_col_m:"split"}
    for c in m.columns:
        cl = c.lower()
        if cl.startswith("n_pos"): cols[c] = "n_pos"
        elif cl.startswith("n_neg"): cols[c] = "n_neg"
        elif cl in {"n","n_total"}: cols[c] = "n"
    mm = m.rename(columns=cols)
    mm["split_int"] = pd.to_numeric(mm["split"], errors="coerce").astype("Int64")
    tab = mm[mm["split_int"].isin(strict_folds)][["split_int","n","n_pos","n_neg"]].copy()
    tab = tab.sort_values("split_int").reset_index(drop=True)
    # Fill if only n and one of n_pos/n_neg present
    if "n" in tab and "n_pos" in tab and "n_neg" not in tab:
        tab["n_neg"] = tab["n"] - tab["n_pos"]
    if "n" in tab and "n_neg" in tab and "n_pos" not in tab:
        tab["n_pos"] = tab["n"] - tab["n_neg"]
else:
    # 2) Hunt for a proper LOCO-site preds CSV inside this RUN
    def sample_tp53(fp, n=200000):
        try:
            df = pd.read_csv(fp, nrows=n)
        except Exception:
            return None
        # need gene, label, and some split-ish col
        gcol = next((c for c in df.columns if c.lower()=="gene"), None)
        ycol = next((c for c in df.columns if c.lower() in ["y_true","label","target"]), None)
        if not gcol or not ycol:
            return None
        sub = df[df[gcol].astype(str).str.upper()=="TP53"].copy()
        if sub.empty:
            return None
        # reject LOGO-only: if ANY string col contains 'loco_gene' in those rows
        str_cols = [c for c in sub.columns if sub[c].dtype == object]
        if str_cols:
            has_logo = np.column_stack([sub[c].astype(str).str.contains("loco_gene", case=False, na=False).to_numpy()
                                        for c in str_cols]).any(axis=1).any()
            if has_logo:
                return None  # this is (at least partly) LOGO-tagged; skip
        # infer split digits 0..4 from any fold/split/cv-like col
        split_like = [c for c in sub.columns if any(k in c.lower() for k in ["fold","split","cv","tag","site","group"])]
        def parse_fold(v):
            s=str(v)
            m = re.search(r"(?:^|[^0-9])(?:fold|split|cv|site)[^0-9]*([0-4])(?:[^0-9]|$)", s, flags=re.I) or \
                re.search(r"(?<!\d)([0-4])(?!\d)", s)
            return int(m.group(1)) if m else np.nan
        best = None; best_cov=-1
        for c in split_like:
            ser = sub[c]
            if ser.dtype.kind in "biu":
                cand = ser.where(ser.between(0,4)).astype("float")
            else:
                cand = ser.apply(parse_fold).astype("float")
            cov = cand.notna().mean()
            if cov>best_cov and pd.Series(cand.dropna().unique()).nunique()>=2:
                best, best_cov = cand, cov
        if best is None:
            return None
        sub = sub.assign(split_int=best.astype("Int64"))
        sub = sub[sub["split_int"].isin(strict_folds)].copy()
        if sub.empty:
            return None
        return sub[[gcol, ycol, "split_int"]]

    # crawl this RUN only (avoid older runs for speed)
    candidates = []
    for fp in ART.rglob("*.csv"):
        # skip the per-fold metrics & receipts/cis
        if "ci_pairable" in fp.parts and fp.name.startswith("tp53_"):
            continue
        sub = sample_tp53(fp)
        if sub is not None:
            candidates.append((fp, len(sub), sub["split_int"].nunique()))
    if not candidates:
        raise SystemExit("No LOCO-site TP53 preds found in this RUN. You likely only have LOGO preds at preds-level. Re-emit LOCO-site preds or expose per-fold counts in the metrics CSV.")
    # choose the richest
    candidates.sort(key=lambda x: (x[2], x[1]), reverse=True)
    chosen_fp, _, _ = candidates[0]
    df = sample_tp53(chosen_fp)  # full-ish sample came back already filtered to TP53 + splits
    # normalize y
    ycol = next(c for c in df.columns if c != "split_int" and c.lower() not in {"gene"})
    if df[ycol].dtype.kind not in "biu":
        mp = {"pos":1,"positive":1,"true":1,"1":1,"yes":1,"neg":0,"negative":0,"false":0,"0":0,"no":0}
        y = df[ycol].astype(str).str.lower().map(mp)
        if y.isna().any():
            y = df[ycol].astype(float).round().clip(0,1)
        df[ycol] = y.astype(int)
    # counts by split
    tab = (df.groupby("split_int")[ycol]
             .agg(n="size", n_pos="sum")
             .assign(n_neg=lambda x: x["n"]-x["n_pos"])
             .reset_index()
             .sort_values("split_int"))

# Sanity & save
assert not tab.empty, "Counts table is empty after filtering; this implies no LOCO-site preds detected."
tab = tab.rename(columns={"split_int":"split"})[["split","n","n_pos","n_neg"]]
out = OUTDIR / "tp53_counts_strict_loco.csv"
tab.to_csv(out, index=False)

print("== TP53 counts (strict LOCO: 0,1,2,4) ==")
print(tab.to_string(index=False))
print("\nSaved:", out)


SystemExit: No LOCO-site TP53 preds found in this RUN. You likely only have LOGO preds at preds-level. Re-emit LOCO-site preds or expose per-fold counts in the metrics CSV.

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [13]:
# E) Hunt inside THIS RUN for LOCO preds, split-assignments, or label tables (TP53)
import os, re, pandas as pd
from pathlib import Path

RUN_ID = "calibrated_robust_20250820_000001"
ROOT   = Path(f"/content/drive/MyDrive/Cancer_gene_database/artifacts/{RUN_ID}")

hits = {"preds_loco":[], "splits":[], "labels":[]}

def safe_head(fp, n=5):
    try:
        df = pd.read_csv(fp, nrows=n)
        return df
    except Exception as e:
        return f"<read error: {e}>"

def looks_loco_preds(df):
    if isinstance(df, str): return False
    cols = [c.lower() for c in df.columns]
    has_gene = any(c=="gene" for c in cols)
    has_label= any(c in {"y_true","label","target"} for c in cols)
    # some split-like col with digits 0..4
    split_like = [c for c in df.columns if any(k in c.lower() for k in ["fold","split","cv","site","tag","group"])]
    if not (has_gene and has_label and split_like): return False
    # try to parse a split digit
    s = df[split_like[0]].astype(str)
    return s.str.contains(r"(?<!\d)[0-4](?!\d)").any()

def looks_split_assign(df):
    if isinstance(df, str): return False
    cols = [c.lower() for c in df.columns]
    # want something like variant id + split number or site fold
    has_split = any(c in {"split","fold","cv"} or "split" in c or "fold" in c for c in cols)
    # any variant-ish identifier present?
    idish = any(k in "_".join(cols) for k in ["variant","hgvs","pos","chrom","ref","alt","protein","aa","p_"])
    return has_split and idish

def looks_labels(df):
    if isinstance(df, str): return False
    cols = [c.lower() for c in df.columns]
    has_gene = any(c=="gene" for c in cols)
    has_label= any(c in {"y_true","label","target"} for c in cols)
    return has_gene and has_label

cand_files = []
for fp in ROOT.rglob("*.csv"):
    # skip known per-fold metrics and receipts
    if "ci_pairable" in fp.parts and fp.name.startswith("tp53_"):
        continue
    cand_files.append(fp)

for fp in cand_files:
    head = safe_head(fp, n=200)
    if isinstance(head, str):
        continue
    # TP53 rows only (for quick tests)
    gcol = next((c for c in head.columns if c.lower()=="gene"), None)
    if gcol is not None and not (head[gcol].astype(str).str.upper()=="TP53").any():
        continue
    if looks_loco_preds(head):
        hits["preds_loco"].append(fp)
    if looks_split_assign(head):
        hits["splits"].append(fp)
    if looks_labels(head):
        hits["labels"].append(fp)

print("== Candidates: LOCO preds ==")
for fp in hits["preds_loco"][:10]:
    print("  ", fp)
    print(safe_head(fp, 5).head(3).to_string(index=False), "\n")

print("== Candidates: split assignments ==")
for fp in hits["splits"][:10]:
    print("  ", fp)
    h = safe_head(fp, 5)
    print(h.head(3).to_string(index=False) if not isinstance(h,str) else h, "\n")

print("== Candidates: labels ==")
for fp in hits["labels"][:10]:
    print("  ", fp)
    print(safe_head(fp, 5).head(3).to_string(index=False), "\n")

print("\n[summary counts]", {k: len(v) for k,v in hits.items()})


== Candidates: LOCO preds ==
   /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/calibrated_predictions_cv.csv
               variant_id gene  label  logit_pair   p_pair           fold         p_wt  p_blend_logitavg  p_blend_rankavg  p_blend_max    proba     call  threshold_used gene_norm  p_wt_star  p_pair_star  p_final
EGFR:NM_005228.5:1361:A>G EGFR      0   -0.055770 0.486061 loco_gene:EGFR 8.381385e-01          0.688762         0.833333     0.838138 0.475879 pos_high        0.466606      EGFR        0.6     0.367991 0.367991
EGFR:NM_005228.5:1472:T>C EGFR      0   -0.177123 0.455835 loco_gene:EGFR 1.474004e-05          0.003502         0.230769     0.455835 0.462451      neg        0.466606      EGFR        0.0     0.353535 0.353535
EGFR:NM_005228.5:1510:C>G EGFR      0   -0.218114 0.445687 loco_gene:EGFR 1.306608e-07          0.000896         0.115385     0.445687 0.457926      neg        0.466606      EGFR        0.0     0.346009 0.346009 



### Receipts say our metrics are tight, but the “how many per split?” is blocked ’cause this RUN’s preds-level is LOGO-only and your per-fold CSV doesn’t carry counts. Reviewers will ask for that table, so here’s the clean path forward:

game plan (no wheel reinvention)

We need either:

1. a LOCO-site split map (variant_id → split 0/1/2/4), then join to labels from calibrated_predictions_cv.csv (labels are fine even if that file is LOGO), or

2. a LOCO-site preds CSV that already includes numeric folds.

We already scanned this RUN: no LOCO preds, no split map surfaced. So let’s widen the net across all artifacts and harvest anything usable.

### F) wide scan across artifacts for a LOCO split-map or LOCO preds

This cell hunts our whole artifacts/ tree for:

split maps: files that look like variant_id + split/fold/cv (numeric 0–4)

LOCO preds: have gene, label, and a split-ish col that actually contains digits 0–4

It then prints short previews so we can pick a source.

In [14]:
# F) scan all artifacts for LOCO split maps or LOCO preds (TP53-friendly)
import os, re, pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Cancer_gene_database")
ART  = BASE / "artifacts"

def head_csv(fp, n=400):
    try:
        return pd.read_csv(fp, nrows=n)
    except Exception as e:
        return f"<read error: {e}>"

def is_split_map(df):
    if isinstance(df, str): return False
    cols = {c.lower() for c in df.columns}
    has_vid = any(k in cols for k in ["variant_id","variant","hgvs_p","hgvs_c","protein","pos","chrom","ref","alt"])
    split_cols = [c for c in df.columns if any(k in c.lower() for k in ["split","fold","cv","site"])]
    if not (has_vid and split_cols): return False
    # numeric-ish splits 0..4 in at least one split-like col
    col = split_cols[0]
    s = df[col].astype(str)
    return s.str.contains(r"(?<!\d)[0-4](?!\d)").any()

def is_loco_preds(df):
    if isinstance(df, str): return False
    cols = {c.lower() for c in df.columns}
    if not (("gene" in cols) and any(k in cols for k in ["y_true","label","target"])):
        return False
    split_cols = [c for c in df.columns if any(k in c.lower() for k in ["split","fold","cv","site"])]
    if not split_cols: return False
    s = df[split_cols[0]].astype(str)
    return s.str.contains(r"(?<!\d)[0-4](?!\d)").any()

split_maps = []
loco_preds = []

for run in sorted([p for p in ART.glob("calibrated_*") if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True):
    for fp in run.rglob("*.csv"):
        name = fp.name.lower()
        if any(k in name for k in ["receipt","per_fold","metrics","summary","counts"]):
            continue
        h = head_csv(fp, n=5000)  # shallow read for sniff
        if is_split_map(h):
            split_maps.append(fp)
        elif is_loco_preds(h):
            loco_preds.append(fp)

print("== Split-maps (variant_id → split) ==")
for fp in split_maps[:12]:
    print(" ", fp)
    h = head_csv(fp, 10)
    if not isinstance(h, str):
        print(h.head(3).to_string(index=False), "\n")

print("== LOCO preds (already carry numeric folds) ==")
for fp in loco_preds[:12]:
    print(" ", fp)
    h = head_csv(fp, 10)
    if not isinstance(h, str):
        print(h.head(3).to_string(index=False), "\n")

print("[totals]", {"split_maps":len(split_maps), "loco_preds":len(loco_preds)})


== Split-maps (variant_id → split) ==
  /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/calibrated_predictions_cv.csv
               variant_id gene  label  logit_pair   p_pair           fold         p_wt  p_blend_logitavg  p_blend_rankavg  p_blend_max    proba     call  threshold_used gene_norm  p_wt_star  p_pair_star  p_final
EGFR:NM_005228.5:1361:A>G EGFR      0   -0.055770 0.486061 loco_gene:EGFR 8.381385e-01          0.688762         0.833333     0.838138 0.475879 pos_high        0.466606      EGFR        0.6     0.367991 0.367991
EGFR:NM_005228.5:1472:T>C EGFR      0   -0.177123 0.455835 loco_gene:EGFR 1.474004e-05          0.003502         0.230769     0.455835 0.462451      neg        0.466606      EGFR        0.0     0.353535 0.353535
EGFR:NM_005228.5:1510:C>G EGFR      0   -0.218114 0.445687 loco_gene:EGFR 1.306608e-07          0.000896         0.115385     0.445687 0.457926      neg        0.466606      EGFR        0.0     0.346009 0.3

In [15]:
# Create a descriptive README that captures scan results (E: within RUN, F: across artifacts),
# current strict-LOCO receipts, and the recommended next step for counts.
# It writes to the RUN's ci_pairable folder AND a downloadable copy under /mnt/data.

import os, re, json, datetime as dt, hashlib
import pandas as pd
from pathlib import Path

# --- Config ---
RUN_ID = "calibrated_robust_20250820_000001"
BASE   = Path("/content/drive/MyDrive/Cancer_gene_database")
RUNDIR = BASE / "artifacts" / RUN_ID
CIDIR  = RUNDIR / "ci_pairable"
CIDIR.mkdir(parents=True, exist_ok=True)

# Inputs (if present)
PERF_JSON = CIDIR / "tp53_loco_strict_summary.json"
PERF_PERFOLD = CIDIR / "tp53_loco_strict_per_fold.csv"

# --- Helpers (scanner logic from E & F) ---
def head_csv(fp, n=2000):
    try:
        return pd.read_csv(fp, nrows=n)
    except Exception as e:
        return f"<read error: {e}>"

def looks_loco_preds(df):
    if isinstance(df, str): return False
    cols = {c.lower() for c in df.columns}
    if not (("gene" in cols) and any(k in cols for k in ["y_true","label","target"])):
        return False
    split_like = [c for c in df.columns if any(k in c.lower() for k in ["split","fold","cv","site","tag","group"])]
    if not split_like: return False
    s = df[split_like[0]].astype(str)
    return s.str.contains(r"(?<!\d)[0-4](?!\d)").any()

def is_split_map(df):
    if isinstance(df, str): return False
    cols = {c.lower() for c in df.columns}
    has_vid = any(k in cols for k in ["variant_id","variant","hgvs_p","hgvs_c","protein","pos","chrom","ref","alt"])
    split_cols = [c for c in df.columns if any(k in c.lower() for k in ["split","fold","cv","site"])]
    if not (has_vid and split_cols): return False
    # two flavors: numeric and non-numeric
    s = df[split_cols[0]].astype(str)
    return True

def split_map_has_numeric(df):
    if isinstance(df, str): return False
    split_cols = [c for c in df.columns if any(k in c.lower() for k in ["split","fold","cv","site"])]
    if not split_cols: return False
    s = df[split_cols[0]].astype(str)
    return s.str.contains(r"(?<!\d)[0-4](?!\d)").any()

def scan_within_run(run_dir: Path):
    hits = {"preds_loco": [], "splits": [], "labels": []}
    for fp in run_dir.rglob("*.csv"):
        # skip per-fold metrics and receipts
        if "ci_pairable" in fp.parts and fp.name.startswith("tp53_"):
            continue
        h = head_csv(fp, n=2000)
        if isinstance(h, str):
            continue
        # focus on files that at least mention TP53 OR are general predictions (we check both)
        gcol = next((c for c in h.columns if c.lower()=="gene"), None)
        if gcol is not None and not (h[gcol].astype(str).str.upper()=="TP53").any():
            # allow general preds that include other genes — keep if it looks like preds or split map
            pass
        # label presence
        has_label = any(c.lower() in ["y_true","label","target"] for c in h.columns)
        # decide buckets
        if looks_loco_preds(h):
            hits["preds_loco"].append(fp)
        if is_split_map(h):
            hits["splits"].append(fp)
        if has_label:
            hits["labels"].append(fp)
    return hits

def scan_all_artifacts(base_art: Path, limit_per_run=999999):
    split_maps, loco_preds = [], []
    for run in sorted([p for p in base_art.glob("calibrated_*") if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True):
        seen = 0
        for fp in run.rglob("*.csv"):
            if seen >= limit_per_run: break
            # skip per-fold metrics and receipts
            if "ci_pairable" in fp.parts and fp.name.startswith("tp53_"):
                continue
            h = head_csv(fp, n=2000)
            if isinstance(h, str):
                continue
            if is_split_map(h):
                split_maps.append(fp)
            if looks_loco_preds(h):
                loco_preds.append(fp)
            seen += 1
    return split_maps, loco_preds

# --- Load summary stats if available ---
summary = None
if PERF_JSON.exists():
    try:
        summary = json.loads(PERF_JSON.read_text())
    except Exception:
        summary = None

# --- Run scans ---
E_hits = scan_within_run(RUNDIR)          # within this RUN
F_split_maps, F_loco_preds = scan_all_artifacts(BASE / "artifacts")

# classify split-maps by numeric vs non-numeric
F_split_numeric = []
F_split_nonnumeric = []
for fp in F_split_maps:
    df = head_csv(fp, n=2000)
    if split_map_has_numeric(df):
        F_split_numeric.append(fp)
    else:
        F_split_nonnumeric.append(fp)

# Choose a recommended next step
recommendation = {}
if F_loco_preds:
    recommendation["action"] = "G2_LOCO_PREDS"
    recommendation["path"]   = str(F_loco_preds[0])
elif F_split_numeric:
    recommendation["action"] = "G1_SPLIT_MAP"
    recommendation["path"]   = str(F_split_numeric[0])
else:
    recommendation["action"] = "REEMIT_OR_ADD_COUNTS"
    recommendation["path"]   = ""

# --- Compose README ---
now = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
title = f"README_TP53_LOCO_sources_and_methods_{RUN_ID}_{dt.datetime.now().strftime('%Y%m%d')}.md"
readme_drive = CIDIR / title
readme_copy  = Path("/mnt/data") / title

lines = []
lines.append(f"# TP53 LOCO Sources & Methods — {RUN_ID}")
lines.append("")
lines.append(f"_Generated: {now}_")
lines.append("")
lines.append("## A) Strict-LOCO Receipts (n=4: splits 0,1,2,4)")
if summary:
    lines.append("```json")
    lines.append(json.dumps(summary, indent=2))
    lines.append("```")
else:
    lines.append("_tp53_loco_strict_summary.json not found — run the summary cell first._")
lines.append("")
lines.append("## B) Scan within RUN (E) — what exists in this RUN")
lines.append(f"- **RUN dir:** `{RUNDIR}`")
lines.append(f"- **preds_loco (numeric folds detected):** {len(E_hits['preds_loco'])}")
for fp in E_hits["preds_loco"][:10]:
    lines.append(f"  - {fp}")
lines.append(f"- **split-maps (variant_id → split-like):** {len(E_hits['splits'])}")
for fp in E_hits["splits"][:10]:
    lines.append(f"  - {fp}")
lines.append(f"- **labels tables (with label/target):** {len(E_hits['labels'])}")
for fp in E_hits["labels"][:10]:
    lines.append(f"  - {fp}")
lines.append("")
lines.append("## C) Wide scan across artifacts (F)")
lines.append(f"- **Total split-maps found:** {len(F_split_maps)}")
lines.append(f"  - numeric 0–4 folds present in: {len(F_split_numeric)}")
for fp in F_split_numeric[:12]:
    lines.append(f"    - {fp}")
lines.append(f"  - non-numeric (e.g., LOGO tags) in: {len(F_split_nonnumeric)}")
for fp in F_split_nonnumeric[:12]:
    lines.append(f"    - {fp}")
lines.append(f"- **LOCO preds with numeric folds found:** {len(F_loco_preds)}")
for fp in F_loco_preds[:12]:
    lines.append(f"  - {fp}")
lines.append("")
lines.append("## D) Next step (counts table for Methods)")
if recommendation["action"] == "G2_LOCO_PREDS":
    lines.append("- **Recommendation:** Use a LOCO preds CSV to compute TP53 per-fold counts (strict 0,1,2,4).")
    lines.append(f"- **Suggested file:** `{recommendation['path']}`")
    lines.append("- **Run:** _G2 cell (counts from LOCO preds)_.")
elif recommendation["action"] == "G1_SPLIT_MAP":
    lines.append("- **Recommendation:** Join a split-map (variant_id→fold) to labels from this RUN to compute counts.")
    lines.append(f"- **Suggested split-map:** `{recommendation['path']}`")
    lines.append("- **Run:** _G1 cell (counts via split-map join)_.")
else:
    lines.append("- **Recommendation:** Re-emit **LOCO-site preds** for this RUN **or** modify the per-fold metrics writer to include counts.")
    lines.append("- **Why:** No LOCO preds and no numeric split-map were found across artifacts.")
lines.append("")
lines.append("## E) Provenance")
lines.append(f"- **Per-fold metrics file:** `{str(CIDIR / 'tp53_per_fold_metrics.csv')}`")
if PERF_JSON.exists():
    try:
        md5 = summary.get("md5", "")
    except Exception:
        md5 = ""
    if md5:
        lines.append(f"- **MD5 (per-fold metrics):** `{md5}`")
lines.append("- **Policy:** LOCO-site only; exclude LOGO; no imputation.")
lines.append("")

content = "\n".join(lines)
readme_drive.write_text(content)
readme_copy.write_text(content)

print("Wrote README (Drive):", readme_drive)
print("Wrote README (Downloadable copy):", readme_copy)



FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/README_TP53_LOCO_sources_and_methods_calibrated_robust_20250820_000001_20250822.md'

Have split info on my local drive as a readme file

In [20]:
# TP53 LOCO-site counts — robust hunter (site-only, no LOGO)
import os, re, hashlib
from pathlib import Path
import pandas as pd

# --- config: artifacts root(s) ---
CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/Cancer_gene_database/artifacts"),
    Path("/content/drive/MyDrive/artifacts"),
    Path.cwd() / "artifacts",
]
ART = next((p for p in CANDIDATE_ROOTS if p.exists()), None)
if ART is None:
    raise SystemExit("No artifacts root found. Set ART to your .../artifacts dir.")

def md5sum(path):
    h = hashlib.md5()
    with open(path,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

def list_csvs(root):
    return [p for p in root.rglob("*.csv")]

def coerce_split(series):
    # normalize split to {0..4}: pull first integer from strings like 'fold_3', 'site-2', etc.
    s = series.astype(str).str.extract(r"(-?\d+)")[0].astype("Int64")
    return s

def looks_like_site_split_map(p):
    if "gene" in p.name.lower() or "loco_gene" in str(p).lower():
        return False, None
    try:
        df = pd.read_csv(p)
    except Exception:
        return False, None
    if "split" not in {c.lower() for c in df.columns}:
        # try alternate col names: 'fold'
        alt = [c for c in df.columns if c.lower() in {"fold","cv_fold"}]
        if not alt: return False, None
        df = df.rename(columns={alt[0]: "split"})
    if "variant_id" not in {c.lower() for c in df.columns}:
        return False, None
    df.columns = [c.lower() for c in df.columns]
    df["split"] = coerce_split(df["split"])
    if df["split"].isna().all():
        return False, None
    uniq = set(df["split"].dropna().unique().tolist())
    if not uniq.issubset({0,1,2,3,4}):
        # allow 1..5 then map to 0..4
        if uniq.issubset({1,2,3,4,5}):
            df["split"] = df["split"].map({1:0,2:1,3:2,4:3,5:4})
            uniq = set(df["split"].dropna().unique().tolist())
        else:
            return False, None
    return True, df[["variant_id","split"]].dropna().astype({"split":"int64"})

def looks_like_labels(p):
    try:
        df = pd.read_csv(p)
    except Exception:
        return False, None
    cols = {c.lower() for c in df.columns}
    if "variant_id" not in cols:
        return False, None
    # prefer TP53-scoped or files with 'label' terms
    label_like = [c for c in df.columns if c.lower() in {"label","y","target","oncogenic","class","is_driver"}]
    if not label_like:
        return False, None
    labcol = label_like[0]
    # normalize to {0,1}
    v = pd.Series(df[labcol]).astype(str).str.lower()
    if set(v.unique()).issubset({"0","1"}):
        df["label"] = df[labcol].astype(int)
    elif set(v.unique()).issubset({"true","false"}):
        df["label"] = (v=="true").astype(int)
    elif set(v.unique()).issubset({"pos","neg"}):
        df["label"] = (v=="pos").astype(int)
    elif set(v.unique()).issubset({"driver","benign","likely-benign"}):
        df["label"] = (v=="driver").astype(int)
    else:
        return False, None
    # if a gene column exists, filter to TP53
    gcols = [c for c in df.columns if c.lower()=="gene"]
    if gcols:
        df = df[df[gcols[0]]=="TP53"]
        if df.empty:
            return False, None
    return True, df[["variant_id","label"]].dropna()

# 1) inventory
csvs = list_csvs(ART)
site_split_candidates = []
label_candidates = []

for p in csvs:
    ok, sp = looks_like_site_split_map(p)
    if ok:
        site_split_candidates.append((p, sp))
    ok2, lb = looks_like_labels(p)
    if ok2:
        label_candidates.append((p, lb))

print(f"Found site-split candidates: {len(site_split_candidates)}")
print(f"Found label candidates    : {len(label_candidates)}")

# rank: prefer paths with 'site' or 'loco_site' or 'ci_pairable'
def rank_split(item):
    p,_ = item
    s = str(p).lower()
    return (
        0 if ("site" in s or "loco_site" in s) else 1,
        0 if "ci_pairable" in s else 1,
        0 if "tp53" in s else 1,
        len(s)
    )

def rank_label(item):
    p,_ = item
    s = str(p).lower()
    return (
        0 if ("label" in s or "gold" in s or "clinvar" in s or "hotspot" in s) else 1,
        0 if "tp53" in s else 1,
        len(s)
    )

site_split_candidates.sort(key=rank_split)
label_candidates.sort(key=rank_label)

if not site_split_candidates:
    raise SystemExit("BLOCKER: No LOCO-site split-map found. You likely only have 'splits_loco_gene.*'. Patch writer to emit site splits or locate the LOCO-site split CSV.")

sp_path, sp_df = site_split_candidates[0]
lb_path, lb_df = label_candidates[0]

print("[using split_map]:", sp_path)
print("  unique splits:", sorted(sp_df['split'].unique()))
print("[using labels  ]:", lb_path)

# 2) join -> strict -> counts
df = sp_df.merge(lb_df, on="variant_id", how="inner")
strict = df[df["split"].isin([0,1,2,4])].copy()
if strict.empty:
    # show what splits we actually have to help debugging
    print("Available splits in split_map:", sorted(sp_df['split'].unique()))
    raise SystemExit("No rows for strict folds {0,1,2,4}. Check that this is LOCO-site (not gene) and that TP53 labels exist for these folds.")

counts = (strict.groupby("split")["label"]
          .agg(n="size", n_pos="sum")
          .to_frame()
          .assign(n_neg=lambda x: x["n"] - x["n_pos"])
          .reset_index()
          .sort_values("split"))
out = sp_path.parent / "tp53_counts_strict_loco.csv"
counts.to_csv(out, index=False)

print("\n[OK] Wrote:", out)
print(counts.to_string(index=False))
print("md5:", md5sum(out))


Found site-split candidates: 20
Found label candidates    : 58
[using split_map]: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/preds_delta_bio_loco_site.csv
  unique splits: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
[using labels  ]: /content/drive/MyDrive/Cancer_gene_database/artifacts/error_analysis_20250820_001733/TP53_youdenJ_TP.csv


AttributeError: 'DataFrame' object has no attribute 'to_frame'

In [21]:
# sanity: show class mix; if it's all 1s or all 0s, you're using a filtered labels file (e.g., TPs only)
print("class balance overall:", strict["label"].value_counts(dropna=False).to_dict())

# counts without .to_frame()
counts = (strict.groupby("split")["label"]
          .agg(n="size", n_pos="sum")
          .reset_index()
          .assign(n_neg=lambda x: x["n"] - x["n_pos"])
          .sort_values("split"))
display(counts)

out = (sp_path.parent if "sp_path" in globals() else Path.cwd()) / "tp53_counts_strict_loco.csv"
counts.to_csv(out, index=False)
print("Wrote:", out)


class balance overall: {1: 38}


,split,n,n_pos,n_neg
0,0,8,8,0
1,1,8,8,0
2,2,11,11,0
3,4,11,11,0


Wrote: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/tp53_counts_strict_loco.csv


In [22]:
# TP53 LOCO-site counts (auto-pick proper labels w/ both classes, avoid TPs-only)
import re, hashlib
from pathlib import Path
import pandas as pd

def md5sum(path):
    h = hashlib.md5()
    with open(path,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

# ---- roots
CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/Cancer_gene_database/artifacts"),
    Path("/content/drive/MyDrive/artifacts"),
    Path.cwd() / "artifacts",
]
ART = next((p for p in CANDIDATE_ROOTS if p.exists()), None)
assert ART is not None, "No artifacts root found."

# ---- helpers
def list_datafiles(root):
    pats = ["*.csv","*.tsv","*.csv.gz","*.tsv.gz"]
    out=[]
    for pat in pats:
        out += list(root.rglob(pat))
    return out

def read_any(p, nrows=None):
    s = str(p).lower()
    # choose sep by suffix
    if s.endswith(".tsv") or s.endswith(".tsv.gz"):
        return pd.read_csv(p, sep="\t", nrows=nrows)
    # try default (handles csv/csv.gz)
    return pd.read_csv(p, nrows=nrows)

def coerce_split(series):
    s = series.astype(str).str.extract(r"(-?\d+)")[0].astype("Int64")
    # Map 1..5 -> 0..4 if needed
    s2 = s.map({1:0,2:1,3:2,4:3,5:4})
    s2 = s2.fillna(s)  # keep 0..4 if already that
    return s2.astype("Int64")

def find_site_split_map(csvs):
    cands=[]
    for p in csvs:
        s=str(p).lower()
        if "loco_gene" in s or "/gene" in s:
            continue
        try:
            df=read_any(p, nrows=200000)
        except Exception:
            continue
        cols={c.lower() for c in df.columns}
        # find split column
        split_col=None
        for c in df.columns:
            if c.lower() in {"split","fold","cv_fold"}:
                split_col=c; break
        if split_col is None or "variant_id" not in cols:
            continue
        df2=df[["variant_id",split_col]].copy()
        df2.columns=["variant_id","split"]
        df2["split"]=coerce_split(df2["split"])
        uniq=set(df2["split"].dropna().unique().tolist())
        if not uniq.issubset({0,1,2,3,4}):
            continue
        score = (
            0 if ("site" in s or "loco_site" in s) else 1,
            0 if "ci_pairable" in s else 1,
            0 if "tp53" in s else 1,
            len(s)
        )
        cands.append((score,p,df2.dropna()))
    assert cands, "No LOCO-site split map found."
    cands.sort(key=lambda x: x[0])
    return cands[0][1], cands[0][2]

def normalize_label(df):
    # choose label-like column
    for c in df.columns:
        cl=c.lower()
        if cl in {"label","y","target","oncogenic","class","is_driver","truth","y_true"}:
            v = df[c].astype(str).str.lower().str.strip()
            # map strings -> {0,1}
            map1 = {
                "driver":1,"oncogenic":1,"pathogenic":1,"likely_pathogenic":1,"lp":1,"pos":1,"true":1,"1":1,
                "benign":0,"likely-benign":0,"likely_benign":0,"lb":0,"neutral":0,"neg":0,"false":0,"0":0
            }
            if v.isin(map1).all():
                df["label"]=v.map(map1).astype(int)
                return True
            # already numeric?
            try:
                vi = pd.to_numeric(df[c], errors="raise")
                if set(vi.dropna().unique()).issubset({0,1}):
                    df["label"]=vi.astype(int)
                    return True
            except Exception:
                pass
    return False

def pick_labels(csvs, sp_df):
    cands=[]
    bad_tokens=("youden","_tp","_fp","_tn","_fn","error_analysis","preds","threshold")
    for p in csvs:
        s=str(p).lower()
        if any(t in s for t in bad_tokens):
            continue  # avoid derived/confounded files
        try:
            df=read_any(p)  # full read (labels are small)
        except Exception:
            continue
        cols={c.lower() for c in df.columns}
        if "variant_id" not in cols:
            continue
        if not normalize_label(df):
            continue
        # TP53 filter if gene present
        gcols=[c for c in df.columns if c.lower()=="gene"]
        if gcols:
            df = df[df[gcols[0]]=="TP53"]
            if df.empty:
                continue
        classes=set(df["label"].unique().tolist())
        if classes != {0,1}:
            continue  # must contain both classes
        # score by overlap with site split map (strict folds only)
        joined = df[["variant_id","label"]].merge(sp_df, on="variant_id", how="inner")
        strict = joined[joined["split"].isin([0,1,2,4])]
        overlap = len(strict)
        if overlap==0:
            continue
        score = (
            0 if any(k in s for k in ["labels","gold","clinvar","hotspot"]) else 1,
            0 if "tp53" in s else 1,
            -overlap,            # more overlap better
            len(s)
        )
        cands.append((score,p,strict))
    assert cands, "No suitable labels table with both classes overlapped with LOCO-site splits."
    cands.sort(key=lambda x: x[0])
    return cands[0][1], cands[0][2]

# ---- run
allfiles = list_datafiles(ART)
sp_path, sp_df = find_site_split_map(allfiles)
lb_path, strict = pick_labels(allfiles, sp_df)

print("[split_map]:", sp_path)
print("[labels   ]:", lb_path)
print("strict folds present:", sorted(strict['split'].unique()))

# sanity: class mix per split
mix = strict.groupby("split")["label"].value_counts().unstack(fill_value=0).rename(columns={0:"n_neg",1:"n_pos"})
print("\nclass mix per split:")
print(mix)

# counts emit
counts = (strict.groupby("split")["label"]
          .agg(n="size", n_pos="sum")
          .reset_index()
          .assign(n_neg=lambda x: x["n"] - x["n_pos"])
          .sort_values("split"))

out = Path(sp_path).parent / "tp53_counts_strict_loco.csv"
counts.to_csv(out, index=False)

print("\n[OK] wrote:", out)
print(counts.to_string(index=False))
print("md5:", md5sum(out))


[split_map]: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv
[labels   ]: /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv
strict folds present: [np.int64(0), np.int64(1), np.int64(2), np.int64(4)]

class mix per split:
label  n_neg  n_pos
split              
0         14     11
1         14     11
2         14     11
4         14     11

[OK] wrote: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/tp53_counts_strict_loco.csv
 split  n  n_pos  n_neg
     0 25     11     14
     1 25     11     14
     2 25     11     14
     4 25     11     14
md5: 0d572eb9dcfe7465c14ab8dd4085cfc3


### TP53 — LOCO-site per-fold counts (strict)

| fold |  n | n_pos | n_neg |
|-----:|---:|------:|------:|
| 0    | 25 |    11 |    14 |
| 1    | 25 |    11 |    14 |
| 2    | 25 |    11 |    14 |
| 4    | 25 |    11 |    14 |

**Policy:** LOCO-site only; exclude LOGO; no imputation; seeds AUROC=42, PR=43.  
**Split-map:** `/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv`  
**Labels:** `/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv`  
**Counts CSV:** `/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/tp53_counts_strict_loco.csv`  
**Counts md5:** `0d572eb9dcfe7465c14ab8dd4085cfc3`


### Add counts into the per-fold metrics

In [24]:
# Cell 1 — locate TP53 per-fold metrics and counts files (prints choices + picks best)
from pathlib import Path
import hashlib, os, time

def md5sum(p):
    h = hashlib.md5()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

# 1) find artifacts root
CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/Cancer_gene_database/artifacts"),
    Path("/content/drive/MyDrive/artifacts"),
    Path.cwd() / "artifacts",
]
ART = next((p for p in CANDIDATE_ROOTS if p.exists()), None)
assert ART is not None, "No artifacts root found."

# 2) search candidates
met_candidates = list(ART.rglob("tp53_per_fold_metrics.csv"))
if not met_candidates:
    met_candidates = list(ART.rglob("*per_fold*metrics*.csv"))

cnt_candidates = list(ART.rglob("tp53_counts_strict_loco.csv"))

print("Metrics candidates found:", len(met_candidates))
for i,p in enumerate(met_candidates,1):
    try:
        print(f"[M{i}] {p} | mtime={time.strftime('%Y-%m-%dT%H:%M:%S', time.localtime(p.stat().st_mtime))} | md5={md5sum(p)}")
    except Exception as e:
        print(f"[M{i}] {p} | <md5 error: {e}>")

print("\nCounts candidates found:", len(cnt_candidates))
for i,p in enumerate(cnt_candidates,1):
    try:
        print(f"[C{i}] {p} | mtime={time.strftime('%Y-%m-%dT%H:%M:%S', time.localtime(p.stat().st_mtime))} | md5={md5sum(p)}")
    except Exception as e:
        print(f"[C{i}] {p} | <md5 error: {e}>")

# 3) pick best guess heuristically
def score_metrics(p: Path):
    s = str(p).lower()
    return (
        0 if "ci_pairable" in s else 1,
        0 if "calibrated_robust_20250820_000001" in s else 1,
        -p.stat().st_mtime,  # newer better
        len(s)
    )

def score_counts(p: Path):
    s = str(p).lower()
    return (
        0 if "sentiment_run_plus_variants_20250816_190607" in s else 1,
        -p.stat().st_mtime,
        len(s)
    )

METRICS_P = None
COUNTS_P = None

if met_candidates:
    met_candidates.sort(key=score_metrics)
    METRICS_P = met_candidates[0]
if cnt_candidates:
    cnt_candidates.sort(key=score_counts)
    COUNTS_P = cnt_candidates[0]

print("\nChosen METRICS_P:", METRICS_P)
print("Chosen COUNTS_P :", COUNTS_P)

# keep paths in globals for the next cell


Metrics candidates found: 1
[M1] /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/tp53_per_fold_metrics.csv | mtime=2025-08-22T01:38:16 | md5=652075be1faf88b14fd84862292d5001

Counts candidates found: 1
[C1] /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/tp53_counts_strict_loco.csv | mtime=2025-08-22T19:44:03 | md5=0d572eb9dcfe7465c14ab8dd4085cfc3

Chosen METRICS_P: /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/tp53_per_fold_metrics.csv
Chosen COUNTS_P : /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/tp53_counts_strict_loco.csv


In [25]:
# Cell — merge counts into TP53 per-fold metrics (robust)
import pandas as pd, re, hashlib, time
from pathlib import Path

def md5sum(p: Path):
    h = hashlib.md5()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

# 0) pick up the paths discovered in the prior cell
assert 'METRICS_P' in globals() and 'COUNTS_P' in globals(), "Run the locator cell first to set METRICS_P and COUNTS_P."
metrics_p = Path(str(METRICS_P))
counts_p  = Path(str(COUNTS_P))

print("METRICS_P:", metrics_p)
print("COUNTS_P :", counts_p)

# 1) load
m = pd.read_csv(metrics_p)
c = pd.read_csv(counts_p)

# 2) normalize split/fold column in metrics -> 'split' int in {0..4}
fold_col = next((col for col in m.columns if col.lower() in {"split","fold","cv_fold"}), None)
assert fold_col, f"No split/fold column found in metrics: {list(m.columns)}"
m = m.rename(columns={fold_col:"split"}).copy()

def coerce_split(s):
    # pull first integer; map 1..5->0..4 if needed
    out = s.astype(str).str.extract(r"(-?\d+)")[0].astype("Int64")
    out = out.map({1:0,2:1,3:2,4:3,5:4}).fillna(out).astype("Int64")
    return out

m["split"] = coerce_split(m["split"])
c["split"] = c["split"].astype("Int64")

# 3) merge (left join to keep metrics rows), then sanity checks
mc = m.merge(c, on="split", how="left")

# sanity: strict LOCO folds must be subset of {0,1,2,4}
present_splits = set(int(x) for x in mc["split"].dropna().unique())
assert present_splits.issubset({0,1,2,4}), f"Unexpected splits present: {sorted(present_splits)}"

# all strict folds should have counts
missing_counts = mc[mc["split"].isin([0,1,2,4]) & (mc[["n","n_pos","n_neg"]].isna().any(axis=1))]
assert missing_counts.empty, f"Counts failed to merge for some strict folds:\n{missing_counts[['split','n','n_pos','n_neg']]}"

# 4) emit
out_p = metrics_p.with_name("tp53_per_fold_metrics_with_counts.csv")
mc.to_csv(out_p, index=False)

print("\n[OK] wrote:", out_p)
print("mtime:", time.strftime("%Y-%m-%dT%H:%M:%S", time.localtime(out_p.stat().st_mtime)))
print("md5  :", md5sum(out_p))

# 5) preview rows per split (quick eyes-on)
preview = mc.loc[mc["split"].isin([0,1,2,4]), ["split","n","n_pos","n_neg"]].drop_duplicates().sort_values("split")
print("\nMerged counts (per split):")
print(preview.to_string(index=False))


METRICS_P: /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/tp53_per_fold_metrics.csv
COUNTS_P : /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/tp53_counts_strict_loco.csv


AssertionError: Unexpected splits present: [0, 1, 3, 53]

# Results recovery

In [26]:
# Sanitize per-fold metrics to LOCO-site strict, then merge counts and emit a clean CSV.
import re, hashlib, time
import pandas as pd
from pathlib import Path

def md5sum(p: Path):
    h = hashlib.md5()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

assert 'METRICS_P' in globals() and 'COUNTS_P' in globals(), "Run the locator cell first."
metrics_p = Path(str(METRICS_P))
counts_p  = Path(str(COUNTS_P))

m = pd.read_csv(metrics_p)
c = pd.read_csv(counts_p)

# 0) keep TP53 only if a gene column exists
gcols = [col for col in m.columns if col.lower()=='gene']
if gcols:
    before = len(m)
    m = m[m[gcols[0]]=='TP53'].copy()
    print(f"[gene filter] kept TP53 rows: {len(m)}/{before}")

# 1) find a fold/split column
cand_cols = [col for col in m.columns if col.lower() in {'split','fold','cv_fold','site_fold','site_split'}]
assert cand_cols, f"No split/fold column in metrics. Columns: {list(m.columns)}"
fold_col = cand_cols[0]

# 2) build a LOCO-site mask from any tag/cv/mode-ish columns (exclude LOGO-gene rows)
text_cols = [col for col in m.columns if any(k in col.lower() for k in ['tag','cv','mode','fold_type','scheme','split_type'])]
if not text_cols:
    # fallback: use the fold column text itself
    text_cols = [fold_col]

row_text = m[text_cols].astype(str).agg(" ".join, axis=1).str.lower()
is_logo_gene = row_text.str.contains("loco_gene")
is_loco_site = row_text.str.contains("loco_site|site")
m_site = m[~is_logo_gene].copy()
print(f"[cv filter] dropped LOGO-gene rows: {int(is_logo_gene.sum())}")

# 3) parse split safely: only accept tokens explicitly tied to split/fold/site to avoid 'TP53' → '53'
def parse_split_safe(series):
    s = series.astype(str)
    # If already numeric and in 0..4, keep
    try:
        sn = pd.to_numeric(s, errors='raise')
        if set(sn.dropna().unique()) <= set(range(0,5)):
            return sn.astype('Int64')
    except Exception:
        pass
    # Token-based: look for '(site|split|fold)[^0-9]*([0-9]+)'
    m = s.str.extract(r'(?i)(?:^|[ _:-])(site|split|fold)\D*([0-9]+)(?:$|[^0-9])')
    num = pd.to_numeric(m[1], errors='coerce').astype('Int64')
    # map 1..5 to 0..4 if needed
    num = num.map({1:0,2:1,3:2,4:3,5:4}).fillna(num).astype('Int64')
    return num

m_site["split"] = parse_split_safe(m_site[fold_col])
# Show what we got
print("[split parse] unique parsed splits:", sorted(x for x in m_site["split"].dropna().unique()))
# 4) strict folds only (0,1,2,4). Drop others (e.g., 3) for the strict line.
strict_mask = m_site["split"].isin([0,1,2,4])
dropped = (~strict_mask).sum()
m_strict = m_site[strict_mask].copy()
print(f"[strict] kept rows with splits {{0,1,2,4}}: {len(m_strict)} (dropped {dropped})")

# 5) merge counts on split
c["split"] = c["split"].astype('Int64')
mc = m_strict.merge(c, on="split", how="left")

# sanity: all strict rows should have counts
missing = mc[mc[["n","n_pos","n_neg"]].isna().any(axis=1)]
assert missing.empty, f"Counts missing for some strict rows:\n{missing[['split','n','n_pos','n_neg']].drop_duplicates()}"

# 6) emit
out_p = metrics_p.with_name("tp53_per_fold_metrics_with_counts_strict.csv")
mc.to_csv(out_p, index=False)
print("\n[OK] wrote:", out_p)
print("mtime:", time.strftime("%Y-%m-%dT%H:%M:%S", time.localtime(out_p.stat().st_mtime)), "| md5:", md5sum(out_p))

# 7) quick preview of counts per split
preview = mc[["split","n","n_pos","n_neg"]].drop_duplicates().sort_values("split")
print("\nMerged counts (per split):")
print(preview.to_string(index=False))


[gene filter] kept TP53 rows: 5/5
[cv filter] dropped LOGO-gene rows: 1
[split parse] unique parsed splits: [np.int64(0), np.int64(1), np.int64(2), np.int64(4)]
[strict] kept rows with splits {0,1,2,4}: 4 (dropped 0)

[OK] wrote: /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/tp53_per_fold_metrics_with_counts_strict.csv
mtime: 2025-08-22T20:03:11 | md5: a4f9bb130646f4cfc0df72298394a2b4

Merged counts (per split):
 split  n  n_pos  n_neg
     0 25     11     14
     1 25     11     14
     2 25     11     14
     4 25     11     14


In [27]:
# Update README with TP53 LOCO-site per-fold counts (strict) + provenance.
import pandas as pd, hashlib, time, textwrap, os, re
from pathlib import Path

def md5sum(p: Path):
    h = hashlib.md5()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

# Inputs (we already created COUNTS_P; adjust if you moved things)
assert 'COUNTS_P' in globals(), "Run the locator/merge steps first to set COUNTS_P."
counts_p = Path(str(COUNTS_P))
splitmap_p = counts_p.parent / "splits_loco_site.csv"  # same dir as counts
labels_p   = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv")

readme_p = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/ci_pairable/README_TP53_LOCO_sources_and_methods_calibrated_robust_20250820_000001_20250822.md")
readme_p.parent.mkdir(parents=True, exist_ok=True)

# Load counts
df = pd.read_csv(counts_p).sort_values("split")
assert set(df["split"]) == {0,1,2,4}, f"Unexpected strict folds: {sorted(df['split'].unique())}"
assert (df["n"] == df["n_pos"] + df["n_neg"]).all(), "n != n_pos + n_neg"

# Build markdown table
rows = "\n".join([f"| {int(r.split)} | {int(r.n)} | {int(r.n_pos)} | {int(r.n_neg)} |" for r in df.itertuples()])
counts_md = f"""
<!-- TP53_COUNTS_START -->
### TP53 — LOCO-site per-fold counts (strict)

| fold |  n | n_pos | n_neg |
|-----:|---:|------:|------:|
{rows}

**Policy:** LOCO-site only; exclude LOGO; no imputation; seeds AUROC=42, PR=43.
**Split-map:** `{splitmap_p}`
**Labels:** `{labels_p}`
**Counts CSV:** `{counts_p}`
**Counts md5:** `{md5sum(counts_p)}`

_Last updated: {time.strftime('%Y-%m-%dT%H:%M:%S')}_
<!-- TP53_COUNTS_END -->
""".strip()+"\n"

# Insert or replace block
if readme_p.exists():
    text = readme_p.read_text()
    if "<!-- TP53_COUNTS_START -->" in text and "<!-- TP53_COUNTS_END -->" in text:
        text = re.sub(r"<!-- TP53_COUNTS_START -->.*?<!-- TP53_COUNTS_END -->", counts_md, text, flags=re.S)
    else:
        # append near top
        text = counts_md + "\n\n" + text
else:
    text = counts_md

readme_p.write_text(text)
print("README updated:", readme_p)
print("md5:", md5sum(readme_p))
print("\nRendered block:\n")
print(counts_md)


README updated: /content/drive/MyDrive/Cancer_gene_database/artifacts/ci_pairable/README_TP53_LOCO_sources_and_methods_calibrated_robust_20250820_000001_20250822.md
md5: b387825e100495e0443c0d9b72aa42ca

Rendered block:

<!-- TP53_COUNTS_START -->
### TP53 — LOCO-site per-fold counts (strict)

| fold |  n | n_pos | n_neg |
|-----:|---:|------:|------:|
| 0 | 25 | 11 | 14 |
| 1 | 25 | 11 | 14 |
| 2 | 25 | 11 | 14 |
| 4 | 25 | 11 | 14 |

**Policy:** LOCO-site only; exclude LOGO; no imputation; seeds AUROC=42, PR=43.  
**Split-map:** `/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv`  
**Labels:** `/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv`  
**Counts CSV:** `/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/tp53_counts_strict_loco.csv`  
**Counts md5:** `0d572eb9dcfe7465c14ab8dd4085cfc3`

_Last upd

In [28]:
# Cell — update tp53_loco_strict_summary.json with counts + provenance (non-destructive)
import json, time, hashlib
from pathlib import Path
import pandas as pd

def md5sum(p: Path):
    h = hashlib.md5()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

# Paths (from the receipts you just locked)
summary_p  = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/ci_pairable/tp53_loco_strict_summary.json")
counts_p   = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/tp53_counts_strict_loco.csv")
splitmap_p = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv")
labels_p   = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv")

# Load current summary (or create shell)
if summary_p.exists():
    s = json.loads(summary_p.read_text())
else:
    s = {}

# Load counts
counts_df = pd.read_csv(counts_p).sort_values("split")
assert set(counts_df["split"]) == {0,1,2,4}, f"Unexpected strict folds: {sorted(counts_df['split'].unique())}"
assert (counts_df["n"] == counts_df["n_pos"] + counts_df["n_neg"]).all(), "n != n_pos + n_neg"

# Merge without clobbering existing keys
s.setdefault("run_id", "calibrated_robust_20250820_000001")
s.setdefault("mapping", {"gene":"TP53","codon_col":"p_wt","context_col":"context"})
s.setdefault("policy", "LOCO-site only; exclude LOGO; no imputation; seeds AUROC=42, PR=43")
s.setdefault("bootstrap", {"n_reps":10000, "seeds":{"auroc":42, "pr":43}})

# Attach counts + provenance
s["counts"] = {
    "strict_folds": [0,1,2,4],
    "table_csv": str(counts_p),
    "md5": md5sum(counts_p),
    "per_fold": counts_df.to_dict(orient="records"),
    "provenance": {
        "split_map": str(splitmap_p),
        "labels": str(labels_p),
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S")
    }
}

# Backup then write
if summary_p.exists():
    backup = summary_p.with_suffix(".json.bak_" + time.strftime("%Y%m%d%H%M%S"))
    backup.write_text(json.dumps(s, indent=2))
summary_p.write_text(json.dumps(s, indent=2))
print("Updated:", summary_p)
print("md5:", md5sum(summary_p))

# Show the counts section we added
print("\ncounts block:")
print(json.dumps(s["counts"], indent=2))


Updated: /content/drive/MyDrive/Cancer_gene_database/artifacts/ci_pairable/tp53_loco_strict_summary.json
md5: 88e47e48a115f7ec3e94bbff37673fc6

counts block:
{
  "strict_folds": [
    0,
    1,
    2,
    4
  ],
  "table_csv": "/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/tp53_counts_strict_loco.csv",
  "md5": "0d572eb9dcfe7465c14ab8dd4085cfc3",
  "per_fold": [
    {
      "split": 0,
      "n": 25,
      "n_pos": 11,
      "n_neg": 14
    },
    {
      "split": 1,
      "n": 25,
      "n_pos": 11,
      "n_neg": 14
    },
    {
      "split": 2,
      "n": 25,
      "n_pos": 11,
      "n_neg": 14
    },
    {
      "split": 4,
      "n": 25,
      "n_pos": 11,
      "n_neg": 14
    }
  ],
  "provenance": {
    "split_map": "/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv",
    "labels": "/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_202

## Gold Standard (receipts-or-nothing)

### A. Core evidence
- [ ] **LOCO-site** per-fold deltas *(context − codon)*, **n = 5** (folds 0–4), 10k bootstrap 95% CIs *(seeds: AUROC=42, PR=43)*, paired Wilcoxon (one-sided) **p < 0.05**.
- [ ] **Per-fold counts** *(strict done)*; extend counts to **all 5 folds** once fold 3 is recovered.

### B. Calibration & scoring quality
- [ ] **Reliability curves** *(pre/post calibration)*, **Brier** deltas, **ECE**.
- [ ] Show rank gains aren’t just noise: report **ΔAUROC**, **ΔPR**, **ΔBrier**.

### C. Clinical utility
- [ ] **Decision Curve Analysis** (net benefit vs thresholds of interest) for **codon vs context**.

### D. Leakage & split integrity
- [ ] Prove **LOCO-site isolation** (no site appears across folds; WT/MUT pairs co-stay).
- [ ] Show **counts match metrics** (per-fold).

### E. Mechanism sanity / robustness
- [ ] **Shuffle flanks** (break context) → effect drops.
- [ ] **Mask CpG/TpC** (hypermutability control) → effect persists.
- [ ] **Flank-length sensitivity** (±30 / ±60 / ±90) → effect stable.
- [ ] *(Optional)* Tiny **CNN-lite + saliency** as illustration (no SOTA claims).

### F. External sanity
- [ ] Small **external TP53 holdout** slice (clearly defined); report deltas with wide CIs to show direction holds.

### G. Repro pack
- [ ] **Per-fold predictions CSV**: `site_id, variant_id, label, split, p_wt, context`.
- [ ] **Env lock** (`requirements.txt`), **RUN_ID**, file paths + **md5**, analysis **seeds**.


## Step 1 (of many): Recover/verify fold 3 for LOCO-site

This cell hunts LOCO-site preds with numeric split 0..4 that carry both scores (p_wt and context or reasonable aliases). If it finds them, it:

Computes per-fold counts for all 5 folds

Recomputes per-fold deltas and updates an tp53_per_fold_with_counts_full.csv

Prints what’s missing if scores aren’t present (so we know exactly what to regenerate)

In [29]:
# Cell — Recover/verify fold 3 (LOCO-site) and emit full per-fold counts + deltas if scores exist
import re, json, time, hashlib
from pathlib import Path
import pandas as pd

def md5sum(p: Path):
    h = hashlib.md5()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

# Roots we’ve been using
ART = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
assert ART.exists(), "Artifacts root not found."

# Helpers
def list_tables(root):
    out=[]
    for suf in (".csv",".tsv",".csv.gz",".tsv.gz"):
        out += list(root.rglob(f"*{suf}"))
    return out

def read_any(p, nrows=None):
    s=str(p).lower()
    sep = "\t" if s.endswith(".tsv") or s.endswith(".tsv.gz") else ","
    return pd.read_csv(p, sep=sep, nrows=nrows)

def coerce_split(series):
    s = series.astype(str).str.extract(r"(-?\d+)")[0].astype("Int64")
    s = s.map({1:0,2:1,3:2,4:3,5:4}).fillna(s).astype("Int64")
    return s

def has_scores(df):
    cols_lower = {c.lower():c for c in df.columns}
    # common aliases
    codon_keys   = ["p_wt","codon","codon_score","score_codon","baseline","p_wt_score"]
    context_keys = ["context","context_score","score_context","p_ctx","p_context"]
    codon_col   = next((cols_lower[k] for k in codon_keys   if k in cols_lower), None)
    context_col = next((cols_lower[k] for k in context_keys if k in cols_lower), None)
    return codon_col, context_col

def pick_loco_site_preds():
    cands=[]
    for p in list_tables(ART):
        s=str(p).lower()
        # must smell like preds and site (not gene)
        if "loco_gene" in s or "gene" in p.name.lower():
            continue
        if not any(k in s for k in ["pred","score","delta","loco_site","site"]):
            continue
        try:
            df = read_any(p, nrows=None)
        except Exception:
            continue
        cols = {c.lower() for c in df.columns}
        # Need variant_id + split-like + label-like + some score columns
        if "variant_id" not in cols:
            continue
        split_col = next((c for c in df.columns if c.lower() in {"split","fold","cv_fold","site_split","site_fold"}), None)
        if not split_col:
            continue
        # normalize split
        df["split"] = coerce_split(df[split_col])
        if not set(df["split"].dropna().unique()).issubset({0,1,2,3,4}):
            continue
        # normalize label
        label_col = next((c for c in df.columns if c.lower() in {"label","y","target","oncogenic","class","is_driver","truth","y_true"}), None)
        if not label_col:
            continue
        v = df[label_col].astype(str).str.lower().str.strip()
        # map to {0,1}
        map1 = {"driver":1,"oncogenic":1,"pathogenic":1,"likely_pathogenic":1,"lp":1,"pos":1,"true":1,"1":1,
                "benign":0,"likely-benign":0,"likely_benign":0,"lb":0,"neutral":0,"neg":0,"false":0,"0":0}
        if v.isin(map1).all():
            df["label"] = v.map(map1).astype(int)
        else:
            try:
                vi = pd.to_numeric(df[label_col], errors="raise")
                if set(vi.dropna().unique()).issubset({0,1}):
                    df["label"] = vi.astype(int)
                else:
                    continue
            except Exception:
                continue
        # TP53 filter if gene col exists
        gcols = [c for c in df.columns if c.lower()=="gene"]
        if gcols:
            df = df[df[gcols[0]]=="TP53"]
            if df.empty:
                continue
        codon_col, context_col = has_scores(df)
        cands.append((p, df, codon_col, context_col))
    return cands

# 1) Try to find a preds table with split 0..4 and (p_wt, context)
cand_preds = pick_loco_site_preds()
print(f"LOCO-site preds candidates: {len(cand_preds)}")

# Prefer names with 'loco_site' then 'ci_pairable'
def score_pred(p, codon_col, context_col):
    s=str(p).lower()
    return (
        0 if "loco_site" in s or "site" in s else 1,
        0 if "ci_pairable" in s else 1,
        0 if (codon_col is not None and context_col is not None) else 1,
        -p.stat().st_mtime,
        len(s)
    )

if not cand_preds:
    raise SystemExit("No LOCO-site preds found. We’ll need to regenerate or persist a per-variant preds CSV with split 0..4 and both scores.")

cand_preds.sort(key=lambda x: score_pred(x[0], x[2], x[3]))
pred_path, pred_df, codon_col, context_col = cand_preds[0]
print("[using preds]:", pred_path)
print("columns snapshot:", sorted(pred_df.columns)[:15])
print("unique splits:", sorted(int(x) for x in pred_df["split"].dropna().unique()))

if codon_col is None or context_col is None:
    print("\n[BLOCKER] This preds file lacks score columns for codon/context.")
    print("Looked for codon aliases: p_wt, codon, codon_score, score_codon, baseline, p_wt_score")
    print("and context aliases: context, context_score, score_context, p_ctx, p_context")
    raise SystemExit("Regenerate or locate a preds CSV that includes both scores.")

# 2) Confirm fold 3 exists and report per-fold counts for all 5 folds
per_fold_counts = (pred_df.groupby("split")["label"]
                   .agg(n="size", n_pos="sum").assign(n_neg=lambda x: x["n"]-x["n_pos"]))
print("\nPer-fold counts (all splits present):")
print(per_fold_counts.to_string())

# 3) Compute per-fold metrics & deltas if both scores exist
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
def safe_pr(y_true, y_prob):
    # AP needs both classes; fallback to NaN if degenerate
    if len(set(y_true))<2:
        return float("nan")
    return average_precision_score(y_true, y_prob)

rows=[]
for sp, sub in pred_df.groupby("split"):
    y = sub["label"].astype(int).values
    pw = sub[codon_col].astype(float).values
    pc = sub[context_col].astype(float).values
    # AUROC / PR / Brier
    au_w = roc_auc_score(y, pw) if len(set(y))==2 else float("nan")
    au_c = roc_auc_score(y, pc) if len(set(y))==2 else float("nan")
    pr_w = safe_pr(y, pw)
    pr_c = safe_pr(y, pc)
    br_w = brier_score_loss(y, pw)
    br_c = brier_score_loss(y, pc)
    rows.append({
        "split": int(sp),
        "n": len(y), "n_pos": int(y.sum()), "n_neg": int((1-y).sum()),
        "AUROC_codon": au_w, "AUROC_context": au_c, "ΔAUROC": (au_c - au_w) if (au_c==au_c and au_w==au_w) else float("nan"),
        "PR_codon": pr_w,   "PR_context": pr_c,     "ΔPR": (pr_c - pr_w) if (pr_c==pr_c and pr_w==pr_w) else float("nan"),
        "Brier_codon": br_w,"Brier_context": br_c,  "ΔBrier": (br_c - br_w)
    })

full_per_fold = pd.DataFrame(rows).sort_values("split")
# Save with counts (all 5 splits if present)
out_dir = pred_path.parent
full_csv = out_dir / "tp53_per_fold_with_counts_full.csv"
full_per_fold.to_csv(full_csv, index=False)
print("\n[OK] Wrote:", full_csv, "| md5:", md5sum(full_csv))

# 4) If split 3 exists, print the strict-vs-n5 status
splits_present = sorted(full_per_fold["split"].unique().tolist())
print("\nSplits present:", splits_present)
print("\nFull per-fold (head):")
print(full_per_fold.to_string(index=False))


LOCO-site preds candidates: 9
[using preds]: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/preds_delta_anchor_loco_site.csv
columns snapshot: ['fold', 'gene', 'label', 'pair_ok', 'split', 'split_mode', 'variant_id', 'y_pred_delta_anchor']
unique splits: [0, 1, 2, 3, 4]

[BLOCKER] This preds file lacks score columns for codon/context.
Looked for codon aliases: p_wt, codon, codon_score, score_codon, baseline, p_wt_score
and context aliases: context, context_score, score_context, p_ctx, p_context


SystemExit: Regenerate or locate a preds CSV that includes both scores.

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


that file’s just delta-only. We need raw codon and context scores. Here’s one cell that hunts for separate preds with those scores (avoids *delta* files), merges them on variant_id+split, filters to TP53 + LOCO-site, and emits the full per-fold table (including fold 3 if present). If it can’t find both, it prints an exact regeneration spec.

In [30]:
# Cell — find/merge LOCO-site preds with raw codon & context scores (no delta),
# then emit full per-fold counts+metrics if found.

import re, time, hashlib
from pathlib import Path
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ART = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
assert ART.exists(), "Artifacts root not found."

def md5sum(p: Path):
    h = hashlib.md5()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

def list_tables(root):
    out=[]
    for suf in (".csv",".tsv",".csv.gz",".tsv.gz"):
        out += list(root.rglob(f"*{suf}"))
    return out

def read_any(p, nrows=None):
    s=str(p).lower()
    sep = "\t" if s.endswith(".tsv") or s.endswith(".tsv.gz") else ","
    try:
        return pd.read_csv(p, sep=sep, nrows=nrows)
    except Exception:
        return None

def coerce_split(series):
    s = series.astype(str).str.extract(r"(-?\d+)")[0].astype("Int64")
    return s.map({1:0,2:1,3:2,4:3,5:4}).fillna(s).astype("Int64")

# Column alias sets
CODON_KEYS   = {"p_wt","codon","codon_score","score_codon","baseline","p_wt_score","y_pred_wt","wt_pred"}
CONTEXT_KEYS = {"context","context_score","score_context","p_ctx","p_context","y_pred_ctx","ctx_pred","blend","bio"}
SPLIT_KEYS   = {"split","fold","cv_fold","site_split","site_fold"}
LABEL_KEYS   = {"label","y","target","oncogenic","class","is_driver","truth","y_true"}

def normalize_label(df, label_col):
    v = df[label_col].astype(str).str.lower().str.strip()
    map1 = {"driver":1,"oncogenic":1,"pathogenic":1,"likely_pathogenic":1,"lp":1,"pos":1,"true":1,"1":1,
            "benign":0,"likely-benign":0,"likely_benign":0,"lb":0,"neutral":0,"neg":0,"false":0,"0":0}
    if v.isin(map1).all():
        df["label"] = v.map(map1).astype(int); return True
    try:
        vi = pd.to_numeric(df[label_col], errors="raise")
        if set(vi.dropna().unique()).issubset({0,1}):
            df["label"] = vi.astype(int); return True
    except Exception:
        pass
    return False

def find_pred_files():
    """Return lists of (path, df, role) where role in {'codon','context'}."""
    codon_preds, context_preds = [], []
    for p in list_tables(ART):
        s = str(p).lower()
        if "delta" in s:                 # skip delta-only
            continue
        if "loco_gene" in s:             # skip LOGO-gene
            continue
        if not any(k in s for k in ["pred","score","prob","loco_site","site"]):
            continue
        df = read_any(p)
        if df is None or df.empty:
            continue
        cols = {c.lower(): c for c in df.columns}
        # require variant_id + split-like + label-like
        if "variant_id" not in cols:
            continue
        split_col = next((cols[k] for k in SPLIT_KEYS if k in cols), None)
        if not split_col:
            continue
        label_col = next((cols[k] for k in LABEL_KEYS if k in cols), None)
        if not label_col:
            continue
        # normalize split + label
        df["split"] = coerce_split(df[split_col])
        if not set(df["split"].dropna().unique()).issubset({0,1,2,3,4}):
            continue
        if not normalize_label(df, label_col):
            continue
        # TP53-only if gene present
        gcols = [c for c in df.columns if c.lower()=="gene"]
        if gcols:
            df = df[df[gcols[0]]=="TP53"]
            if df.empty:
                continue
        # detect score role
        codon_col   = next((cols[k] for k in CODON_KEYS   if k in cols), None)
        context_col = next((cols[k] for k in CONTEXT_KEYS if k in cols), None)

        # We want files with exactly ONE role each (codon-only or context-only)
        role = None
        if codon_col and not context_col:
            role = "codon"; df = df[["variant_id","split","label",codon_col]].rename(columns={codon_col:"p_wt"})
            codon_preds.append((p, df))
        elif context_col and not codon_col:
            role = "context"; df = df[["variant_id","split","label",context_col]].rename(columns={context_col:"context"})
            context_preds.append((p, df))
        elif codon_col and context_col:
            # jackpot: single file has both
            df = df[["variant_id","split","label",codon_col,context_col]].rename(columns={codon_col:"p_wt",context_col:"context"})
            codon_preds.append((p, df.copy()))
            context_preds.append((p, df.copy()))
        else:
            continue
    return codon_preds, context_preds

codon_files, context_files = find_pred_files()
print(f"Found codon-only/both files: {len(codon_files)}")
print(f"Found context-only/both files: {len(context_files)}")

# Heuristic: prefer paths with 'loco_site' and 'ci_pairable'; avoid 'delta'
def score(p):
    s = str(p).lower()
    return (
        0 if "loco_site" in s or "site" in s else 1,
        0 if "ci_pairable" in s else 1,
        -p.stat().st_mtime,
        len(s)
    )

if not codon_files or not context_files:
    print("\n[BLOCKER] Could not find both codon and context score tables (non-delta) for TP53 LOCO-site.")
    print("→ Regenerate a single preds CSV with columns:")
    print("   variant_id, split (0..4), label, p_wt, context   # LOCO-site only, TP53 filtered or filterable by 'gene'")
    print("   Save as: .../ci_pairable/preds_loco_site_tp53.csv")
else:
    codon_files.sort(key=lambda x: score(x[0]))
    context_files.sort(key=lambda x: score(x[0]))
    pc_path, pc = codon_files[0][0], codon_files[0][1]
    cx_path, cx = context_files[0][0], context_files[0][1]
    print("\n[using codon ]:", pc_path)
    print("[using context]:", cx_path)

    # Merge on variant_id+split; intersect labels just in case
    merged = pc.merge(cx, on=["variant_id","split","label"], how="inner")
    if merged.empty:
        print("\n[BLOCKER] Codon/context tables don’t overlap on (variant_id, split, label).")
    else:
        # Per-fold counts (expect all 0..4)
        print("\nPer-fold counts (merged codon+context):")
        cf = merged.groupby("split")["label"].agg(n="size", n_pos="sum")
        cf = cf.to_frame().assign(n_neg=lambda x: x["n"]-x["n_pos"])
        print(cf.to_string())

        # Per-fold metrics & deltas
        def safe_pr(y_true, y_prob):
            return average_precision_score(y_true, y_prob) if len(set(y_true))==2 else float("nan")

        rows=[]
        for sp, sub in merged.groupby("split"):
            y = sub["label"].astype(int).values
            pw = sub["p_wt"].astype(float).values
            pc = sub["context"].astype(float).values
            au_w = roc_auc_score(y, pw) if len(set(y))==2 else float("nan")
            au_c = roc_auc_score(y, pc) if len(set(y))==2 else float("nan")
            pr_w = safe_pr(y, pw)
            pr_c = safe_pr(y, pc)
            br_w = brier_score_loss(y, pw)
            br_c = brier_score_loss(y, pc)
            rows.append({
                "split": int(sp),
                "n": len(y), "n_pos": int(y.sum()), "n_neg": int((1-y).sum()),
                "AUROC_codon": au_w, "AUROC_context": au_c, "ΔAUROC": (au_c - au_w) if (au_c==au_c and au_w==au_w) else float("nan"),
                "PR_codon": pr_w,   "PR_context": pr_c,     "ΔPR": (pr_c - pr_w) if (pr_c==pr_c and pr_w==pr_w) else float("nan"),
                "Brier_codon": br_w,"Brier_context": br_c,  "ΔBrier": (br_c - br_w)
            })
        full_per_fold = pd.DataFrame(rows).sort_values("split")
        out_dir = cx_path.parent
        full_csv = out_dir / "tp53_per_fold_with_counts_full.csv"
        full_per_fold.to_csv(full_csv, index=False)
        print("\n[OK] Wrote:", full_csv, "| md5:", md5sum(full_csv))
        print("\nSplits present:", sorted(full_per_fold['split'].unique().tolist()))
        print("\nFull per-fold:")
        print(full_per_fold.to_string(index=False))


Found codon-only/both files: 5
Found context-only/both files: 1

[using codon ]: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/preds_wt_only_control_LOCO-SITE.csv
[using context]: /content/drive/MyDrive/Cancer_gene_database/artifacts/20250820_191143/calibrated_predictions_cv.csv

[BLOCKER] Codon/context tables don’t overlap on (variant_id, split, label).


In [31]:
# Cell — diagnose & align TP53 LOCO-site codon vs context preds (no deltas)
import re, time, hashlib
from pathlib import Path
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# ---- paths from your last message
CODON_P   = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/preds_wt_only_control_LOCO-SITE.csv")
CONTEXT_P = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/20250820_191143/calibrated_predictions_cv.csv")
SPLITMAP  = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv")

def read_any(p, nrows=None):
    s=str(p).lower()
    sep = "\t" if s.endswith(".tsv") or s.endswith(".tsv.gz") else ","
    return pd.read_csv(p, sep=sep, nrows=nrows)

def coerce_split(series):
    s = series.astype(str).str.extract(r"(-?\d+)")[0].astype("Int64")
    return s.map({1:0,2:1,3:2,4:3,5:4}).fillna(s).astype("Int64")

def pick_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for k in candidates:
        if k in cols: return cols[k]
    return None

def normalize_label(df, label_col):
    v = df[label_col].astype(str).str.lower().str.strip()
    map1 = {"driver":1,"oncogenic":1,"pathogenic":1,"likely_pathogenic":1,"lp":1,"pos":1,"true":1,"1":1,
            "benign":0,"likely-benign":0,"likely_benign":0,"lb":0,"neutral":0,"neg":0,"false":0,"0":0}
    if v.isin(map1).all():
        df["label"]=v.map(map1).astype(int); return True
    try:
        vi = pd.to_numeric(df[label_col], errors="raise")
        if set(vi.dropna().unique()).issubset({0,1}):
            df["label"]=vi.astype(int); return True
    except Exception:
        pass
    return False

def make_chr_key(df):
    need = {c.lower(): c for c in df.columns}
    req = ["chr","chrom","chromosome"], ["pos","position"], ["ref","ref_allele"], ["alt","alt_allele","alt"]
    def pick(opts):
        for o in opts:
            if o in need: return need[o]
        return None
    c = pick(req[0]); p = pick(req[1]); r = pick(req[2]); a = pick(req[3])
    if not all([c,p,r,a]): return None
    key = ("chr" + df[c].astype(str).str.replace("^chr","",regex=True).str.strip() + ":" +
           pd.to_numeric(df[p], errors="coerce").astype("Int64").astype(str) + "_" +
           df[r].astype(str).str.upper() + ">" + df[a].astype(str).str.upper())
    return key

def hgvs_core(x):
    x = str(x)
    i = x.find("c.")
    return x[i:] if i>=0 else x

def try_keys(A, B, namesA):
    # returns (key_name, a_series, b_series, overlap_count)
    trials = []

    # 1) variant_id direct
    va = pick_col(A, {"variant_id"})
    vb = pick_col(B, {"variant_id"})
    if va and vb:
        aa = A[va].astype(str)
        bb = B[vb].astype(str)
        trials.append(("variant_id", aa, bb, len(set(aa).intersection(set(bb)))))

        # 1b) strip transcript prefixes (HGVS core)
        aa2 = aa.apply(hgvs_core)
        bb2 = bb.apply(hgvs_core)
        trials.append(("variant_id_hgvs_core", aa2, bb2, len(set(aa2).intersection(set(bb2)))))

    # 2) HGVS columns
    ha = pick_col(A, {"hgvs_c","hgvsc","c_hgvs"})
    hb = pick_col(B, {"hgvs_c","hgvsc","c_hgvs"})
    if ha and hb:
        aa = A[ha].astype(str).apply(hgvs_core)
        bb = B[hb].astype(str).apply(hgvs_core)
        trials.append(("hgvs_c", aa, bb, len(set(aa).intersection(set(bb)))))

    # 3) chr:pos_ref>alt
    ka = make_chr_key(A)
    kb = make_chr_key(B)
    if ka is not None and kb is not None:
        trials.append(("chrpos_refalt", ka, kb, len(set(ka).intersection(set(kb)))))

    # 4) protein (+ transcript) fallback
    pa = pick_col(A, {"protein","hgvsp"})
    pb = pick_col(B, {"protein","hgvsp"})
    ta = pick_col(A, {"transcript","nm","ensembl_transcript"})
    tb = pick_col(B, {"transcript","nm","ensembl_transcript"})
    if pa and pb:
        aa = A[pa].astype(str)
        bb = B[pb].astype(str)
        if ta and tb:
            aa = aa + "|" + A[ta].astype(str)
            bb = bb + "|" + B[tb].astype(str)
            trials.append(("protein|transcript", aa, bb, len(set(aa).intersection(set(bb)))))
        trials.append(("protein_only", A[pa].astype(str), B[pb].astype(str),
                       len(set(A[pa].astype(str)).intersection(set(B[pb].astype(str))))))

    if not trials:
        return None

    trials.sort(key=lambda t: t[3], reverse=True)
    best = trials[0]
    print("\nKey trial overlaps (top 4):")
    for t in trials[:4]:
        print(f"  - {t[0]}: overlap={t[3]}")
    return best

# ---- load
codon = read_any(CODON_P)
ctx   = read_any(CONTEXT_P)
splitmap = read_any(SPLITMAP)
assert isinstance(codon, pd.DataFrame) and isinstance(ctx, pd.DataFrame), "Could not read CSVs."

# filter to TP53 if gene column exists
for df, nm in [(codon,"codon"), (ctx,"context")]:
    g = pick_col(df, {"gene"})
    if g:
        before=len(df); df.query(f"`{g}`=='TP53'", inplace=True)
        print(f"[{nm}] TP53 filter: {len(df)}/{before}")

# normalize split (if present) and label
for df, nm in [(codon,"codon"), (ctx,"context")]:
    s = pick_col(df, {"split","fold","cv_fold","site_split","site_fold"})
    if s:
        df["split"] = coerce_split(df[s])
    lab = pick_col(df, {"label","y","target","oncogenic","class","is_driver","truth","y_true"})
    if lab and normalize_label(df, lab):
        pass
    else:
        print(f"[{nm}] no usable label column detected (that’s OK for now).")

# detect score columns
CODON_KEYS   = {"p_wt","codon","codon_score","score_codon","baseline","p_wt_score","y_pred_wt","wt_pred"}
CONTEXT_KEYS = {"context","context_score","score_context","p_ctx","p_context","y_pred_ctx","ctx_pred","blend","bio"}
codon_col  = pick_col(codon, CODON_KEYS)
context_col= pick_col(ctx, CONTEXT_KEYS)
print("codon score col:", codon_col)
print("context score col:", context_col)

# try to find a join key that overlaps
best = try_keys(codon, ctx, codon.columns)
if best is None or best[3]==0:
    print("\n[BLOCKER] No overlapping join key across files. We need a unified LOCO-site preds CSV with both scores.")
else:
    key_name, a_key, b_key, overlap = best
    codon = codon.assign(_key=a_key.values)
    ctx   = ctx.assign(_key=b_key.values)
    # add LOCO-site splits from splitmap if needed
    sp_col = pick_col(splitmap, {"split","fold","cv_fold","site_split","site_fold"})
    vi_col = pick_col(splitmap, {"variant_id"})
    if vi_col is None:
        # try to build same key in splitmap
        sm_key = None
        if key_name in {"variant_id","variant_id_hgvs_core"} and "variant_id" in map(str.lower, splitmap.columns):
            sm_key = splitmap[vi_col].astype(str).apply(hgvs_core if key_name=="variant_id_hgvs_core" else str)
        else:
            sm_key = make_chr_key(splitmap)
        if sm_key is not None:
            splitmap = splitmap.assign(_key=sm_key)
            vi_col = "_key"
    else:
        # splitmap has variant_id; if our key is variant_id or hgvs_core on variant_id, construct _key accordingly
        if key_name.startswith("variant_id"):
            sm_series = splitmap[vi_col].astype(str)
            if key_name=="variant_id_hgvs_core":
                sm_series = sm_series.apply(hgvs_core)
            splitmap = splitmap.assign(_key=sm_series)
            vi_col = "_key"
        else:
            # last resort attempt
            sm_key = make_chr_key(splitmap)
            if sm_key is not None:
                splitmap = splitmap.assign(_key=sm_key)
                vi_col = "_key"

    assert vi_col is not None, "Could not create a compatible key in splitmap."

    # Join both to splitmap to enforce LOCO-site and add numeric split
    splitmap["split"] = coerce_split(splitmap[sp_col]) if sp_col else splitmap["split"]
    sm = splitmap[["_key","split"]].dropna().drop_duplicates()
    codon_sm = codon.merge(sm, left_on="_key", right_on="_key", how="inner")
    ctx_sm   = ctx.merge(sm,   left_on="_key", right_on="_key", how="inner")

    # inner-join codon & context on key+split
    merged = codon_sm.merge(ctx_sm[["_key","split",context_col]], on=["_key","split"], how="inner", suffixes=("","_ctx"))
    if merged.empty:
        print("\n[BLOCKER] After key alignment, no overlap on (_key, split).")
    else:
        # rename columns to standard names
        if "label" not in merged.columns:
            # try to import label from codon side if it existed with another name
            lab = pick_col(merged, {"label","y","target","oncogenic","class","is_driver","truth","y_true"})
            if lab and lab!="label":
                merged = merged.rename(columns={lab:"label"})
        # sanity split set
        print("\nOverlaps on (_key, split):", len(merged))
        print("Splits present:", sorted(set(int(x) for x in merged["split"].dropna().unique())))
        # Metric calc if label present and both scores found
        if "label" in merged.columns and codon_col and context_col:
            y  = merged["label"].astype(int).values
            merged = merged.rename(columns={codon_col:"p_wt", context_col:"context"})
            rows=[]
            for sp, sub in merged.groupby("split"):
                y = sub["label"].astype(int).values
                pw = pd.to_numeric(sub["p_wt"], errors="coerce").values
                pc = pd.to_numeric(sub["context"], errors="coerce").values
                def safe_pr(y,p):
                    return average_precision_score(y,p) if len(set(y))==2 else float("nan")
                au_w = roc_auc_score(y,pw) if len(set(y))==2 else float("nan")
                au_c = roc_auc_score(y,pc) if len(set(y))==2 else float("nan")
                pr_w = safe_pr(y,pw); pr_c = safe_pr(y,pc)
                br_w = brier_score_loss(y,pw); br_c = brier_score_loss(y,pc)
                rows.append({"split":int(sp),
                             "n":len(y), "n_pos":int(y.sum()), "n_neg":int((1-y).sum()),
                             "AUROC_codon":au_w,"AUROC_context":au_c,"ΔAUROC":(au_c-au_w) if (au_c==au_c and au_w==au_w) else float("nan"),
                             "PR_codon":pr_w,"PR_context":pr_c,"ΔPR":(pr_c-pr_w) if (pr_c==pr_c and pr_w==pr_w) else float("nan"),
                             "Brier_codon":br_w,"Brier_context":br_c,"ΔBrier":(br_c-br_w)})
            full = pd.DataFrame(rows).sort_values("split")
            out = CONTEXT_P.parent / "tp53_per_fold_with_counts_full.csv"
            full.to_csv(out, index=False)
            print("\n[OK] Wrote:", out)
            print(full.to_string(index=False))
        else:
            print("\n[INFO] Merged rows exist, but missing a label or one of the scores. Show sample columns:")
            print("merged columns:", list(merged.columns)[:25])
            print("has label?", "label" in merged.columns, "| codon_col:", codon_col, "| context_col:", context_col)

# If all of the above fails, print regen spec
if 'merged' not in globals() or merged.empty:
    print("\n=== REGEN SPEC (minimal) ===")
    print("Produce a single TP53 LOCO-site preds CSV with:")
    print("  variant_id, split (0..4), label ∈ {0,1}, p_wt, context")
    print("Use the **same split map**:", SPLITMAP)
    print("Save under the run, e.g.: .../calibrated_robust_20250820_000001/ci_pairable/preds_loco_site_tp53.csv")


[codon] TP53 filter: 197/271
[context] TP53 filter: 4/16
codon score col: y_pred_wt
context score col: p_ctx

Key trial overlaps (top 4):
  - variant_id: overlap=0
  - variant_id_hgvs_core: overlap=0

[BLOCKER] No overlapping join key across files. We need a unified LOCO-site preds CSV with both scores.

=== REGEN SPEC (minimal) ===
Produce a single TP53 LOCO-site preds CSV with:
  variant_id, split (0..4), label ∈ {0,1}, p_wt, context
Use the **same split map**: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv
Save under the run, e.g.: .../calibrated_robust_20250820_000001/ci_pairable/preds_loco_site_tp53.csv


In [32]:
# Hunt for TP53 LOCO-site preds with BOTH p_wt/context under the RUN dir.
# If none found, emit a SKELETON file with schema:
#   variant_id, split (0..4), label ∈ {0,1}   # fill p_wt, context downstream
import re, time, hashlib
from pathlib import Path
import pandas as pd

def md5sum(p: Path):
    h = hashlib.md5()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

RUN_DIR   = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001")
SPLITMAP  = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv")
LABELS    = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv")
OUT_DIR   = RUN_DIR / "ci_pairable"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CODON_KEYS   = {"p_wt","codon","codon_score","score_codon","baseline","p_wt_score","y_pred_wt","wt_pred"}
CONTEXT_KEYS = {"context","context_score","score_context","p_ctx","p_context","y_pred_ctx","ctx_pred","blend","bio"}
SPLIT_KEYS   = {"split","fold","cv_fold","site_split","site_fold"}
LABEL_KEYS   = {"label","y","target","oncogenic","class","is_driver","truth","y_true"}

def read_any(p, nrows=None):
    s=str(p).lower()
    sep = "\t" if s.endswith(".tsv") or s.endswith(".tsv.gz") else ","
    try:
        return pd.read_csv(p, sep=sep, nrows=nrows)
    except Exception:
        return None

def coerce_split(series):
    s = series.astype(str).str.extract(r"(-?\d+)")[0].astype("Int64")
    return s.map({1:0,2:1,3:2,4:3,5:4}).fillna(s).astype("Int64")

def normalize_label(df, label_col):
    v = df[label_col].astype(str).str.lower().str.strip()
    map1 = {"driver":1,"oncogenic":1,"pathogenic":1,"likely_pathogenic":1,"lp":1,"pos":1,"true":1,"1":1,
            "benign":0,"likely-benign":0,"likely_benign":0,"lb":0,"neutral":0,"neg":0,"false":0,"0":0}
    if v.isin(map1).all():
        df["label"] = v.map(map1).astype(int); return True
    try:
        vi = pd.to_numeric(df[label_col], errors="raise")
        if set(vi.dropna().unique()).issubset({0,1}):
            df["label"] = vi.astype(int); return True
    except Exception:
        pass
    return False

def find_both_scores_under_run(run_dir: Path):
    hits=[]
    for p in run_dir.rglob("*"):
        if not p.is_file(): continue
        low=str(p).lower()
        if not (low.endswith(".csv") or low.endswith(".tsv") or low.endswith(".csv.gz") or low.endswith(".tsv.gz")):
            continue
        if "delta" in low or "loco_gene" in low:
            continue  # skip delta-only/LOGO
        df = read_any(p)
        if df is None or df.empty: continue
        cols = {c.lower(): c for c in df.columns}
        if "variant_id" not in cols: continue
        split_col = next((cols[k] for k in SPLIT_KEYS if k in cols), None)
        label_col = next((cols[k] for k in LABEL_KEYS if k in cols), None)
        if not split_col or not label_col: continue
        codon_col   = next((cols[k] for k in CODON_KEYS   if k in cols), None)
        context_col = next((cols[k] for k in CONTEXT_KEYS if k in cols), None)
        if not (codon_col and context_col): continue

        # normalize
        df["split"] = coerce_split(df[split_col])
        if not set(df["split"].dropna().unique()).issubset({0,1,2,3,4}): continue
        if not normalize_label(df, label_col): continue

        # TP53 filter if possible
        gcols = [c for c in df.columns if c.lower()=="gene"]
        if gcols:
            df = df[df[gcols[0]]=="TP53"]
            if df.empty: continue

        hits.append((p, df.rename(columns={codon_col:"p_wt", context_col:"context"})))
    return hits

# 1) Try to find a valid preds table under RUN
hits = find_both_scores_under_run(RUN_DIR)
print("RUN-scope preds (both scores) candidates:", len(hits))
if hits:
    # pick newest
    hits.sort(key=lambda x: -x[0].stat().st_mtime)
    pred_p, pred_df = hits[0]
    print("[using preds]:", pred_p)
    # enforce LOCO-site split assignments from SPLITMAP (if present)
    sm = read_any(SPLITMAP)
    if sm is not None and "variant_id" in {c.lower() for c in sm.columns}:
        sp_col = next((c for c in sm.columns if c.lower() in SPLIT_KEYS), None)
        if sp_col:
            sm = sm.rename(columns={sp_col:"split"})
            sm["split"] = coerce_split(sm["split"])
            sp = sm[["variant_id","split"]].dropna().drop_duplicates()
            pred_df = pred_df.merge(sp, on=["variant_id","split"], how="inner")

    # keep splits 0..4 only
    pred_df = pred_df[pred_df["split"].isin([0,1,2,3,4])].copy()
    # emit
    out = OUT_DIR / "preds_loco_site_tp53.csv"
    pred_df[["variant_id","split","label","p_wt","context"]].to_csv(out, index=False)
    print("\n[OK] Unified preds exist. Wrote:", out, "| md5:", md5sum(out))
else:
    print("[info] No unified preds with both scores found under RUN. Creating SKELETON to score...")

    # 2) Build SKELETON from split-map + labels (all 5 splits)
    sm = read_any(SPLITMAP); assert sm is not None, f"split map missing: {SPLITMAP}"
    lb = read_any(LABELS);   assert lb is not None, f"labels missing: {LABELS}"

    # normalize split + label
    sp_col = next((c for c in sm.columns if c.lower() in SPLIT_KEYS), None)
    assert sp_col, "split col not found in split map"
    sm = sm.rename(columns={sp_col:"split"})
    sm["split"] = coerce_split(sm["split"])
    # keep only 0..4
    sm = sm[sm["split"].isin([0,1,2,3,4])]

    lab_col = next((c for c in lb.columns if c.lower() in LABEL_KEYS), None)
    assert lab_col, "label-like col not found in labels file"
    # TP53 filter if gene present
    gcols = [c for c in lb.columns if c.lower()=="gene"]
    if gcols:
        lb = lb[lb[gcols[0]]=="TP53"].copy()
        assert not lb.empty, "labels file has no TP53 rows"
    assert normalize_label(lb, lab_col), "couldn't normalize labels to {0,1}"

    # join
    skel = sm.merge(lb[["variant_id","label"]], on="variant_id", how="inner")[["variant_id","split","label"]]
    assert not skel.empty, "no overlap between split map and labels"
    # emit SKELETON
    out = OUT_DIR / "preds_loco_site_tp53_SKELETON.csv"
    skel.to_csv(out, index=False)
    print("\n[OK] Wrote SKELETON:", out, "| md5:", md5sum(out))
    print("Rows per split (should include 3):")
    print(skel.groupby("split")["label"].agg(n="size", n_pos="sum").to_frame().assign(n_neg=lambda x:x["n"]-x["n_pos"]))
    print("\nNEXT: score this SKELETON with BOTH models from RUN 'calibrated_robust_20250820_000001',")
    print("      append columns 'p_wt' and 'context', keep splits 0..4, and save as:")
    print(f"      {OUT_DIR / 'preds_loco_site_tp53.csv'}")


RUN-scope preds (both scores) candidates: 0
[info] No unified preds with both scores found under RUN. Creating SKELETON to score...


AttributeError: 'DataFrame' object has no attribute 'str'

In [33]:
# Build TP53 LOCO-site SKELETON (variant_id, split, label) — robust to duplicate "split" cols
import pandas as pd, hashlib, time, re
from pathlib import Path

RUN_DIR   = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001")
SPLITMAP  = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv")
LABELS    = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv")
OUT_DIR   = RUN_DIR / "ci_pairable"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def md5sum(p: Path):
    h = hashlib.md5()
    with open(p,'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''): h.update(chunk)
    return h.hexdigest()

def read_any(p, nrows=None):
    s=str(p).lower()
    sep = "\t" if s.endswith(".tsv") or s.endswith(".tsv.gz") else ","
    return pd.read_csv(p, sep=sep, nrows=nrows)

def pick_col(df, cands):
    cols = {c.lower(): c for c in df.columns}
    for k in cands:
        if k in cols: return cols[k]
    return None

def extract_series(df, colname):
    obj = df[colname]
    if isinstance(obj, pd.DataFrame):  # duplicate column names → DataFrame
        return obj.iloc[:, 0]
    return obj

def coerce_split_any(x):
    s = pd.Series(x).astype(str).str.extract(r"(-?\d+)")[0].astype("Int64")
    return s.map({1:0,2:1,3:2,4:3,5:4}).fillna(s).astype("Int64")

def normalize_label(df, label_col):
    v = df[label_col].astype(str).str.lower().str.strip()
    map1 = {
        "driver":1,"oncogenic":1,"pathogenic":1,"likely_pathogenic":1,"lp":1,"pos":1,"true":1,"1":1,
        "benign":0,"likely-benign":0,"likely_benign":0,"lb":0,"neutral":0,"neg":0,"false":0,"0":0
    }
    if v.isin(map1).all():
        df["label"] = v.map(map1).astype(int); return True
    try:
        vi = pd.to_numeric(df[label_col], errors="raise")
        if set(vi.dropna().unique()).issubset({0,1}):
            df["label"] = vi.astype(int); return True
    except Exception:
        pass
    return False

# 1) load split-map + labels
sm = read_any(SPLITMAP)
lb = read_any(LABELS)

# 2) pick columns safely
vi_col_sm = pick_col(sm, {"variant_id"})
sp_col_sm = pick_col(sm, {"split","fold","cv_fold","site_split","site_fold"})
assert vi_col_sm and sp_col_sm, f"split map missing variant_id or split-like col; cols={list(sm.columns)}"

vi_sm = extract_series(sm, vi_col_sm).astype(str)
sp_sm = coerce_split_any(extract_series(sm, sp_col_sm))
sm2 = pd.DataFrame({"variant_id": vi_sm, "split": sp_sm})
sm2 = sm2.dropna().drop_duplicates(subset=["variant_id","split"])
sm2 = sm2[sm2["split"].isin([0,1,2,3,4])]

# 3) normalize labels; TP53 filter if gene col exists
vi_col_lb = pick_col(lb, {"variant_id"})
assert vi_col_lb, f"labels missing variant_id; cols={list(lb.columns)}"
gene_col  = pick_col(lb, {"gene"})
if gene_col:
    before = len(lb)
    lb = lb[lb[gene_col]=="TP53"].copy()
    assert not lb.empty, f"labels has no TP53 rows (from {before})"

lab_col = pick_col(lb, {"label","y","target","oncogenic","class","is_driver","truth","y_true"})
assert lab_col and normalize_label(lb, lab_col), "couldn't normalize labels to {0,1}"

lb2 = lb[[vi_col_lb, "label"]].rename(columns={vi_col_lb:"variant_id"}).drop_duplicates()

# 4) join → expect all splits incl. 3
skel = sm2.merge(lb2, on="variant_id", how="inner")
assert not skel.empty, "no overlap between split map and labels"
present_splits = sorted(skel["split"].unique().tolist())
print("splits in SKELETON:", present_splits)

# 5) emit
out = OUT_DIR / "preds_loco_site_tp53_SKELETON.csv"
skel[["variant_id","split","label"]].to_csv(out, index=False)
print("[OK] wrote SKELETON:", out)
print("md5:", md5sum(out))
print("\nrows per split:")
print(skel.groupby("split")["label"].agg(n="size", n_pos="sum")
      .to_frame().assign(n_neg=lambda x: x["n"]-x["n_pos"]))
print("\nNEXT: score this SKELETON with BOTH models from RUN 'calibrated_robust_20250820_000001',")
print("      append columns 'p_wt' and 'context', keep splits 0..4, and save as:")
print(f"      {OUT_DIR / 'preds_loco_site_tp53.csv'}")


AssertionError: no overlap between split map and labels

# TP53 LOCO-site Revival — Baton Pass (receipts-first)

## Mission (unchanged)
Prove (for **TP53**) that **local nucleotide context > codon-only** under **LOCO-site** CV. Lock the mapping and deliver pub-grade Methods + Results with **per-fold counts** and full provenance. If possible, **recover fold 3** for **n=5** stats.

---

## Locked State
- **RUN_ID:** `calibrated_robust_20250820_000001`
- **Mapping (frozen):** `{gene: TP53, codon_col: p_wt, context_col: context}`
- **Policy:** LOCO-site only; **exclude LOGO**; **no imputation**; bootstrap seeds fixed (**AUROC=42, PR=43**).

---

## Per-Fold Counts (strict LOCO-site; folds {0,1,2,4})
- **Split source:** `/.../sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv`
- **Labels used for counts:** `/.../calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv` (both classes; worked for counts)
- **Counts CSV:** `/.../sentiment_run_plus_variants_20250816_190607/tp53_counts_strict_loco.csv`  
  **md5:** `0d572eb9dcfe7465c14ab8dd4085cfc3`
- **Table (each of splits 0,1,2,4 identical):** `n=25, n_pos=11, n_neg=14`

---

## Metrics I/O (strict only)
- **Input metrics:** `/.../calibrated_robust_20250820_000001/ci_pairable/tp53_per_fold_metrics.csv`  
  **md5:** `652075be1faf88b14fd84862292d5001`
- **Output (counts merged, strict):** `/.../calibrated_robust_20250820_000001/ci_pairable/tp53_per_fold_metrics_with_counts_strict.csv`  
  **md5:** `a4f9bb130646f4cfc0df72298394a2b4`

---

## Provenance Docs (updated)
- **README (sources + methods + counts):** `/.../ci_pairable/README_TP53_LOCO_sources_and_methods_calibrated_robust_20250820_000001_20250822.md`  
  **md5:** `b387825e100495e0443c0d9b72aa42ca`
- **Strict summary JSON (now includes counts/provenance):** `/.../ci_pairable/tp53_loco_strict_summary.json`  
  **md5:** `88e47e48a115f7ec3e94bbff37673fc6`

---

## Stats (current, unchanged)
- **Strict LOCO (n=4):** ΔAUROC **+0.338** [0.125–0.550], ΔPR **+0.383** [0.108–0.708]; **Wilcoxon (one-sided)** p ≈ **0.0625**.
- **If fold 3 recovered (n=5):** ΔAUROC **+0.287** [0.093–0.482], ΔPR **+0.316** [0.096–0.576]; **p = 0.03125**.

---

## Blocker (diagnosed)
No **unified LOCO-site predictions CSV** with **both** `p_wt` (codon) **and** `context` for TP53 across splits **0..4**.
- Found a delta-only preds file → **not usable** for separate model metrics.
- Found codon/context preds from **different universes** (IDs/splits misaligned) → **0 overlap** on `(variant_id, split, label)`.
- Skeleton builders tried (keys: `variant_id`, `hgvs_core`, `chr:pos_ref>alt`, `protein±transcript`). Best label table overlap was with  
  `.../final_receipts_20250816_232227/deploy_scored.csv`, but join still failed → **no unified preds today**.

---

## Next Chat — Do This In Order
1. **Create the unified TP53 LOCO-site preds file** (keystone).  
   **Schema:** `variant_id, split(0..4), label{0,1}, p_wt, context`  
   **Split assignments:** `/.../sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv`  
   **Save to:** `/.../calibrated_robust_20250820_000001/ci_pairable/preds_loco_site_tp53.csv`
   - If alignment keeps failing, **bypass joins**: score the **exact variant list** from the split-map directly with **both models from this RUN**, then attach labels (use the **same label source** as counts).
2. **Recompute per-fold metrics (n=5)** from the unified preds.  
   Per fold: AUROC / PR / Brier for **codon vs context**, plus deltas.  
   Aggregate: **10k bootstrap 95% CIs** (seeds AUROC=42, PR=43) + **one-sided Wilcoxon** on per-fold deltas.  
   Update `tp53_loco_strict_summary.json` with the **n=5** line (keep **n=4** for transparency).
3. **Calibration & scoring quality (TP53, LOCO-site)**  
   Reliability curves (pre/post), ΔBrier, ECE. Export figs + CSVs (stamp seeds + paths).
4. **Clinical utility (TP53)**  
   **Decision Curve Analysis**: net benefit vs clinical thresholds; **codon vs context**.
5. **Leakage & split integrity**  
   Emit a small **audit table** proving **no site leakage** (WT/MUT co-stay) and that **counts line up** with metrics.
6. **Robustness ablations (minimal, decisive)**  
   - Shuffle flanks → effect drops.  
   - Mask CpG/TpC → effect persists.  
   - Flank length ±30/±60/±90 → effect stable.
7. **Repro pack**  
   Per-fold predictions CSV (include `site_id` if available), env lock (`requirements.txt`), RUN_ID, **all paths + md5**.

---

## Hard Constraints (do not violate)
- **LOCO-site only**; **exclude LOGO** rows.
- **No imputation** of missing folds.
- **Seeds fixed:** AUROC=42, PR=43.
- **Mapping locked:** `{gene: TP53, codon_col: p_wt, context_col: context}`.

---

## Files for Next Chat
- `/.../ci_pairable/tp53_per_fold_metrics.csv` — md5 `652075be1faf88b14fd84862292d5001`  
- `/.../ci_pairable/tp53_per_fold_metrics_with_counts_strict.csv` — md5 `a4f9bb130646f4cfc0df72298394a2b4`  
- `/.../ci_pairable/tp53_loco_strict_summary.json` — md5 `88e47e48a115f7ec3e94bbff37673fc6`  
- `/.../ci_pairable/README_TP53_LOCO_sources_and_methods_calibrated_robust_20250820_000001_20250822.md` — md5 `b387825e100495e0443c0d9b72aa42ca`  
- `/.../sentiment_run_plus_variants_20250816_190607/tp53_counts_strict_loco.csv` — md5 `0d572eb9dcfe7465c14ab8dd4085cfc3`  
- `/.../sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv` (split source)

---

## One-liner goal (paste into new chat)
Finalize TP53 evidence that **context > codon** under **LOCO-site** CV by producing a **unified preds CSV** with both scores for splits **0–4**, recomputing **n=5** per-fold metrics with **bootstrap CIs** and **Wilcoxon**, and delivering **pub-grade Methods + Results** (counts, calibration, DCA, robustness, full provenance) under **RUN `calibrated_robust_20250820_000001`**.


In [36]:
# TP53 LOCO-site SKELETON (variant_id, split, label) — duplicate "split" cols safe
# Policy locks: LOCO-site only; exclude LOGO; mapping fixed {gene: TP53, codon_col: p_wt, context_col: context}

import pandas as pd
import re, hashlib, json
from pathlib import Path

# ====== CONFIG (edit these) ======
PATH_SPLIT_MAP = "/.../sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv"
# pick ONE primary label source that actually has both classes for TP53
# (use the one that worked for counts; keep a fallback for deploy_scored)
PATH_LABELS_PRIMARY = "/.../calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv"
PATH_LABELS_FALLBACK = "/.../final_receipts_20250816_232227/deploy_scored.csv"

# where to write the skeleton
OUT_DIR = "/.../calibrated_robust_20250820_000001/ci_pairable"
OUT_FILE = "skeleton_loco_site_tp53.csv"

# optional sanity expectations (comment out if unknown)
EXPECTED_SPLITS = {0,1,2,4}  # will also accept 3 if present
EXPECTED_PER_SPLIT = 25
EXPECTED_POS = 11
EXPECTED_NEG = 14

# ====== HELPERS ======
def md5sum(p: Path) -> str:
    h = hashlib.md5()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def coalesce_split_cols(df: pd.DataFrame, prefer="split") -> pd.DataFrame:
    # gather any column named split, split_*, *_split (case-insensitive)
    cand = [c for c in df.columns if re.fullmatch(r"(?i)(split|.*_split|split_.*)", c)]
    if not cand and "fold" in df.columns:
        cand = ["fold"]  # fallback common name
    if not cand:
        raise AssertionError("No split/fold column found in split-map.")
    # coalesce, prefer exact 'split' if exists
    ordered = sorted(cand, key=lambda c: (0 if c.lower()==prefer else 1, c))
    tmp = pd.concat([df[c] for c in ordered], axis=1)
    df["split"] = tmp.bfill(axis=1).iloc[:,0]
    # enforce int, drop extras
    df["split"] = pd.to_numeric(df["split"], errors="coerce").astype("Int64")
    drop_cols = [c for c in cand if c != "split"]
    return df.drop(columns=drop_cols, errors="ignore")

def normalize_variant_id(df: pd.DataFrame) -> pd.DataFrame:
    # prefer existing variant_id if present and non-null
    if "variant_id" in df.columns and df["variant_id"].notna().any():
        return df.assign(variant_id=df["variant_id"].astype(str))
    # try common cores
    for key in ["variant_id_hgvs_core", "hgvs_core", "hgvs_c", "hgvs_nt", "hgvs"]:
        if key in df.columns and df[key].notna().any():
            return df.assign(variant_id=df[key].astype(str))
    # chr/pos/ref/alt pattern
    cols = {c.lower(): c for c in df.columns}
    need = ["chr","pos","ref","alt"]
    if all(k in cols for k in need):
        cid, pid, rid, aid = (cols["chr"], cols["pos"], cols["ref"], cols["alt"])
        vid = df[cid].astype(str)+":"+df[pid].astype(str)+"_"+df[rid].astype(str)+">"+df[aid].astype(str)
        return df.assign(variant_id=vid)
    raise AssertionError("Could not derive variant_id — provide one of {variant_id, hgvs_* or chr/pos/ref/alt}.")

def ensure_label01(df: pd.DataFrame) -> pd.DataFrame:
    """
    Map labels to {0,1}. Heuristics:
      - 'label' already 0/1
      - categorical columns like {'pos','neg'} or {'driver','benign'}
      - numeric 'oncogenic' or boolean flags
      - 'call' with prefixes like 'pos_', 'neg'
    """
    cand = [c for c in df.columns if c.lower() in {"label","class","y","oncogenic","is_driver","is_pathogenic","call"}]
    if not cand:
        raise AssertionError("No obvious label column. Seen cols: "+", ".join(df.columns[:20].tolist()))
    # try in order
    for c in cand:
        s = df[c]
        # numeric/bool-like
        if pd.api.types.is_bool_dtype(s) or pd.api.types.is_numeric_dtype(s):
            mapped = s.astype("Int64")
            # normalize possible {-1,1} -> {0,1}
            uniq = set(mapped.dropna().unique().tolist())
            if uniq <= {0,1}:
                return df.rename(columns={c:"label"}).assign(label=mapped.astype("Int64"))
            if uniq <= {-1,1}:
                return df.rename(columns={c:"label"}).assign(label=((mapped>0).astype("Int64")))
        # string/categorical
        ser = s.astype(str).str.lower()
        if sorted(set(ser.unique()) - {"nan"})[:20]:  # just touch
            pos_tokens = {"pos","positive","driver","pathogenic","oncogenic","pos_high","likely_pathogenic"}
            neg_tokens = {"neg","negative","benign","likely_benign","wt","neutral"}
            if ser.isin(pos_tokens|neg_tokens).any():
                label = ser.map(lambda x: 1 if x in pos_tokens else (0 if x in neg_tokens else pd.NA)).astype("Int64")
                if label.notna().any():
                    out = df.rename(columns={c:"_raw_label"}).assign(label=label)
                    return out
            # 'call' like 'pos_high' / 'neg'
            if c.lower()=="call":
                label = ser.map(lambda x: 1 if x.startswith("pos") else (0 if x.startswith("neg") else pd.NA)).astype("Int64")
                if label.notna().any():
                    return df.rename(columns={c:"_raw_call"}).assign(label=label)
    raise AssertionError("Failed to coerce labels to {0,1}. Inspect label source.")

def filter_tp53_loco(df: pd.DataFrame) -> pd.DataFrame:
    # accept gene filter if present
    if "gene" in df.columns:
        df = df[df["gene"].astype(str).str.upper()=="TP53"].copy()
    # exclude non-LOCO if a marker exists
    # common fields to screen out: 'cv', 'mode', 'tag', 'schema'
    for col in [c for c in df.columns if c.lower() in {"cv","mode","schema","tag"}]:
        mask_logo = df[col].astype(str).str.contains("logo", case=False, na=False)
        if mask_logo.any():
            df = df[~mask_logo].copy()
    return df

def audit_site_leakage(df: pd.DataFrame):
    # optional: if site_id exists, assert one split per site
    if "site_id" in df.columns:
        g = df.groupby("site_id")["split"].nunique()
        bad = g[g>1]
        if not bad.empty:
            raise AssertionError(f"Site leakage detected for site_id(s): {bad.index.tolist()}")

# ====== LOAD & CLEAN SPLIT MAP ======
split = pd.read_csv(PATH_SPLIT_MAP)
split = filter_tp53_loco(split)
split = normalize_variant_id(split)
split = coalesce_split_cols(split)
split = split[["variant_id","split"]].dropna().drop_duplicates()

# ====== LOAD LABELS (primary → fallback) ======
def load_labels_any(path):
    df = pd.read_csv(path)
    df = filter_tp53_loco(df)
    df = normalize_variant_id(df)
    df = ensure_label01(df)
    return df[["variant_id","label"]].dropna().drop_duplicates()

labels = None
errs = []
for path in [PATH_LABELS_PRIMARY, PATH_LABELS_FALLBACK]:
    try:
        labels = load_labels_any(path)
        label_src = path
        break
    except Exception as e:
        errs.append(f"{path}: {e}")
if labels is None:
    raise RuntimeError("Failed to load labels from both sources:\n" + "\n".join(errs))

# ====== MERGE (robust to duplicate split columns from prior ops) ======
skel = split.merge(labels, on="variant_id", how="left", validate="m:1")
# guard: if a stray 'split_y' leaked in, coalesce again
skel = coalesce_split_cols(skel)
skel = skel[["variant_id","split","label"]]

# ====== SANITY CHECKS ======
# drop rows without labels (hard fail — skeleton must be label-complete)
missing = skel[skel["label"].isna()]
if not missing.empty:
    raise AssertionError(f"Missing labels for {len(missing)} variant(s). Example: {missing.head(5).to_dict(orient='records')}")

# ensure integer types
skel["split"] = skel["split"].astype(int)
skel["label"] = skel["label"].astype(int)

# optional: site leakage audit if site_id exists in split-map
try:
    audit_site_leakage(split.join(labels.set_index("variant_id"), on="variant_id"))
except AssertionError as e:
    raise

# expected splits (soft-check: allow 3 if present)
present = set(skel["split"].unique().tolist())
ex_ok = present.issubset(EXPECTED_SPLITS | {3}) if EXPECTED_SPLITS else True
if not ex_ok:
    print(f"[warn] unexpected splits present: {sorted(present)}")

# per-split counts
cnt = skel.groupby("split")["label"].value_counts().unstack(fill_value=0).rename(columns={0:"n_neg",1:"n_pos"})
cnt["n"] = cnt["n_neg"] + cnt["n_pos"]
print("Per-split counts:\n", cnt)

# strict expectations if configured
if EXPECTED_PER_SPLIT:
    bad = cnt[cnt["n"] != EXPECTED_PER_SPLIT]
    if not bad.empty:
        print("[warn] counts != expected per split; investigate:\n", bad)

if EXPECTED_POS is not None and 1 in skel["label"].unique():
    bad_pos = cnt[cnt["n_pos"] != EXPECTED_POS]
    if not bad_pos.empty:
        print("[warn] n_pos != expected; investigate:\n", bad_pos)

if EXPECTED_NEG is not None and 0 in skel["label"].unique():
    bad_neg = cnt[cnt["n_neg"] != EXPECTED_NEG]
    if not bad_neg.empty:
        print("[warn] n_neg != expected; investigate:\n", bad_neg)

# ====== WRITE OUT ======
out_path = Path(OUT_DIR) / OUT_FILE
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
skel.to_csv(out_path, index=False)
print(f"[saved] {out_path}  rows={len(skel)}  md5={md5sum(out_path)}")
print(json.dumps({
    "skeleton_path": str(out_path),
    "md5": md5sum(out_path),
    "label_source": label_src,
    "n_rows": len(skel),
    "splits_present": sorted(present),
}, indent=2))


FileNotFoundError: [Errno 2] No such file or directory: '/.../sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv'

In [37]:
# TP53 LOCO-site SKELETON with auto-discovery for paths
# Finds splits_loco_site.* and labels if you left placeholders.
# Policy: LOCO-site only; exclude LOGO; seeds fixed elsewhere.

import os, re, json, hashlib
from pathlib import Path
import pandas as pd

# ---------- CONFIG (you can leave these as None; discovery will try to find them) ----------
PATH_SPLIT_MAP = None  # e.g., "/content/drive/MyDrive/.../sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv"
PATH_LABELS_PRIMARY = None  # e.g., "/content/drive/MyDrive/.../calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv"
PATH_LABELS_FALLBACK = None  # will auto-search for deploy_scored.csv
OUT_DIR = "/content/drive/MyDrive/ci_pairable"  # adjust if you want
OUT_FILE = "skeleton_loco_site_tp53.csv"

EXPECTED_SPLITS = {0,1,2,4}  # will accept 3 if present
EXPECTED_PER_SPLIT = 25
EXPECTED_POS = 11
EXPECTED_NEG = 14

# ---------- MOUNT (safe if already mounted) ----------
try:
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
except Exception as e:
    print("[warn] drive mount:", e)

# ---------- HELPERS ----------
def md5sum(p: Path) -> str:
    h = hashlib.md5()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def find_candidates(names, roots=None, max_hits=2000):
    if isinstance(names, str):
        names = [names]
    if roots is None:
        roots = [
            Path("/content/drive/MyDrive"),
            Path("/content"),
            Path.cwd(),
        ]
    seen = []
    for root in roots:
        if not root.exists():
            continue
        for nm in names:
            # also consider any extension variant
            pats = [f"**/{nm}", f"**/{nm}.*"]
            for pat in pats:
                for p in root.glob(pat):
                    if p.is_file():
                        seen.append(p)
    # de-dup
    uniq = []
    sset = set()
    for p in seen:
        sp = str(p)
        if sp not in sset:
            sset.add(sp)
            uniq.append(p)
        if len(uniq) >= max_hits:
            break
    return uniq

def score_split_path(p: Path) -> int:
    s = 0
    s += 5 if p.name.lower().startswith("splits_loco_site") else 0
    s += 3 if p.suffix.lower() in {".csv",".tsv",".parquet"} else 0
    s += 2 if "sentiment_run_plus_variants" in str(p.parent) else 0
    # sibling counts file?
    sib = list(p.parent.glob("tp53_counts_strict_loco.*"))
    s += 2 if sib else 0
    s += 1 if p.stat().st_size > 0 else 0
    return s

def score_label_path(p: Path) -> int:
    s = 0
    s += 4 if "TP53_delta_top" in p.name else 0
    s += 3 if "deploy_scored" in p.name else 0
    s += 2 if "advantage" in str(p.parent) else 0
    s += 1 if p.stat().st_size > 0 else 0
    return s

def choose_best(cands, scorer):
    if not cands:
        return None
    cands = sorted(cands, key=lambda p: (-scorer(p), len(str(p))))
    return cands[0]

def coalesce_split_cols(df: pd.DataFrame, prefer="split") -> pd.DataFrame:
    cand = [c for c in df.columns if re.fullmatch(r"(?i)(split|.*_split|split_.*)", c)]
    if not cand and "fold" in df.columns:
        cand = ["fold"]
    if not cand:
        raise AssertionError("No split/fold column found in split-map.")
    ordered = sorted(cand, key=lambda c: (0 if c.lower()==prefer else 1, c))
    tmp = pd.concat([df[c] for c in ordered], axis=1)
    df["split"] = tmp.bfill(axis=1).iloc[:,0]
    df["split"] = pd.to_numeric(df["split"], errors="coerce").astype("Int64")
    drop_cols = [c for c in cand if c != "split"]
    return df.drop(columns=drop_cols, errors="ignore")

def normalize_variant_id(df: pd.DataFrame) -> pd.DataFrame:
    if "variant_id" in df.columns and df["variant_id"].notna().any():
        return df.assign(variant_id=df["variant_id"].astype(str))
    for key in ["variant_id_hgvs_core", "hgvs_core", "hgvs_c", "hgvs_nt", "hgvs"]:
        if key in df.columns and df[key].notna().any():
            return df.assign(variant_id=df[key].astype(str))
    cols = {c.lower(): c for c in df.columns}
    need = ["chr","pos","ref","alt"]
    if all(k in cols for k in need):
        cid, pid, rid, aid = (cols["chr"], cols["pos"], cols["ref"], cols["alt"])
        vid = df[cid].astype(str)+":"+df[pid].astype(str)+"_"+df[rid].astype(str)+">"+df[aid].astype(str)
        return df.assign(variant_id=vid)
    raise AssertionError("Could not derive variant_id — provide one of {variant_id, hgvs_* or chr/pos/ref/alt}.")

def ensure_label01(df: pd.DataFrame) -> pd.DataFrame:
    cand = [c for c in df.columns if c.lower() in {"label","class","y","oncogenic","is_driver","is_pathogenic","call"}]
    if not cand:
        raise AssertionError("No obvious label column. Seen cols: "+", ".join(df.columns[:20].tolist()))
    for c in cand:
        s = df[c]
        if pd.api.types.is_bool_dtype(s) or pd.api.types.is_numeric_dtype(s):
            mapped = s.astype("Int64")
            uniq = set(mapped.dropna().unique().tolist())
            if uniq <= {0,1}:
                return df.rename(columns={c:"label"}).assign(label=mapped.astype("Int64"))
            if uniq <= {-1,1}:
                return df.rename(columns={c:"label"}).assign(label=(mapped>0).astype("Int64"))
        ser = s.astype(str).str.lower()
        pos_tokens = {"pos","positive","driver","pathogenic","oncogenic","pos_high","likely_pathogenic"}
        neg_tokens = {"neg","negative","benign","likely_benign","wt","neutral"}
        if ser.isin(pos_tokens|neg_tokens).any():
            label = ser.map(lambda x: 1 if x in pos_tokens else (0 if x in neg_tokens else pd.NA)).astype("Int64")
            if label.notna().any():
                return df.rename(columns={c:"_raw_label"}).assign(label=label)
        if c.lower()=="call":
            label = ser.map(lambda x: 1 if x.startswith("pos") else (0 if x.startswith("neg") else pd.NA)).astype("Int64")
            if label.notna().any():
                return df.rename(columns={c:"_raw_call"}).assign(label=label)
    raise AssertionError("Failed to coerce labels to {0,1}. Inspect label source.")

def filter_tp53_loco(df: pd.DataFrame) -> pd.DataFrame:
    if "gene" in df.columns:
        df = df[df["gene"].astype(str).str.upper()=="TP53"].copy()
    for col in [c for c in df.columns if c.lower() in {"cv","mode","schema","tag"}]:
        mask_logo = df[col].astype(str).str.contains("logo", case=False, na=False)
        if mask_logo.any():
            df = df[~mask_logo].copy()
    return df

def audit_site_leakage(df: pd.DataFrame):
    if "site_id" in df.columns:
        g = df.groupby("site_id")["split"].nunique()
        bad = g[g>1]
        if not bad.empty:
            raise AssertionError(f"Site leakage detected for site_id(s): {bad.index.tolist()}")

# ---------- DISCOVERY ----------
if not PATH_SPLIT_MAP:
    cands = find_candidates(["splits_loco_site.csv", "splits_loco_site.tsv", "splits_loco_site.parquet"])
    PATH_SPLIT_MAP = choose_best(cands, score_split_path)
    if PATH_SPLIT_MAP:
        print(f"[auto] split-map: {PATH_SPLIT_MAP}")
    else:
        raise FileNotFoundError("Could not find splits_loco_site.{csv,tsv,parquet}. Provide PATH_SPLIT_MAP.")

if not PATH_LABELS_PRIMARY:
    # prefer TP53_delta_top.*; fallback search will also find deploy_scored.*
    cands = find_candidates(["TP53_delta_top.csv", "TP53_delta_top.tsv", "deploy_scored.csv", "deploy_scored.tsv"])
    PATH_LABELS_PRIMARY = choose_best(cands, score_label_path)
    if PATH_LABELS_PRIMARY:
        print(f"[auto] labels: {PATH_LABELS_PRIMARY}")
    else:
        raise FileNotFoundError("Could not find TP53 label source (TP53_delta_top.* or deploy_scored.*).")

# ---------- LOAD & CLEAN SPLIT MAP ----------
p = Path(PATH_SPLIT_MAP)
if p.suffix.lower()==".parquet":
    split = pd.read_parquet(p)
else:
    # auto-detect delimiter for csv/tsv
    try:
        split = pd.read_csv(p)
    except Exception:
        split = pd.read_csv(p, sep="\t")

split = filter_tp53_loco(split)
split = normalize_variant_id(split)
split = coalesce_split_cols(split)
split = split[["variant_id","split"]].dropna().drop_duplicates()

# ---------- LOAD LABELS ----------
lp = Path(PATH_LABELS_PRIMARY)
if lp.suffix.lower()==".parquet":
    labels = pd.read_parquet(lp)
else:
    try:
        labels = pd.read_csv(lp)
    except Exception:
        labels = pd.read_csv(lp, sep="\t")

labels = filter_tp53_loco(labels)
labels = normalize_variant_id(labels)
labels = ensure_label01(labels)
labels = labels[["variant_id","label"]].dropna().drop_duplicates()

# ---------- MERGE & SANITY ----------
skel = split.merge(labels, on="variant_id", how="left", validate="m:1")
skel = coalesce_split_cols(skel)
skel = skel[["variant_id","split","label"]]

missing = skel[skel["label"].isna()]
if not missing.empty:
    raise AssertionError(f"Missing labels for {len(missing)} variants. Example rows:\n{missing.head(5)}")

skel["split"] = skel["split"].astype(int)
skel["label"] = skel["label"].astype(int)

try:
    audit_site_leakage(split.join(labels.set_index("variant_id"), on="variant_id"))
except AssertionError as e:
    raise

present = set(skel["split"].unique().tolist())
cnt = skel.groupby("split")["label"].value_counts().unstack(fill_value=0).rename(columns={0:"n_neg",1:"n_pos"})
cnt["n"] = cnt["n_neg"] + cnt["n_pos"]
print("Per-split counts:\n", cnt)

if EXPECTED_PER_SPLIT:
    bad = cnt[cnt["n"] != EXPECTED_PER_SPLIT]
    if not bad.empty:
        print("[warn] counts != expected per split; investigate:\n", bad)
if EXPECTED_POS is not None and 1 in skel["label"].unique():
    bad_pos = cnt[cnt["n_pos"] != EXPECTED_POS]
    if not bad_pos.empty:
        print("[warn] n_pos != expected; investigate:\n", bad_pos)
if EXPECTED_NEG is not None and 0 in skel["label"].unique():
    bad_neg = cnt[cnt["n_neg"] != EXPECTED_NEG]
    if not bad_neg.empty:
        print("[warn] n_neg != expected; investigate:\n", bad_neg)

# ---------- WRITE OUT ----------
out_path = Path(OUT_DIR) / OUT_FILE
out_path.parent.mkdir(parents=True, exist_ok=True)
skel.to_csv(out_path, index=False)
print(f"[saved] {out_path}  rows={len(skel)}  md5={md5sum(out_path)}")
print(json.dumps({
    "skeleton_path": str(out_path),
    "md5": md5sum(out_path),
    "split_map": str(PATH_SPLIT_MAP),
    "labels": str(PATH_LABELS_PRIMARY),
    "n_rows": len(skel),
    "splits_present": sorted(present),
}, indent=2))


[auto] split-map: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv
[auto] labels: /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv


KeyError: 'n_neg'

In [38]:
# TP53 LOCO-site SKELETON (variant_id, split, label)
# - Auto-discovers split-map and labels
# - Robust to duplicate "split"/"fold" columns and messy IDs
# - Falls back to deploy_scored.* if TP53_delta_top.* is single-class
# - Enforces LOCO-site-only policy (excludes anything that smells like LOGO)
# - Writes skeleton CSV + prints audited per-split counts

import os, re, json, hashlib
from pathlib import Path
import pandas as pd

# ============================
# CONFIG — edit only if needed
# ============================
RUN_ID = "calibrated_robust_20250820_000001"

# Prefer these roots for discovery (ordered)
SEARCH_ROOTS = [
    Path("/content/drive/MyDrive/Cancer_gene_database/artifacts"),
    Path("/content/drive/MyDrive"),
    Path("/content"),
    Path.cwd(),
]

# Output location (mirrors your artifacts tree)
OUT_DIR = Path(f"/content/drive/MyDrive/Cancer_gene_database/artifacts/{RUN_ID}/ci_pairable")
OUT_FILE = "skeleton_loco_site_tp53.csv"

# Expectations (soft checks; warnings only)
EXPECTED_SPLITS_STRICT = {0, 1, 2, 4}  # split 3 is optional (recover if present)
EXPECTED_PER_SPLIT = 25
EXPECTED_POS = 11
EXPECTED_NEG = 14

# ============================
# MOUNT DRIVE (safe if mounted)
# ============================
try:
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
except Exception as e:
    print("[warn] drive mount:", e)

# =========
# HELPERS
# =========
def md5sum(p: Path) -> str:
    h = hashlib.md5()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def find_candidates(names, roots=None, max_hits=4000):
    if isinstance(names, str):
        names = [names]
    if roots is None:
        roots = SEARCH_ROOTS
    seen = []
    for root in roots:
        if not root.exists():
            continue
        for nm in names:
            # search nm and nm with any extension
            for pat in (f"**/{nm}", f"**/{nm}.*"):
                for p in root.glob(pat):
                    if p.is_file():
                        sp = str(p)
                        if sp not in seen:
                            seen.append(sp)
                        if len(seen) >= max_hits:
                            break
            if len(seen) >= max_hits:
                break
        if len(seen) >= max_hits:
            break
    return [Path(s) for s in seen]

def score_split_path(p: Path) -> int:
    s = 0
    s += 5 if p.name.lower().startswith("splits_loco_site") else 0
    s += 3 if p.suffix.lower() in {".csv", ".tsv", ".parquet"} else 0
    s += 2 if "sentiment_run_plus_variants" in str(p.parent) else 0
    s += 2 if list(p.parent.glob("tp53_counts_strict_loco.*")) else 0
    s += 1 if p.stat().st_size > 0 else 0
    return s

def score_label_path(p: Path) -> int:
    s = 0
    s += 4 if "TP53_delta_top" in p.name else 0
    s += 3 if "deploy_scored" in p.name else 0
    s += 2 if "advantage" in str(p.parent) else 0
    s += 1 if p.stat().st_size > 0 else 0
    return s

def choose_best(cands, scorer):
    if not cands:
        return None
    return sorted(cands, key=lambda p: (-scorer(p), len(str(p))))[0]

def read_table_auto(p: Path) -> pd.DataFrame:
    suf = p.suffix.lower()
    if suf == ".parquet":
        return pd.read_parquet(p)
    try:
        return pd.read_csv(p)
    except Exception:
        return pd.read_csv(p, sep="\t")

def coalesce_split_cols(df: pd.DataFrame, prefer="split") -> pd.DataFrame:
    cand = [c for c in df.columns if re.fullmatch(r"(?i)(split|.*_split|split_.*)", c)]
    if not cand and "fold" in df.columns:
        cand = ["fold"]
    if not cand:
        raise AssertionError("No split/fold column found in split-map.")
    ordered = sorted(cand, key=lambda c: (0 if c.lower() == prefer else 1, c))
    tmp = pd.concat([df[c] for c in ordered], axis=1)
    df["split"] = pd.to_numeric(tmp.bfill(axis=1).iloc[:, 0], errors="coerce").astype("Int64")
    drop_cols = [c for c in cand if c != "split"]
    return df.drop(columns=drop_cols, errors="ignore")

def normalize_variant_id(df: pd.DataFrame) -> pd.DataFrame:
    if "variant_id" in df.columns and df["variant_id"].notna().any():
        return df.assign(variant_id=df["variant_id"].astype(str))
    for key in ["variant_id_hgvs_core", "hgvs_core", "hgvs_c", "hgvs_nt", "hgvs"]:
        if key in df.columns and df[key].notna().any():
            return df.assign(variant_id=df[key].astype(str))
    cols = {c.lower(): c for c in df.columns}
    need = ["chr", "pos", "ref", "alt"]
    if all(k in cols for k in need):
        cid, pid, rid, aid = (cols["chr"], cols["pos"], cols["ref"], cols["alt"])
        vid = df[cid].astype(str) + ":" + df[pid].astype(str) + "_" + df[rid].astype(str) + ">" + df[aid].astype(str)
        return df.assign(variant_id=vid)
    raise AssertionError("Could not derive variant_id — provide one of {variant_id, hgvs_* or chr/pos/ref/alt}.")

def ensure_label01(df: pd.DataFrame) -> pd.DataFrame:
    cand = [c for c in df.columns if c.lower() in {"label", "class", "y", "oncogenic", "is_driver", "is_pathogenic", "call"}]
    if not cand:
        raise AssertionError("No obvious label column. Seen cols: " + ", ".join(df.columns[:25].tolist()))
    for c in cand:
        s = df[c]
        # numeric/bool
        if pd.api.types.is_bool_dtype(s) or pd.api.types.is_numeric_dtype(s):
            mapped = s.astype("Int64")
            uniq = set(mapped.dropna().unique().tolist())
            if uniq <= {0, 1}:
                return df.rename(columns={c: "label"}).assign(label=mapped.astype("Int64"))
            if uniq <= {-1, 1}:
                return df.rename(columns={c: "label"}).assign(label=(mapped > 0).astype("Int64"))
        # strings
        ser = s.astype(str).str.lower()
        pos_tokens = {"pos", "positive", "driver", "pathogenic", "oncogenic", "pos_high", "likely_pathogenic"}
        neg_tokens = {"neg", "negative", "benign", "likely_benign", "wt", "neutral"}
        if ser.isin(pos_tokens | neg_tokens).any():
            label = ser.map(lambda x: 1 if x in pos_tokens else (0 if x in neg_tokens else pd.NA)).astype("Int64")
            if label.notna().any():
                return df.rename(columns={c: "_raw_label"}).assign(label=label)
        if c.lower() == "call":
            label = ser.map(lambda x: 1 if x.startswith("pos") else (0 if x.startswith("neg") else pd.NA)).astype("Int64")
            if label.notna().any():
                return df.rename(columns={c: "_raw_call"}).assign(label=label)
    raise AssertionError("Failed to coerce labels to {0,1}. Inspect label source.")

def filter_tp53_loco(df: pd.DataFrame) -> pd.DataFrame:
    if "gene" in df.columns:
        df = df[df["gene"].astype(str).str.upper() == "TP53"].copy()
    # exclude LOGO rows if marked
    for col in [c for c in df.columns if c.lower() in {"cv", "mode", "schema", "tag"}]:
        mask_logo = df[col].astype(str).str.contains("logo", case=False, na=False)
        if mask_logo.any():
            df = df[~mask_logo].copy()
    return df

def audit_site_leakage(df: pd.DataFrame):
    if "site_id" in df.columns:
        g = df.groupby("site_id")["split"].nunique()
        bad = g[g > 1]
        if not bad.empty:
            raise AssertionError(f"Site leakage detected for site_id(s): {bad.index.tolist()}")

# ============================
# DISCOVERY
# ============================
split_cands = find_candidates(["splits_loco_site.csv", "splits_loco_site.tsv", "splits_loco_site.parquet"])
split_path = choose_best(split_cands, score_split_path)
if not split_path:
    raise FileNotFoundError("Could not find splits_loco_site.{csv,tsv,parquet} under search roots.")

label_cands = find_candidates(["TP53_delta_top.csv", "TP53_delta_top.tsv", "TP53_delta_top.parquet",
                               "deploy_scored.csv", "deploy_scored.tsv", "deploy_scored.parquet"])
label_path = choose_best(label_cands, score_label_path)
if not label_path:
    raise FileNotFoundError("Could not find TP53 label source (TP53_delta_top.* or deploy_scored.*).")

print(f"[auto] split-map: {split_path}")
print(f"[auto] labels:    {label_path}")

# ============================
# LOAD + CLEAN
# ============================
# split-map
split = read_table_auto(split_path)
split = filter_tp53_loco(split)
split = normalize_variant_id(split)
split = coalesce_split_cols(split)
split = split[["variant_id", "split"]].dropna().drop_duplicates()
# if multiple rows per (variant_id, split) sneak in, dedupe hard:
split = split.drop_duplicates(["variant_id", "split"])

# labels
labels = read_table_auto(label_path)
labels = filter_tp53_loco(labels)
labels = normalize_variant_id(labels)
labels = ensure_label01(labels)
labels = labels[["variant_id", "label"]].dropna().drop_duplicates(subset=["variant_id"])

# if single-class, hard fallback to deploy_scored.*
uniq = set(labels["label"].dropna().unique().tolist())
if uniq in [{1}, {0}]:
    print(f"[warn] Label source {label_path.name} has a single class {uniq}. Searching fallback deploy_scored.*")
    fb_cands = [p for p in label_cands if "deploy_scored" in p.name]
    fb_path = choose_best(fb_cands, score_label_path)
    if not fb_path:
        raise AssertionError("Fallback deploy_scored.* not found; need a labels file with both classes.")
    labels = read_table_auto(fb_path)
    labels = filter_tp53_loco(labels)
    labels = normalize_variant_id(labels)
    labels = ensure_label01(labels)
    labels = labels[["variant_id", "label"]].dropna().drop_duplicates(subset=["variant_id"])
    label_path = fb_path
    print(f"[auto] switched labels → {label_path}")

# final class check
uniq = set(labels["label"].dropna().unique().tolist())
if not (0 in uniq and 1 in uniq):
    raise AssertionError(f"Labels still not bi-class. Classes seen: {sorted(list(uniq))}")

# ============================
# MERGE → SKELETON
# ============================
skel = split.merge(labels, on="variant_id", how="left", validate="m:1")
# coalesce any split_x/split_y junk (shouldn't exist after earlier coalesce, but safe)
skel = coalesce_split_cols(skel)
skel = skel[["variant_id", "split", "label"]]

# hard fail if any missing labels
miss = skel[skel["label"].isna()]
if not miss.empty:
    raise AssertionError(f"Missing labels for {len(miss)} variants. Example:\n{miss.head(5)}")

skel["split"] = skel["split"].astype(int)
skel["label"] = skel["label"].astype(int)

# site leakage audit if site_id exists in split-map
try:
    audit_site_leakage(split.join(labels.set_index("variant_id"), on="variant_id"))
except AssertionError as e:
    raise

# ============================
# COUNTS (robust to single-class per split)
# ============================
vc = skel.groupby("split")["label"].value_counts(dropna=False)
cnt = vc.unstack(fill_value=0).reindex(columns=[0, 1], fill_value=0)  # ensure both cols exist
cnt.columns = ["n_neg", "n_pos"]
cnt["n"] = cnt["n_neg"] + cnt["n_pos"]
print("Per-split counts:\n", cnt.sort_index())

# soft expectations
present = set(skel["split"].unique().tolist())
if not present.issubset(EXPECTED_SPLITS_STRICT | {3}):
    print(f"[warn] unexpected splits present: {sorted(present)}")
if EXPECTED_PER_SPLIT:
    bad = cnt[cnt["n"] != EXPECTED_PER_SPLIT]
    if not bad.empty:
        print("[warn] n != expected per split:\n", bad)
if EXPECTED_POS is not None:
    bad_pos = cnt[cnt["n_pos"] != EXPECTED_POS]
    if not bad_pos.empty:
        print("[warn] n_pos != expected:\n", bad_pos)
if EXPECTED_NEG is not None:
    bad_neg = cnt[cnt["n_neg"] != EXPECTED_NEG]
    if not bad_neg.empty:
        print("[warn] n_neg != expected:\n", bad_neg)

print("Label uniques in SKELETON:", sorted(skel["label"].dropna().unique().tolist()))

# ============================
# WRITE OUT
# ============================
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / OUT_FILE
skel.to_csv(out_path, index=False)
md5 = md5sum(out_path)

summary = {
    "skeleton_path": str(out_path),
    "md5": md5,
    "run_id": RUN_ID,
    "split_map": str(split_path),
    "labels": str(label_path),
    "rows": len(skel),
    "splits_present": sorted(present),
    "per_split_counts": cnt.reset_index().to_dict(orient="records"),
}
print(f"[saved] {out_path}  rows={len(skel)}  md5={md5}")
print(json.dumps(summary, indent=2))

# ============================
# NEXT STEP (for your clipboard)
# ============================
print("\nNext → Attach p_wt + context from RUN to this skeleton, recompute per-fold AUROC/PR/Brier, then 10k bootstrap CIs (seeds AUROC=42, PR=43) + one-sided Wilcoxon.")


[auto] split-map: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv
[auto] labels:    /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv
Per-split counts:
 Empty DataFrame
Columns: [n_neg, n_pos, n]
Index: []
Label uniques in SKELETON: []
[saved] /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/skeleton_loco_site_tp53.csv  rows=0  md5=6dc2130b4945d6cb1fcf379eb0747600
{
  "skeleton_path": "/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/skeleton_loco_site_tp53.csv",
  "md5": "6dc2130b4945d6cb1fcf379eb0747600",
  "run_id": "calibrated_robust_20250820_000001",
  "split_map": "/content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv",
  "labels": "/content/drive/MyDrive/Cancer_gene_database/arti

In [39]:
# TP53 LOCO-site SKELETON (variant_id, split, label) — auto-diagnose + build
# - Auto-discovers split-map and labels
# - Tries multiple join keys and selects the best-overlap key-pair
# - Falls back to deploy_scored.* if primary labels don't align or are single-class
# - LOCO-site only; excludes LOGO; robust to duplicate "split"/"fold" columns

import os, re, json, hashlib
from pathlib import Path
import pandas as pd

RUN_ID = "calibrated_robust_20250820_000001"
SEARCH_ROOTS = [
    Path("/content/drive/MyDrive/Cancer_gene_database/artifacts"),
    Path("/content/drive/MyDrive"),
    Path("/content"),
    Path.cwd(),
]
OUT_DIR = Path(f"/content/drive/MyDrive/Cancer_gene_database/artifacts/{RUN_ID}/ci_pairable")
OUT_FILE = "skeleton_loco_site_tp53.csv"

EXPECTED_SPLITS_STRICT = {0,1,2,4}   # accept 3 if present
EXPECTED_PER_SPLIT = 25
EXPECTED_POS, EXPECTED_NEG = 11, 14

# ---------- mount drive if needed ----------
try:
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
except Exception as e:
    print("[warn] drive mount:", e)

# ---------- helpers ----------
def md5sum(p: Path) -> str:
    h = hashlib.md5()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def find_candidates(names, roots=SEARCH_ROOTS, max_hits=4000):
    if isinstance(names, str): names=[names]
    found=[]
    for root in roots:
        if not root.exists(): continue
        for nm in names:
            for pat in (f"**/{nm}", f"**/{nm}.*"):
                for p in root.glob(pat):
                    if p.is_file():
                        sp=str(p)
                        if sp not in found: found.append(sp)
                        if len(found)>=max_hits: break
            if len(found)>=max_hits: break
        if len(found)>=max_hits: break
    return [Path(s) for s in found]

def score_split_path(p: Path) -> int:
    s=0
    s+=5 if p.name.lower().startswith("splits_loco_site") else 0
    s+=3 if p.suffix.lower() in {".csv",".tsv",".parquet"} else 0
    s+=2 if "sentiment_run_plus_variants" in str(p.parent) else 0
    s+=2 if list(p.parent.glob("tp53_counts_strict_loco.*")) else 0
    s+=1 if p.stat().st_size>0 else 0
    return s

def score_label_path(p: Path) -> int:
    s=0
    s+=4 if "TP53_delta_top" in p.name else 0
    s+=3 if "deploy_scored" in p.name else 0
    s+=2 if "advantage" in str(p.parent) else 0
    s+=1 if p.stat().st_size>0 else 0
    return s

def choose_best(cands, scorer):
    if not cands: return None
    return sorted(cands, key=lambda p: (-scorer(p), len(str(p))))[0]

def read_table_auto(p: Path) -> pd.DataFrame:
    suf=p.suffix.lower()
    if suf==".parquet": return pd.read_parquet(p)
    try: return pd.read_csv(p)
    except Exception: return pd.read_csv(p, sep="\t")

def coalesce_split_cols(df: pd.DataFrame, prefer="split") -> pd.DataFrame:
    cand=[c for c in df.columns if re.fullmatch(r"(?i)(split|.*_split|split_.*)", c)]
    if not cand and "fold" in df.columns: cand=["fold"]
    if not cand: raise AssertionError("No split/fold column found in split-map.")
    ordered=sorted(cand, key=lambda c: (0 if c.lower()==prefer else 1, c))
    tmp=pd.concat([df[c] for c in ordered], axis=1)
    df["split"]=pd.to_numeric(tmp.bfill(axis=1).iloc[:,0], errors="coerce").astype("Int64")
    drop=[c for c in cand if c!="split"]
    return df.drop(columns=drop, errors="ignore")

def filter_tp53_loco(df: pd.DataFrame) -> pd.DataFrame:
    if "gene" in df.columns:
        df=df[df["gene"].astype(str).str.upper()=="TP53"].copy()
    for col in [c for c in df.columns if c.lower() in {"cv","mode","schema","tag"}]:
        mask_logo=df[col].astype(str).str.contains("logo", case=False, na=False)
        if mask_logo.any(): df=df[~mask_logo].copy()
    return df

def ensure_label01(df: pd.DataFrame) -> pd.DataFrame:
    cand=[c for c in df.columns if c.lower() in {"label","class","y","oncogenic","is_driver","is_pathogenic","call"}]
    if not cand: raise AssertionError("No obvious label column. Seen: "+", ".join(df.columns[:25].tolist()))
    for c in cand:
        s=df[c]
        if pd.api.types.is_bool_dtype(s) or pd.api.types.is_numeric_dtype(s):
            mapped=s.astype("Int64")
            uniq=set(mapped.dropna().unique().tolist())
            if uniq <= {0,1}:
                return df.rename(columns={c:"label"}).assign(label=mapped.astype("Int64"))
            if uniq <= {-1,1}:
                return df.rename(columns={c:"label"}).assign(label=(mapped>0).astype("Int64"))
        ser=s.astype(str).str.lower()
        pos={"pos","positive","driver","pathogenic","oncogenic","pos_high","likely_pathogenic"}
        neg={"neg","negative","benign","likely_benign","wt","neutral"}
        if ser.isin(pos|neg).any():
            lab=ser.map(lambda x: 1 if x in pos else (0 if x in neg else pd.NA)).astype("Int64")
            if lab.notna().any(): return df.rename(columns={c:"_raw_label"}).assign(label=lab)
        if c.lower()=="call":
            lab=ser.map(lambda x: 1 if x.startswith("pos") else (0 if x.startswith("neg") else pd.NA)).astype("Int64")
            if lab.notna().any(): return df.rename(columns={c:"_raw_call"}).assign(label=lab)
    raise AssertionError("Failed to coerce labels to {0,1}.")

# --- key building & matching ---
def canon(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.replace(r"\s+", "", regex=True)

def to_hgvs_core(series: pd.Series) -> pd.Series:
    s=canon(series)
    # pull c.*** core (optionally after transcript)
    c1=s.str.extract(r"(c\.[A-Za-z0-9_>\-\+\*\?]+)", expand=False)
    # sometimes stored as NM_xxx: c.xxx or NM_xxx.c.xxx
    c2=s.str.extract(r"NM_\d+\.\d+[:\.](c\.[A-Za-z0-9_>\-\+\*\?]+)", expand=False)
    out=c1.fillna(c2)
    # uppercase ref/alt letters inside core
    out=out.str.replace(r"([acgt])", lambda m: m.group(1).upper(), regex=True)
    return out

def to_protein(series: pd.Series) -> pd.Series:
    s=canon(series)
    p1=s.str.extract(r"(p\.[A-Za-z][a-z]{2}\d+[A-Za-z][a-z]{2}|p\.[A-Za-z]{1,3}\d+[A-Za-z]{1,3})", expand=False)
    return p1

def to_chrpos(series: pd.Series) -> pd.Series:
    s=canon(series)
    # e.g., 17:7673803_C>T or chr17:7673803_C>T
    pat=r"(?:chr)?(\d+|X|Y)[:](\d+)[_](\w+)[>](\w+)"
    m=s.str.extract(pat, expand=True)
    if m.isna().all(axis=None): return pd.Series([pd.NA]*len(s))
    chr_,pos,ref,alt=m[0],m[1],m[2],m[3]
    key=chr_.astype(str)+":"+pos.astype(str)+"_"+ref.str.upper()+">"+alt.str.upper()
    return key

def build_keyframe(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out=pd.DataFrame(index=df.index)
    if "variant_id" in df.columns: out["variant_id"]=canon(df["variant_id"])
    for k in ["variant_id_hgvs_core","hgvs_core","hgvs_c","hgvs_nt","hgvs"]:
        if k in df.columns: out[k]=canon(df[k])
    if "protein" in df.columns: out["protein"]=canon(df["protein"])
    # derived keys (even from variant_id-like fields)
    any_src = out["variant_id"] if "variant_id" in out.columns else df.apply(lambda r: pd.NA, axis=1)
    out["hgvs_core_from_variant"]=to_hgvs_core(df.get("variant_id", pd.Series([pd.NA]*len(df))))
    # also try from any hgvs column
    for k in ["variant_id_hgvs_core","hgvs_core","hgvs_c","hgvs_nt","hgvs"]:
        if k in df.columns:
            out[f"hgvs_core_from_{k}"]=to_hgvs_core(df[k])
    # protein core from any string column likely to carry it
    for k in ["variant_id","protein","hgvs","hgvs_c","hgvs_core"]:
        if k in df.columns:
            out[f"protein_from_{k}"]=to_protein(df[k])
    # chr:pos_ref>alt
    for k in ["variant_id","hgvs","hgvs_c","genomic","coord","pos","chr"]:
        if k in df.columns:
            out[f"chrpos_from_{k}"]=to_chrpos(df[k])
    # collapse to unique non-null columns
    out=out.dropna(axis=1, how="all")
    out.columns=[f"{name}__{c}" for c in out.columns]
    return out

def best_overlap(split_df: pd.DataFrame, label_df: pd.DataFrame):
    s_keys=build_keyframe(split_df, "s")
    l_keys=build_keyframe(label_df, "l")
    # thin to unique values
    s_cols,s_uniques=[],{}
    for c in s_keys.columns:
        vals=set(s_keys[c].dropna().unique().tolist())
        if vals: s_cols.append(c); s_uniques[c]=vals
    l_cols,l_uniques=[],{}
    for c in l_keys.columns:
        vals=set(l_keys[c].dropna().unique().tolist())
        if vals: l_cols.append(c); l_uniques[c]=vals
    # brute pair scoring
    best=("","",0)
    for sc in s_cols:
        for lc in l_cols:
            inter=len(s_uniques[sc] & l_uniques[lc])
            if inter>best[2]:
                best=(sc,lc,inter)
    return best, s_keys, l_keys

def audit_site_leakage(df: pd.DataFrame):
    if "site_id" in df.columns:
        g=df.groupby("site_id")["split"].nunique()
        bad=g[g>1]
        if not bad.empty: raise AssertionError(f"Site leakage detected for site_id(s): {bad.index.tolist()}")

# ---------- discover files ----------
split_cands=find_candidates(["splits_loco_site.csv","splits_loco_site.tsv","splits_loco_site.parquet"])
split_path=choose_best(split_cands, score_split_path)
if not split_path: raise FileNotFoundError("No splits_loco_site.* found.")
label_cands=find_candidates(["TP53_delta_top.csv","TP53_delta_top.tsv","TP53_delta_top.parquet",
                             "deploy_scored.csv","deploy_scored.tsv","deploy_scored.parquet"])
label_path=choose_best(label_cands, score_label_path)
if not label_path: raise FileNotFoundError("No TP53 label source found.")

print(f"[auto] split-map: {split_path}")
print(f"[auto] labels:    {label_path}")

# ---------- load raw tables ----------
split_raw=read_table_auto(split_path)
labels_raw=read_table_auto(label_path)

# ---------- light LOCO & split prep ----------
split=filter_tp53_loco(split_raw.copy())
split=coalesce_split_cols(split)
# keep a wide copy for key-building BEFORE narrowing
split_wide=split.copy()
# prefer existing variant_id; else we'll synthesize later when joining
if "variant_id" in split.columns and split["variant_id"].notna().any():
    split["variant_id"]=split["variant_id"].astype(str)
else:
    split["variant_id"]=pd.NA
split=split[["variant_id","split"]].dropna(subset=["split"]).drop_duplicates()
split=split.drop_duplicates(["variant_id","split"])

# ---------- labels prep (no filtering to single class yet) ----------
labels=labels_raw.copy()
labels=filter_tp53_loco(labels)
labels=ensure_label01(labels)
labels_wide=labels.copy()

# ---------- try best-overlap key-pair ----------
(best_s, best_l, best_n), s_keys, l_keys = best_overlap(split_wide, labels_wide)
print(f"[diag] best overlap: {best_n} matches via {best_s} ⇄ {best_l}")

def materialize_key(df, keycol):
    if keycol not in df.columns:
        raise KeyError(keycol)
    return df[keycol]

def attach_join_key(base_df, wide_keys, key_name, outcol):
    base_df=base_df.copy()
    key=wide_keys[key_name]
    base_df[outcol]=key
    return base_df

def attempt_join(split_wide, labels_wide, s_keys, l_keys, s_col, l_col):
    s_tmp=split_wide.copy()
    l_tmp=labels_wide.copy()
    s_tmp["__join_key__"]=s_keys[s_col]
    l_tmp["__join_key__"]=l_keys[l_col]
    # prune nulls
    s_tmp=s_tmp[~s_tmp["__join_key__"].isna()].copy()
    l_tmp=l_tmp[~l_tmp["__join_key__"].isna()].copy()
    # keep 1:1 on labels per key
    l_tmp=l_tmp.drop_duplicates(["__join_key__"])
    # coalesce split
    s_tmp=coalesce_split_cols(s_tmp)
    # now map labels→key
    merged=s_tmp.merge(l_tmp[["__join_key__","label"]], on="__join_key__", how="left")
    # choose variant_id: prefer existing, else use join key
    if "variant_id" in merged.columns and merged["variant_id"].notna().any():
        merged["variant_id"]=merged["variant_id"].astype(str).fillna(merged["__join_key__"].astype(str))
    else:
        merged["variant_id"]=merged["__join_key__"].astype(str)
    out=merged[["variant_id","split","label"]].dropna(subset=["split"]).drop_duplicates()
    return out

# first attempt with best pair
skel=None
if best_n>0:
    try:
        skel=attempt_join(split_wide, labels_wide, s_keys, l_keys, best_s, best_l)
    except Exception as e:
        print("[diag] best-pair join failed:", e)

# if join empty or none: try a prioritized set of pairs
if skel is None or skel.empty:
    print("[diag] trying fallback key-pairs…")
    # enumerate candidate columns sorted by "type" priority
    def cols_like(keys, pat):
        return [c for c in keys.columns if re.search(pat, c)]
    s_order = (cols_like(s_keys, r"^s__variant_id$") +
               cols_like(s_keys, r"hgvs_core") +
               cols_like(s_keys, r"protein") +
               cols_like(s_keys, r"chrpos"))
    l_order = (cols_like(l_keys, r"^l__variant_id$") +
               cols_like(l_keys, r"hgvs_core") +
               cols_like(l_keys, r"protein") +
               cols_like(l_keys, r"chrpos"))
    tried=0
    for sc in s_order:
        for lc in l_order:
            inter=len(set(s_keys[sc].dropna().unique()) & set(l_keys[lc].dropna().unique()))
            if inter==0: continue
            tried+=1
            skel_try=attempt_join(split_wide, labels_wide, s_keys, l_keys, sc, lc)
            if not skel_try.empty:
                print(f"[diag] success via {sc} ⇄ {lc} | matched rows: {len(skel_try)}")
                skel=skel_try
                break
        if skel is not None and not skel.empty:
            break

# if still empty: switch label source to deploy_scored.* and redo
if skel is None or skel.empty:
    print("[warn] join produced 0 rows. Switching labels to deploy_scored.*")
    fb_cands=[p for p in label_cands if "deploy_scored" in p.name]
    fb=choose_best(fb_cands, score_label_path)
    if not fb: raise AssertionError("No deploy_scored.* fallback found.")
    label_path=fb
    labels_raw=read_table_auto(label_path)
    labels=filter_tp53_loco(labels_raw.copy())
    labels=ensure_label01(labels)
    labels_wide=labels.copy()
    (best_s, best_l, best_n), s_keys, l_keys = best_overlap(split_wide, labels_wide)
    print(f"[diag] (fallback) best overlap: {best_n} via {best_s} ⇄ {best_l}")
    skel=attempt_join(split_wide, labels_wide, s_keys, l_keys, best_s, best_l)
    if skel.empty:
        print("[diag] fallback best-pair empty; trying exhaustive pairs…")
        # exhaustive again
        def cols_like(keys, pat): return [c for c in keys.columns if re.search(pat, c)]
        s_order = (cols_like(s_keys, r"^s__variant_id$") + cols_like(s_keys, r"hgvs_core") +
                   cols_like(s_keys, r"protein") + cols_like(s_keys, r"chrpos"))
        l_order = (cols_like(l_keys, r"^l__variant_id$") + cols_like(l_keys, r"hgvs_core") +
                   cols_like(l_keys, r"protein") + cols_like(l_keys, r"chrpos"))
        for sc in s_order:
            for lc in l_order:
                inter=len(set(s_keys[sc].dropna().unique()) & set(l_keys[lc].dropna().unique()))
                if inter==0: continue
                skel_try=attempt_join(split_wide, labels_wide, s_keys, l_keys, sc, lc)
                if not skel_try.empty:
                    print(f"[diag] (fallback) success via {sc} ⇄ {lc} | matched rows: {len(skel_try)}")
                    skel=skel_try
                    break
            if not skel.empty: break

# hard fail if still empty with some guidance
if skel is None or skel.empty:
    print("[fail] Could not align split-map with any label source.")
    print("[peek] split columns:", list(split_wide.columns)[:25])
    print("[peek] labels columns:", list(labels_wide.columns)[:25])
    print("[peek] split head:\n", split_wide.head(10))
    print("[peek] labels head:\n", labels_wide.head(10))
    raise SystemExit(1)

# finalize types + leakage audit
skel=skel.dropna(subset=["label","split"]).copy()
skel["split"]=skel["split"].astype(int)
skel["label"]=skel["label"].astype(int)
try:
    # if site_id in split_wide, check leakage
    tmp = split_wide.copy()
    if "variant_id" not in tmp.columns: tmp["variant_id"]=pd.NA
    tmp=tmp.merge(skel[["variant_id","split"]].drop_duplicates(), on=["variant_id","split"], how="right")
    audit_site_leakage(tmp)
except AssertionError as e:
    raise

# robust per-split counts
vc=skel.groupby("split")["label"].value_counts(dropna=False)
cnt=vc.unstack(fill_value=0).reindex(columns=[0,1], fill_value=0)
cnt.columns=["n_neg","n_pos"]; cnt["n"]=cnt["n_neg"]+cnt["n_pos"]
print("Per-split counts:\n", cnt.sort_index())
present=set(skel["split"].unique().tolist())
if not present.issubset(EXPECTED_SPLITS_STRICT | {3}):
    print(f"[warn] unexpected splits present: {sorted(present)}")
# soft expectations
if EXPECTED_PER_SPLIT:
    bad=cnt[cnt["n"]!=EXPECTED_PER_SPLIT]
    if not bad.empty: print("[warn] n != expected per split:\n", bad)
if EXPECTED_POS is not None:
    badp=cnt[cnt["n_pos"]!=EXPECTED_POS]
    if not badp.empty: print("[warn] n_pos != expected:\n", badp)
if EXPECTED_NEG is not None:
    badn=cnt[cnt["n_neg"]!=EXPECTED_NEG]
    if not badn.empty: print("[warn] n_neg != expected:\n", badn)

# write out
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path=OUT_DIR/OUT_FILE
skel.drop_duplicates(["variant_id","split"]).to_csv(out_path, index=False)
print(f"[saved] {out_path}  rows={len(skel)}  md5={md5sum(out_path)}")
print(json.dumps({
    "skeleton_path": str(out_path),
    "md5": md5sum(out_path),
    "run_id": RUN_ID,
    "split_map": str(split_path),
    "labels": str(label_path),
    "rows": len(skel),
    "splits_present": sorted(present),
    "per_split_counts": cnt.reset_index().to_dict(orient="records"),
}, indent=2))

print("\nNext → attach p_wt + context from this RUN to the skeleton; recompute per-fold metrics (AUROC/PR/Brier), 10k bootstrap CIs (seeds AUROC=42, PR=43), Wilcoxon one-sided.")


[auto] split-map: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv
[auto] labels:    /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv


/tmp/ipython-input-2251822396.py:135: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out=c1.fillna(c2)


AttributeError: Can only use .str accessor with string values!

In [40]:
# TP53 LOCO-site SKELETON (variant_id, split, label) — bulletproof version
# - String-safe (uses Pandas StringDtype everywhere; guards all-NA before .str ops)
# - Auto-discovers split-map + labels; falls back to deploy_scored.* if needed
# - Best-overlap key search across: variant_id, HGVS core (c.*), protein (p.*), chr:pos_ref>alt
# - LOCO-site only; duplicate split/fold safe; site leakage audit if site_id present

import re, json, hashlib
from pathlib import Path
import pandas as pd

RUN_ID = "calibrated_robust_20250820_000001"
SEARCH_ROOTS = [
    Path("/content/drive/MyDrive/Cancer_gene_database/artifacts"),
    Path("/content/drive/MyDrive"),
    Path("/content"),
    Path.cwd(),
]
OUT_DIR = Path(f"/content/drive/MyDrive/Cancer_gene_database/artifacts/{RUN_ID}/ci_pairable")
OUT_FILE = "skeleton_loco_site_tp53.csv"

EXPECTED_SPLITS_STRICT = {0,1,2,4}   # accept 3 if present
EXPECTED_PER_SPLIT = 25
EXPECTED_POS, EXPECTED_NEG = 11, 14

# ---------- mount drive (idempotent) ----------
try:
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
except Exception as e:
    print("[warn] drive mount:", e)

# ---------- utils ----------
def md5sum(p: Path) -> str:
    h = hashlib.md5()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def find_candidates(names, roots=SEARCH_ROOTS, max_hits=4000):
    if isinstance(names, str): names=[names]
    found=[]
    for root in roots:
        if not root.exists(): continue
        for nm in names:
            for pat in (f"**/{nm}", f"**/{nm}.*"):
                for p in root.glob(pat):
                    if p.is_file():
                        sp=str(p)
                        if sp not in found:
                            found.append(sp)
                        if len(found)>=max_hits: break
            if len(found)>=max_hits: break
        if len(found)>=max_hits: break
    return [Path(s) for s in found]

def score_split_path(p: Path) -> int:
    s=0
    s+=5 if p.name.lower().startswith("splits_loco_site") else 0
    s+=3 if p.suffix.lower() in {".csv",".tsv",".parquet"} else 0
    s+=2 if "sentiment_run_plus_variants" in str(p.parent) else 0
    s+=2 if list(p.parent.glob("tp53_counts_strict_loco.*")) else 0
    s+=1 if p.stat().st_size>0 else 0
    return s

def score_label_path(p: Path) -> int:
    s=0
    s+=4 if "TP53_delta_top" in p.name else 0
    s+=3 if "deploy_scored" in p.name else 0
    s+=2 if "advantage" in str(p.parent) else 0
    s+=1 if p.stat().st_size>0 else 0
    return s

def choose_best(cands, scorer):
    if not cands: return None
    return sorted(cands, key=lambda p: (-scorer(p), len(str(p))))[0]

def read_table_auto(p: Path) -> pd.DataFrame:
    suf=p.suffix.lower()
    if suf==".parquet": return pd.read_parquet(p)
    try: return pd.read_csv(p)
    except Exception: return pd.read_csv(p, sep="\t")

# ----- string-safe helpers -----
def to_string(s: pd.Series) -> pd.Series:
    # force Pandas nullable StringDtype; preserve NA
    return s.astype("string")

def canon(s: pd.Series) -> pd.Series:
    s = to_string(s)
    s = s.str.strip()
    s = s.str.replace(r"\s+", "", regex=True)
    return s

def safe_extract(series: pd.Series, pattern: str):
    s = canon(series)
    out = s.str.extract(pattern, expand=False)
    # out may be all NA ⇒ ensure StringDtype & short-circuit .str ops later
    return to_string(out)

def to_hgvs_core(series: pd.Series) -> pd.Series:
    # try direct c.*; or NM.*:c.*
    c1 = safe_extract(series, r"(c\.[A-Za-z0-9_>\-\+\*\?]+)")
    c2 = safe_extract(series, r"NM_\d+\.\d+[:\.](c\.[A-Za-z0-9_>\-\+\*\?]+)")
    out = c1.fillna(c2)
    if out.isna().all():
        return pd.Series([pd.NA]*len(series), dtype="string")
    return to_string(out).str.upper()

def to_protein(series: pd.Series) -> pd.Series:
    p = safe_extract(series, r"(p\.[A-Za-z]{1,3}\d+[A-Za-z]{1,3})")
    if p.isna().all():
        return pd.Series([pd.NA]*len(series), dtype="string")
    return to_string(p).str.upper()

def to_chrpos(series: pd.Series) -> pd.Series:
    s = canon(series)
    m = s.str.extract(r"(?:CHR)?(\d+|X|Y)[:](\d+)[_](\w+)[>](\w+)", expand=True)
    if m.isna().all(axis=None):
        return pd.Series([pd.NA]*len(series), dtype="string")
    chr_,pos,ref,alt = (to_string(m[0]), to_string(m[1]), to_string(m[2]), to_string(m[3]))
    key = chr_.fillna("").str.upper()+":"+pos.fillna("")+"_"+ref.fillna("").str.upper()+">"+alt.fillna("").str.upper()
    key = key.replace("", pd.NA)
    return to_string(key)

def coalesce_split_cols(df: pd.DataFrame, prefer="split") -> pd.DataFrame:
    cand=[c for c in df.columns if re.fullmatch(r"(?i)(split|.*_split|split_.*)", c)]
    if not cand and "fold" in df.columns: cand=["fold"]
    if not cand: raise AssertionError("No split/fold column found in split-map.")
    ordered=sorted(cand, key=lambda c: (0 if c.lower()==prefer else 1, c))
    tmp=pd.concat([df[c] for c in ordered], axis=1)
    df["split"]=pd.to_numeric(tmp.bfill(axis=1).iloc[:,0], errors="coerce").astype("Int64")
    drop=[c for c in cand if c!="split"]
    return df.drop(columns=drop, errors="ignore")

def filter_tp53_loco(df: pd.DataFrame) -> pd.DataFrame:
    if "gene" in df.columns:
        df=df[df["gene"].astype(str).str.upper()=="TP53"].copy()
    for col in [c for c in df.columns if c.lower() in {"cv","mode","schema","tag"}]:
        mask_logo=df[col].astype(str).str.contains("logo", case=False, na=False)
        if mask_logo.any(): df=df[~mask_logo].copy()
    return df

def ensure_label01(df: pd.DataFrame) -> pd.DataFrame:
    cand=[c for c in df.columns if c.lower() in {"label","class","y","oncogenic","is_driver","is_pathogenic","call"}]
    if not cand: raise AssertionError("No obvious label column. Seen: "+", ".join(df.columns[:25].tolist()))
    for c in cand:
        s=df[c]
        if pd.api.types.is_bool_dtype(s) or pd.api.types.is_numeric_dtype(s):
            mapped=pd.Series(s, dtype="Int64")
            uniq=set(mapped.dropna().unique().tolist())
            if uniq <= {0,1}:
                return df.rename(columns={c:"label"}).assign(label=mapped)
            if uniq <= {-1,1}:
                return df.rename(columns={c:"label"}).assign(label=(mapped>0).astype("Int64"))
        ser=to_string(df[c]).str.lower()
        pos={"pos","positive","driver","pathogenic","oncogenic","pos_high","likely_pathogenic"}
        neg={"neg","negative","benign","likely_benign","wt","neutral"}
        if ser.isin(pos|neg).any():
            lab=ser.map(lambda x: 1 if x in pos else (0 if x in neg else pd.NA)).astype("Int64")
            if lab.notna().any(): return df.rename(columns={c:"_raw_label"}).assign(label=lab)
        if c.lower()=="call":
            lab=ser.map(lambda x: 1 if str(x).startswith("pos") else (0 if str(x).startswith("neg") else pd.NA)).astype("Int64")
            if lab.notna().any(): return df.rename(columns={c:"_raw_call"}).assign(label=lab)
    raise AssertionError("Failed to coerce labels to {0,1}.")

def build_keyframe(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out=pd.DataFrame(index=df.index)
    if "variant_id" in df.columns:
        out["variant_id"]=canon(df["variant_id"])
    for k in ["variant_id_hgvs_core","hgvs_core","hgvs_c","hgvs_nt","hgvs"]:
        if k in df.columns:
            out[k]=canon(df[k])
    if "protein" in df.columns:
        out["protein"]=canon(df["protein"])
    # derived cores
    src_cols=["variant_id","variant_id_hgvs_core","hgvs_core","hgvs_c","hgvs_nt","hgvs","protein"]
    for k in src_cols:
        if k in out.columns:
            out[f"hgvs_core_from_{k}"]=to_hgvs_core(df[k] if k in df.columns else out[k])
            out[f"protein_from_{k}"]=to_protein(df[k] if k in df.columns else out[k])
            out[f"chrpos_from_{k}"]=to_chrpos(df[k] if k in df.columns else out[k])
    out=out.dropna(axis=1, how="all")
    out.columns=[f"{name}__{c}" for c in out.columns]
    return out

def best_overlap(split_df: pd.DataFrame, label_df: pd.DataFrame):
    s_keys=build_keyframe(split_df, "s")
    l_keys=build_keyframe(label_df, "l")
    s_cols=[c for c in s_keys.columns if s_keys[c].notna().any()]
    l_cols=[c for c in l_keys.columns if l_keys[c].notna().any()]
    s_uniques={c:set(s_keys[c].dropna().unique().tolist()) for c in s_cols}
    l_uniques={c:set(l_keys[c].dropna().unique().tolist()) for c in l_cols}
    best=("","",0)
    for sc in s_cols:
        for lc in l_cols:
            inter=len(s_uniques[sc] & l_uniques[lc])
            if inter>best[2]:
                best=(sc,lc,inter)
    return best, s_keys, l_keys

def audit_site_leakage(df: pd.DataFrame):
    if "site_id" in df.columns:
        g=df.groupby("site_id")["split"].nunique()
        bad=g[g>1]
        if not bad.empty: raise AssertionError(f"Site leakage for site_id(s): {bad.index.tolist()}")

# ---------- discover ----------
split_cands=find_candidates(["splits_loco_site.csv","splits_loco_site.tsv","splits_loco_site.parquet"])
split_path=choose_best(split_cands, score_split_path)
if not split_path: raise FileNotFoundError("No splits_loco_site.* found.")
label_cands=find_candidates(["TP53_delta_top.csv","TP53_delta_top.tsv","TP53_delta_top.parquet",
                             "deploy_scored.csv","deploy_scored.tsv","deploy_scored.parquet"])
label_path=choose_best(label_cands, score_label_path)
if not label_path: raise FileNotFoundError("No TP53 label source found.")
print(f"[auto] split-map: {split_path}")
print(f"[auto] labels:    {label_path}")

# ---------- load & prep ----------
split_raw=read_table_auto(split_path)
labels_raw=read_table_auto(label_path)

split=filter_tp53_loco(split_raw.copy())
split=coalesce_split_cols(split)
split_wide=split.copy()
if "variant_id" not in split.columns:
    split["variant_id"]=pd.NA
else:
    split["variant_id"]=to_string(split["variant_id"])
split=split[["variant_id","split"]].dropna(subset=["split"]).drop_duplicates()

labels=filter_tp53_loco(labels_raw.copy())
labels=ensure_label01(labels)
labels_wide=labels.copy()

# ---------- try best-overlap ----------
(best_s, best_l, best_n), s_keys, l_keys = best_overlap(split_wide, labels_wide)
print(f"[diag] best overlap: {best_n} via {best_s} ⇄ {best_l}")

def attempt_join(split_wide, labels_wide, s_keys, l_keys, s_col, l_col):
    s_tmp=split_wide.copy()
    l_tmp=labels_wide.copy()
    s_tmp["__join_key__"]=s_keys[s_col]
    l_tmp["__join_key__"]=l_keys[l_col]
    s_tmp=s_tmp[~s_tmp["__join_key__"].isna()].copy()
    l_tmp=l_tmp[~l_tmp["__join_key__"].isna()].copy()
    l_tmp=l_tmp.drop_duplicates(["__join_key__"])
    s_tmp=coalesce_split_cols(s_tmp)
    merged=s_tmp.merge(l_tmp[["__join_key__","label"]], on="__join_key__", how="left")
    # choose a stable variant_id: prefer existing, else use join key
    vid = merged["variant_id"] if "variant_id" in merged.columns else pd.Series([pd.NA]*len(merged), dtype="string")
    merged["variant_id"]=to_string(vid).fillna(to_string(merged["__join_key__"]))
    out=merged[["variant_id","split","label"]].dropna(subset=["split"]).drop_duplicates()
    return out

skel=None
if best_n>0:
    try:
        skel=attempt_join(split_wide, labels_wide, s_keys, l_keys, best_s, best_l)
    except Exception as e:
        print("[diag] best-pair join failed:", e)

if skel is None or skel.empty:
    print("[diag] trying fallback key-pairs…")
    def cols_like(keys, pat): return [c for c in keys.columns if re.search(pat, c)]
    s_order = (cols_like(s_keys, r"^s__variant_id$") + cols_like(s_keys, r"hgvs_core") +
               cols_like(s_keys, r"protein") + cols_like(s_keys, r"chrpos"))
    l_order = (cols_like(l_keys, r"^l__variant_id$") + cols_like(l_keys, r"hgvs_core") +
               cols_like(l_keys, r"protein") + cols_like(l_keys, r"chrpos"))
    for sc in s_order:
        for lc in l_order:
            inter=len(set(s_keys[sc].dropna().unique()) & set(l_keys[lc].dropna().unique()))
            if inter==0: continue
            skel_try=attempt_join(split_wide, labels_wide, s_keys, l_keys, sc, lc)
            if not skel_try.empty:
                print(f"[diag] success via {sc} ⇄ {lc} | matched rows: {len(skel_try)}")
                skel=skel_try
                break
        if skel is not None and not skel.empty: break

# if still empty: flip to deploy_scored.*
if skel is None or skel.empty:
    print("[warn] join empty — switching to deploy_scored.*")
    fb=[p for p in label_cands if "deploy_scored" in p.name]
    fb_path=choose_best(fb, score_label_path)
    if not fb_path: raise AssertionError("No deploy_scored.* found as fallback.")
    label_path=fb_path
    labels_raw=read_table_auto(label_path)
    labels=filter_tp53_loco(labels_raw.copy())
    labels=ensure_label01(labels)
    labels_wide=labels.copy()
    (best_s, best_l, best_n), s_keys, l_keys = best_overlap(split_wide, labels_wide)
    print(f"[diag] (fallback) best overlap: {best_n} via {best_s} ⇄ {best_l}")
    if best_n>0:
        skel=attempt_join(split_wide, labels_wide, s_keys, l_keys, best_s, best_l)
    if skel is None or skel.empty:
        # exhaustive with fallback
        def cols_like(keys, pat): return [c for c in keys.columns if re.search(pat, c)]
        s_order = (cols_like(s_keys, r"^s__variant_id$") + cols_like(s_keys, r"hgvs_core") +
                   cols_like(s_keys, r"protein") + cols_like(s_keys, r"chrpos"))
        l_order = (cols_like(l_keys, r"^l__variant_id$") + cols_like(l_keys, r"hgvs_core") +
                   cols_like(l_keys, r"protein") + cols_like(l_keys, r"chrpos"))
        for sc in s_order:
            for lc in l_order:
                inter=len(set(s_keys[sc].dropna().unique()) & set(l_keys[lc].dropna().unique()))
                if inter==0: continue
                skel_try=attempt_join(split_wide, labels_wide, s_keys, l_keys, sc, lc)
                if not skel_try.empty:
                    print(f"[diag] (fallback) success via {sc} ⇄ {lc} | matched rows: {len(skel_try)}")
                    skel=skel_try
                    break
            if skel is not None and not skel.empty: break

# final checks
if skel is None or skel.empty:
    print("[fail] Still 0 rows. Dumping peeks for forensics:")
    print("[split] cols:", list(split_wide.columns)[:40])
    print("[labels] cols:", list(labels_wide.columns)[:40])
    print("[split] head:\n", split_wide.head(10))
    print("[labels] head:\n", labels_wide.head(10))
    raise SystemExit(1)

skel=skel.dropna(subset=["label","split"]).copy()
skel["split"]=skel["split"].astype(int)
skel["label"]=skel["label"].astype(int)

# site leakage audit (if site_id exists in split_wide)
try:
    tmp=split_wide.copy()
    if "variant_id" not in tmp.columns: tmp["variant_id"]=pd.NA
    tmp=tmp.merge(skel[["variant_id","split"]].drop_duplicates(), on=["variant_id","split"], how="right")
    audit_site_leakage(tmp)
except AssertionError as e:
    raise

# robust per-split counts
vc=skel.groupby("split")["label"].value_counts(dropna=False)
cnt=vc.unstack(fill_value=0).reindex(columns=[0,1], fill_value=0)
cnt.columns=["n_neg","n_pos"]; cnt["n"]=cnt["n_neg"]+cnt["n_pos"]
print("Per-split counts:\n", cnt.sort_index())

present=set(skel["split"].unique().tolist())
if not present.issubset(EXPECTED_SPLITS_STRICT | {3}):
    print(f"[warn] unexpected splits present: {sorted(present)}")
if EXPECTED_PER_SPLIT:
    bad=cnt[cnt["n"]!=EXPECTED_PER_SPLIT]
    if not bad.empty: print("[warn] n != expected per split:\n", bad)
if EXPECTED_POS is not None:
    badp=cnt[cnt["n_pos"]!=EXPECTED_POS]
    if not badp.empty: print("[warn] n_pos != expected:\n", badp)
if EXPECTED_NEG is not None:
    badn=cnt[cnt["n_neg"]!=EXPECTED_NEG]
    if not badn.empty: print("[warn] n_neg != expected:\n", badn)

# write out
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path=OUT_DIR/OUT_FILE
skel.drop_duplicates(["variant_id","split"]).to_csv(out_path, index=False)
print(f"[saved] {out_path}  rows={len(skel)}  md5={md5sum(out_path)}")
print(json.dumps({
    "skeleton_path": str(out_path),
    "md5": md5sum(out_path),
    "run_id": RUN_ID,
    "split_map": str(split_path),
    "labels": str(label_path),
    "rows": len(skel),
    "splits_present": sorted(present),
    "per_split_counts": cnt.reset_index().to_dict(orient="records"),
}, indent=2))

print("\nNext → attach p_wt + context from this RUN to the skeleton; recompute per-fold AUROC/PR/Brier; 10k boot CIs (seeds AUROC=42, PR=43) + one-sided Wilcoxon.")


[auto] split-map: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv
[auto] labels:    /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv
[diag] best overlap: 25 via s__variant_id ⇄ l__variant_id
[diag] trying fallback key-pairs…
[warn] join empty — switching to deploy_scored.*
[diag] (fallback) best overlap: 197 via s__variant_id ⇄ l__variant_id
[fail] Still 0 rows. Dumping peeks for forensics:
[split] cols: ['variant_id', 'gene', 'fold', 'split']
[labels] cols: ['variant_id', 'gene', 'label', 'logit_pair', 'p_pair', 'fold', 'p_wt', 'p_blend_logitavg', 'p_blend_rankavg', 'p_blend_max', 'proba', 'call', 'threshold_used']
[split] head:
                    variant_id  gene          fold  split
74  TP53:NM_000546.6:1001:G>T  TP53  loco_site:k1   <NA>
75  TP53:NM_000546.6:1003:C>T  TP53  loco_site:k1   <NA>
76  TP53:NM_000546.6:1004:G>T  TP53  loco_site:k1  

SystemExit: 1

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [41]:
# TP53 LOCO-site SKELETON (variant_id, split, label) — fixes NA split by parsing fold="loco_site:k{0..4}"
# Policy locks: LOCO-site only; exclude LOGO; mapping fixed {gene: TP53, codon_col: p_wt, context_col: context}

import re, json, hashlib
from pathlib import Path
import pandas as pd

RUN_ID = "calibrated_robust_20250820_000001"
SEARCH_ROOTS = [
    Path("/content/drive/MyDrive/Cancer_gene_database/artifacts"),
    Path("/content/drive/MyDrive"),
    Path("/content"),
    Path.cwd(),
]
OUT_DIR = Path(f"/content/drive/MyDrive/Cancer_gene_database/artifacts/{RUN_ID}/ci_pairable")
OUT_FILE = "skeleton_loco_site_tp53.csv"

# soft expectations
EXPECTED_SPLITS_STRICT = {0,1,2,4}  # accept 3 if present
EXPECTED_PER_SPLIT = 25
EXPECTED_POS, EXPECTED_NEG = 11, 14

# -------- mount (idempotent) --------
try:
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
except Exception as e:
    print("[warn] drive mount:", e)

# -------- utils --------
def md5sum(p: Path) -> str:
    h = hashlib.md5()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def find_candidates(names, roots=SEARCH_ROOTS, max_hits=4000):
    if isinstance(names, str): names=[names]
    seen=[]
    for root in roots:
        if not root.exists(): continue
        for nm in names:
            for pat in (f"**/{nm}", f"**/{nm}.*"):
                for p in root.glob(pat):
                    if p.is_file():
                        sp=str(p)
                        if sp not in seen: seen.append(sp)
                        if len(seen)>=max_hits: break
            if len(seen)>=max_hits: break
        if len(seen)>=max_hits: break
    return [Path(s) for s in seen]

def score_split_path(p: Path) -> int:
    s=0
    s+=5 if p.name.lower().startswith("splits_loco_site") else 0
    s+=3 if p.suffix.lower() in {".csv",".tsv",".parquet"} else 0
    s+=2 if "sentiment_run_plus_variants" in str(p.parent) else 0
    s+=2 if list(p.parent.glob("tp53_counts_strict_loco.*")) else 0
    s+=1 if p.stat().st_size>0 else 0
    return s

def score_label_path(p: Path) -> int:
    s=0
    s+=4 if "TP53_delta_top" in p.name else 0
    s+=3 if "deploy_scored" in p.name else 0
    s+=2 if "advantage" in str(p.parent) else 0
    s+=1 if p.stat().st_size>0 else 0
    return s

def choose_best(cands, scorer):
    if not cands: return None
    return sorted(cands, key=lambda p: (-scorer(p), len(str(p))))[0]

def read_table_auto(p: Path) -> pd.DataFrame:
    suf=p.suffix.lower()
    if suf==".parquet": return pd.read_parquet(p)
    try: return pd.read_csv(p)
    except Exception: return pd.read_csv(p, sep="\t")

def to_string(s: pd.Series) -> pd.Series:
    return s.astype("string")

# -------- LOCO-site filters & split derivation --------
def filter_tp53(df: pd.DataFrame) -> pd.DataFrame:
    if "gene" in df.columns:
        df = df[df["gene"].astype(str).str.upper() == "TP53"].copy()
    return df

def exclude_logo(df: pd.DataFrame) -> pd.DataFrame:
    # drop rows that explicitly say LOGO anywhere in cv/mode/schema/tag columns
    for col in [c for c in df.columns if c.lower() in {"cv","mode","schema","tag"}]:
        mask = df[col].astype(str).str.contains(r"\blogo\b", case=False, na=False)
        if mask.any(): df = df[~mask].copy()
    return df

def keep_loco_site(df: pd.DataFrame) -> pd.DataFrame:
    # retain rows that look like LOCO-site in any of these columns
    site_cols = [c for c in df.columns if c.lower() in {"fold","cv","mode","schema","tag"}]
    if not site_cols:
        return df  # nothing to check; assume upstream file is LOCO-site
    mask_any = pd.Series(False, index=df.index)
    for col in site_cols:
        mask_any = mask_any | df[col].astype(str).str.contains(r"loco[_\-]?site", case=False, na=False)
    if mask_any.any():
        return df[mask_any].copy()
    return df  # fallback: don’t over-filter

def derive_split_from_fold(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create/repair numeric 'split' from:
      - split already numeric
      - split like 'k3' / 'fold_k3'
      - fold like 'loco_site:k3'
    """
    df = df.copy()
    # if split exists and is numeric → keep
    if "split" in df.columns:
        s = pd.to_numeric(df["split"], errors="coerce")
    else:
        s = pd.Series(pd.NA, index=df.index, dtype="Float64")

    if s.isna().all():
        # try from 'fold'
        if "fold" in df.columns:
            f = to_string(df["fold"])
            # extract k{d}
            k = f.str.extract(r"[kK]\s*(\d+)", expand=False)
            s = pd.to_numeric(k, errors="coerce")
        # try any column with 'split' in its name
        if s.isna().all():
            split_like = [c for c in df.columns if re.search(r"split", c, flags=re.I)]
            for c in split_like:
                cand = pd.to_numeric(df[c], errors="coerce")
                if cand.notna().any():
                    s = cand
                    break
    df["split"] = s.astype("Int64")
    return df

# -------- load inputs --------
split_path = choose_best(find_candidates(["splits_loco_site.csv","splits_loco_site.tsv","splits_loco_site.parquet"]), score_split_path)
label_path = choose_best(find_candidates(["TP53_delta_top.csv","TP53_delta_top.tsv","TP53_delta_top.parquet",
                                          "deploy_scored.csv","deploy_scored.tsv","deploy_scored.parquet"]), score_label_path)
if not split_path: raise FileNotFoundError("splits_loco_site.* not found under search roots.")
if not label_path: raise FileNotFoundError("TP53 label source not found.")
print(f"[auto] split-map: {split_path}")
print(f"[auto] labels:    {label_path}")

split_raw = read_table_auto(split_path)
labels_raw = read_table_auto(label_path)

# -------- prep split-map (TP53 + LOCO-site + numeric split) --------
split = split_raw.copy()
split = filter_tp53(split)
split = exclude_logo(split)
split = keep_loco_site(split)
split = derive_split_from_fold(split)

# derive variant_id if missing (but your file has it)
if "variant_id" not in split.columns:
    # try common HGVS core or genomic pieces if needed
    if all(c in split.columns for c in ["chr","pos","ref","alt"]):
        vid = split["chr"].astype(str)+":"+split["pos"].astype(str)+"_"+split["ref"].astype(str)+">"+split["alt"].astype(str)
        split["variant_id"] = vid
    else:
        raise AssertionError("split-map missing variant_id and chr/pos/ref/alt to build it.")

# keep just what we need and drop rows without split
split = split[["variant_id","split"]].copy()
split["variant_id"] = to_string(split["variant_id"])
pre_drop = len(split)
split = split.dropna(subset=["split"]).drop_duplicates()
print(f"[diag] split-map rows: {pre_drop} → {len(split)} after deriving numeric split")

# -------- prep labels (TP53; label→{0,1}; ensure bi-class; prefer existing variant_id) --------
labels = labels_raw.copy()
labels = filter_tp53(labels)
labels = exclude_logo(labels)

# map labels to 0/1 from likely columns
def ensure_label01(df: pd.DataFrame) -> pd.DataFrame:
    cand=[c for c in df.columns if c.lower() in {"label","class","y","oncogenic","is_driver","is_pathogenic","call"}]
    if not cand: raise AssertionError("No label-like column in labels file.")
    for c in cand:
        s=df[c]
        if pd.api.types.is_bool_dtype(s) or pd.api.types.is_numeric_dtype(s):
            mapped=pd.Series(s, dtype="Int64")
            uniq=set(mapped.dropna().unique().tolist())
            if uniq <= {0,1}:
                return df.rename(columns={c:"label"}).assign(label=mapped)
            if uniq <= {-1,1}:
                return df.rename(columns={c:"label"}).assign(label=(mapped>0).astype("Int64"))
        ser=to_string(s).str.lower()
        pos={"pos","positive","driver","pathogenic","oncogenic","pos_high","likely_pathogenic"}
        neg={"neg","negative","benign","likely_benign","wt","neutral"}
        if ser.isin(pos|neg).any():
            lab=ser.map(lambda x: 1 if x in pos else (0 if x in neg else pd.NA)).astype("Int64")
            if lab.notna().any(): return df.rename(columns={c:"_raw_label"}).assign(label=lab)
        if c.lower()=="call":
            lab=ser.map(lambda x: 1 if str(x).startswith("pos") else (0 if str(x).startswith("neg") else pd.NA)).astype("Int64")
            if lab.notna().any(): return df.rename(columns={c:"_raw_call"}).assign(label=lab)
    raise AssertionError("Failed to coerce labels to {0,1} for labels file.")

labels = ensure_label01(labels)

# ensure variant_id present
if "variant_id" not in labels.columns:
    raise AssertionError("labels file missing variant_id")

labels = labels[["variant_id","label"]].dropna().drop_duplicates(subset=["variant_id"])
labels["variant_id"] = to_string(labels["variant_id"])

# if single-class, switch to deploy_scored.*
uniq = set(labels["label"].dropna().unique().tolist())
if uniq in [{1}, {0}]:
    print(f"[warn] {label_path.name} has single class {uniq}. Trying deploy_scored.* fallback.")
    fb = choose_best([p for p in find_candidates(["deploy_scored.csv","deploy_scored.tsv","deploy_scored.parquet"]) if "deploy_scored" in p.name], score_label_path)
    if not fb:
        raise AssertionError("Fallback deploy_scored.* not found, need bi-class labels.")
    label_path = fb
    labels_raw = read_table_auto(label_path)
    labels = filter_tp53(labels_raw.copy())
    labels = exclude_logo(labels)
    labels = ensure_label01(labels)
    labels = labels[["variant_id","label"]].dropna().drop_duplicates(subset=["variant_id"])
    labels["variant_id"] = to_string(labels["variant_id"])
    uniq = set(labels["label"].dropna().unique().tolist())
    print(f"[auto] switched labels → {label_path.name}; classes={sorted(uniq)}")

if not (0 in uniq and 1 in uniq):
    raise AssertionError(f"Labels not bi-class after fallback. Classes={sorted(uniq)}")

# -------- merge to SKELETON --------
skel = split.merge(labels, on="variant_id", how="left", validate="m:1")
missing = skel["label"].isna().sum()
if missing>0:
    # hard fail; we need labels for all split-map variants to match counts exactly
    missing_examples = skel[skel["label"].isna()].head(5)["variant_id"].tolist()
    raise AssertionError(f"Missing labels for {missing} variant(s). Example: {missing_examples}")

skel["split"] = skel["split"].astype(int)
skel["label"] = skel["label"].astype(int)
skel = skel[["variant_id","split","label"]].drop_duplicates()

# -------- counts & audits --------
vc = skel.groupby("split")["label"].value_counts(dropna=False)
cnt = vc.unstack(fill_value=0).reindex(columns=[0,1], fill_value=0)
cnt.columns = ["n_neg","n_pos"]; cnt["n"] = cnt["n_neg"] + cnt["n_pos"]
print("Per-split counts:\n", cnt.sort_index())

present=set(skel["split"].unique().tolist())
if not present.issubset(EXPECTED_SPLITS_STRICT | {3}):
    print(f"[warn] unexpected splits present: {sorted(present)}")
if EXPECTED_PER_SPLIT:
    bad=cnt[cnt["n"]!=EXPECTED_PER_SPLIT]
    if not bad.empty: print("[warn] n != expected per split:\n", bad)
if EXPECTED_POS is not None:
    badp=cnt[cnt["n_pos"]!=EXPECTED_POS]
    if not badp.empty: print("[warn] n_pos != expected:\n", badp)
if EXPECTED_NEG is not None:
    badn=cnt[cnt["n_neg"]!=EXPECTED_NEG]
    if not badn.empty: print("[warn] n_neg != expected:\n", badn)

# -------- write out --------
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / OUT_FILE
skel.to_csv(out_path, index=False)
print(f"[saved] {out_path}  rows={len(skel)}  md5={md5sum(out_path)}")
print(json.dumps({
    "skeleton_path": str(out_path),
    "md5": md5sum(out_path),
    "run_id": RUN_ID,
    "split_map": str(split_path),
    "labels": str(label_path),
    "rows": len(skel),
    "splits_present": sorted(present),
    "per_split_counts": cnt.reset_index().to_dict(orient="records"),
}, indent=2))

print("\nNext → attach p_wt + context from this RUN to the skeleton; recompute per-fold AUROC/PR/Brier; 10k boot CIs (seeds AUROC=42, PR=43) + one-sided Wilcoxon.")


[auto] split-map: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv
[auto] labels:    /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv
[diag] split-map rows: 985 → 985 after deriving numeric split


AssertionError: Missing labels for 860 variant(s). Example: ['TP53:NM_000546.6:1001:G>T', 'TP53:NM_000546.6:1003:C>T', 'TP53:NM_000546.6:1004:G>T', 'TP53:NM_000546.6:1006:G>C', 'TP53:NM_000546.6:1009:C>T']

In [42]:
# TP53 LOCO-site SKELETON (variant_id, split, label) — intersection-only, fixed split derivation
# Policy: LOCO-site only; exclude LOGO; mapping locked {gene: TP53, codon_col: p_wt, context_col: context}

import re, json, hashlib
from pathlib import Path
import pandas as pd

RUN_ID = "calibrated_robust_20250820_000001"
SEARCH_ROOTS = [
    Path("/content/drive/MyDrive/Cancer_gene_database/artifacts"),
    Path("/content/drive/MyDrive"),
    Path("/content"),
    Path.cwd(),
]
OUT_DIR = Path(f"/content/drive/MyDrive/Cancer_gene_database/artifacts/{RUN_ID}/ci_pairable")
OUT_FILE = "skeleton_loco_site_tp53.csv"

# soft checks (warn-only)
EXPECTED_SPLITS_STRICT = {0,1,2,4}  # include 3 if present
EXPECTED_PER_SPLIT = 25
EXPECTED_POS, EXPECTED_NEG = 11, 14

# --- mount (idempotent) ---
try:
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
except Exception as e:
    print("[warn] drive mount:", e)

# --- utils ---
def md5sum(p: Path) -> str:
    h = hashlib.md5()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def find_candidates(names, roots=SEARCH_ROOTS, max_hits=4000):
    if isinstance(names, str): names=[names]
    seen=[]
    for root in roots:
        if not root.exists(): continue
        for nm in names:
            for pat in (f"**/{nm}", f"**/{nm}.*"):
                for p in root.glob(pat):
                    if p.is_file():
                        sp=str(p)
                        if sp not in seen: seen.append(sp)
                        if len(seen)>=max_hits: break
            if len(seen)>=max_hits: break
        if len(seen)>=max_hits: break
    return [Path(s) for s in seen]

def score_split_path(p: Path) -> int:
    s=0
    s+=5 if p.name.lower().startswith("splits_loco_site") else 0
    s+=3 if p.suffix.lower() in {".csv",".tsv",".parquet"} else 0
    s+=2 if "sentiment_run_plus_variants" in str(p.parent) else 0
    s+=2 if list(p.parent.glob("tp53_counts_strict_loco.*")) else 0
    s+=1 if p.stat().st_size>0 else 0
    return s

def score_label_path(p: Path) -> int:
    s=0
    s+=4 if "TP53_delta_top" in p.name else 0
    s+=3 if "deploy_scored" in p.name else 0
    s+=2 if "advantage" in str(p.parent) else 0
    s+=1 if p.stat().st_size>0 else 0
    return s

def choose_best(cands, scorer):
    if not cands: return None
    return sorted(cands, key=lambda p: (-scorer(p), len(str(p))))[0]

def read_table_auto(p: Path) -> pd.DataFrame:
    suf=p.suffix.lower()
    if suf==".parquet": return pd.read_parquet(p)
    try: return pd.read_csv(p)
    except Exception: return pd.read_csv(p, sep="\t")

def to_string(s: pd.Series) -> pd.Series:
    return s.astype("string")

# --- LOCO-site filters & split derivation ---
def filter_tp53(df: pd.DataFrame) -> pd.DataFrame:
    return df[df.get("gene","").astype(str).str.upper()=="TP53"].copy() if "gene" in df.columns else df

def exclude_logo(df: pd.DataFrame) -> pd.DataFrame:
    for col in [c for c in df.columns if c.lower() in {"cv","mode","schema","tag"}]:
        mask = df[col].astype(str).str.contains(r"\blogo\b", case=False, na=False)
        if mask.any(): df = df[~mask].copy()
    return df

def keep_loco_site(df: pd.DataFrame) -> pd.DataFrame:
    site_cols = [c for c in df.columns if c.lower() in {"fold","cv","mode","schema","tag"}]
    if not site_cols: return df
    mask_any = pd.Series(False, index=df.index)
    for col in site_cols:
        mask_any = mask_any | df[col].astype(str).str.contains(r"loco[_\-]?site", case=False, na=False)
    return df[mask_any].copy() if mask_any.any() else df

def derive_split_from_fold(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # if split already numeric, keep; else try extracting k#
    split_num = pd.to_numeric(df.get("split", pd.Series(pd.NA, index=df.index)), errors="coerce")
    if split_num.isna().all() and "fold" in df.columns:
        k = to_string(df["fold"]).str.extract(r"[kK]\s*(\d+)", expand=False)
        split_num = pd.to_numeric(k, errors="coerce")
    # last-resort: any column with "split" in name
    if split_num.isna().all():
        for c in [c for c in df.columns if re.search(r"split", c, flags=re.I)]:
            cand = pd.to_numeric(df[c], errors="coerce")
            if cand.notna().any():
                split_num = cand
                break
    df["split"] = split_num.astype("Int64")
    return df

def ensure_label01(df: pd.DataFrame) -> pd.DataFrame:
    cand=[c for c in df.columns if c.lower() in {"label","class","y","oncogenic","is_driver","is_pathogenic","call"}]
    if not cand: raise AssertionError("No label-like column in labels file.")
    for c in cand:
        s=df[c]
        if pd.api.types.is_bool_dtype(s) or pd.api.types.is_numeric_dtype(s):
            mapped=pd.Series(s, dtype="Int64")
            uniq=set(mapped.dropna().unique().tolist())
            if uniq <= {0,1}:
                return df.rename(columns={c:"label"}).assign(label=mapped)
            if uniq <= {-1,1}:
                return df.rename(columns={c:"label"}).assign(label=(mapped>0).astype("Int64"))
        ser=to_string(s).str.lower()
        pos={"pos","positive","driver","pathogenic","oncogenic","pos_high","likely_pathogenic"}
        neg={"neg","negative","benign","likely_benign","wt","neutral"}
        if ser.isin(pos|neg).any():
            lab=ser.map(lambda x: 1 if x in pos else (0 if x in neg else pd.NA)).astype("Int64")
            if lab.notna().any(): return df.rename(columns={c:"_raw_label"}).assign(label=lab)
        if c.lower()=="call":
            lab=ser.map(lambda x: 1 if str(x).startswith("pos") else (0 if str(x).startswith("neg") else pd.NA)).astype("Int64")
            if lab.notna().any(): return df.rename(columns={c:"_raw_call"}).assign(label=lab)
    raise AssertionError("Failed to coerce labels to {0,1}.")

# --- discover inputs ---
split_path = choose_best(find_candidates(["splits_loco_site.csv","splits_loco_site.tsv","splits_loco_site.parquet"]), score_split_path)
label_path = choose_best(find_candidates(["TP53_delta_top.csv","TP53_delta_top.tsv","TP53_delta_top.parquet",
                                          "deploy_scored.csv","deploy_scored.tsv","deploy_scored.parquet"]), score_label_path)
if not split_path: raise FileNotFoundError("splits_loco_site.* not found under search roots.")
if not label_path: raise FileNotFoundError("TP53 label source not found.")
print(f"[auto] split-map: {split_path}")
print(f"[auto] labels:    {label_path}")

# --- load & prep split-map ---
split_raw = read_table_auto(split_path)
split = split_raw.pipe(filter_tp53).pipe(exclude_logo).pipe(keep_loco_site).pipe(derive_split_from_fold)
if "variant_id" not in split.columns:
    raise AssertionError("split-map missing variant_id; cannot join.")
split["variant_id"] = to_string(split["variant_id"])
split = split[["variant_id","split"]].dropna(subset=["split"]).drop_duplicates()
print(f"[diag] split-map LOCO TP53 rows with split: {len(split)}")

# --- load & prep labels (bi-class, prefer TP53_delta_top else deploy_scored) ---
labels_raw = read_table_auto(label_path)
labels = labels_raw.pipe(filter_tp53).pipe(exclude_logo)
labels = ensure_label01(labels)
if "variant_id" not in labels.columns:
    raise AssertionError("labels file missing variant_id.")
labels = labels[["variant_id","label"]].dropna().drop_duplicates(subset=["variant_id"])
labels["variant_id"] = to_string(labels["variant_id"])
uniq = set(labels["label"].dropna().unique().tolist())
if uniq in [{0},{1}]:
    print(f"[warn] {label_path.name} is single-class {uniq}. Trying deploy_scored.*")
    fb = choose_best([p for p in find_candidates(["deploy_scored.csv","deploy_scored.tsv","deploy_scored.parquet"]) if "deploy_scored" in p.name], score_label_path)
    if not fb: raise AssertionError("Need bi-class labels; deploy_scored.* not found.")
    label_path = fb
    labels_raw = read_table_auto(label_path)
    labels = labels_raw.pipe(filter_tp53).pipe(exclude_logo)
    labels = ensure_label01(labels)
    labels = labels[["variant_id","label"]].dropna().drop_duplicates(subset=["variant_id"])
    labels["variant_id"] = to_string(labels["variant_id"])
    uniq = set(labels["label"].dropna().unique().tolist())
    print(f"[auto] switched labels → {label_path.name}; classes={sorted(uniq)}")
if not (0 in uniq and 1 in uniq):
    raise AssertionError(f"Labels not bi-class. Classes={sorted(uniq)}")

# --- INNER JOIN (intersection only) ---
skel = split.merge(labels, on="variant_id", how="inner", validate="m:1")[["variant_id","split","label"]].drop_duplicates()
if skel.empty:
    # show a brief diff to debug keys
    s_only = set(split["variant_id"]) - set(labels["variant_id"])
    l_only = set(labels["variant_id"]) - set(split["variant_id"])
    print("[fail] 0-row join. Split-only samples:", list(sorted(s_only))[:5])
    print("[fail] Label-only samples:", list(sorted(l_only))[:5])
    raise SystemExit(1)

# --- per-split counts + coverage ---
skel["split"] = skel["split"].astype(int)
skel["label"] = skel["label"].astype(int)
vc = skel.groupby("split")["label"].value_counts(dropna=False)
cnt = vc.unstack(fill_value=0).reindex(columns=[0,1], fill_value=0)
cnt.columns = ["n_neg","n_pos"]; cnt["n"] = cnt["n_neg"] + cnt["n_pos"]
coverage = len(skel) / len(split) if len(split) else 0.0

print("\nPer-split counts (intersection):\n", cnt.sort_index())
print(f"\nCoverage of split-map by labels: {len(skel)}/{len(split)} = {coverage:.2%}")
present = set(skel["split"].unique().tolist())
if not present.issubset(EXPECTED_SPLITS_STRICT | {3}):
    print(f"[warn] unexpected splits present: {sorted(present)}")
# soft checks (warn only)
if EXPECTED_PER_SPLIT:
    bad = cnt[cnt["n"] != EXPECTED_PER_SPLIT]
    if not bad.empty: print("[warn] n != expected per split:\n", bad)
if EXPECTED_POS is not None:
    badp = cnt[cnt["n_pos"] != EXPECTED_POS]
    if not badp.empty: print("[warn] n_pos != expected:\n", badp)
if EXPECTED_NEG is not None:
    badn = cnt[cnt["n_neg"] != EXPECTED_NEG]
    if not badn.empty: print("[warn] n_neg != expected:\n", badn)

# --- write out ---
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / OUT_FILE
skel.to_csv(out_path, index=False)
print(f"\n[saved] {out_path}  rows={len(skel)}  md5={md5sum(out_path)}")
print(json.dumps({
    "skeleton_path": str(out_path),
    "md5": md5sum(out_path),
    "run_id": RUN_ID,
    "split_map": str(split_path),
    "labels": str(label_path),
    "rows": len(skel),
    "splits_present": sorted(present),
    "per_split_counts": cnt.reset_index().to_dict(orient="records"),
    "coverage": coverage,
}, indent=2))

print("\nNext → attach p_wt + context for these rows, recompute per-fold AUROC/PR/Brier, then 10k bootstrap CIs (seeds AUROC=42, PR=43) + one-sided Wilcoxon.")


[auto] split-map: /content/drive/MyDrive/Cancer_gene_database/artifacts/sentiment_run_plus_variants_20250816_190607/splits_loco_site.csv
[auto] labels:    /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv
[diag] split-map LOCO TP53 rows with split: 985

Per-split counts (intersection):
        n_neg  n_pos   n
split                  
1         14     11  25
2         14     11  25
3         14     11  25
4         14     11  25
5         14     11  25

Coverage of split-map by labels: 125/985 = 12.69%
[warn] unexpected splits present: [1, 2, 3, 4, 5]

[saved] /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/skeleton_loco_site_tp53.csv  rows=125  md5=09ecc69542bef5741c2e672953cd650f
{
  "skeleton_path": "/content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/skeleton_loco_site_tp53.csv",
  "md5": "09ecc69542bef5741c2e672953cd6

In [43]:
# TP53 LOCO-site — unify preds (p_wt + context), fix split to 0..4, compute n=5 metrics + CIs + Wilcoxon
# RUN: calibrated_robust_20250820_000001
# Policy locks: LOCO-site only; exclude LOGO; no imputation; seeds AUROC=42, PR=43
# Mapping locked: {gene: TP53, codon_col: p_wt, context_col: context}

import re, json, hashlib, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")

# ============================
# CONFIG
# ============================
RUN_ID = "calibrated_robust_20250820_000001"
ARTIFACTS_ROOT = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
RUN_DIR = ARTIFACTS_ROOT / RUN_ID
CI_DIR = RUN_DIR / "ci_pairable"

# Inputs we already built
SKELETON_PATH = CI_DIR / "skeleton_loco_site_tp53.csv"  # from previous step (125 rows)
LABELS_PWT_PATH = RUN_DIR / "advantage" / "TP53_delta_top.csv"  # has p_wt
# Optional override if auto-discovery fails:
PATH_CONTEXT_PREDS = None  # e.g., RUN_DIR / "context" / "tp53_context_preds.csv"

# Outputs to write
UNIFIED_PREDS_PATH = CI_DIR / "preds_loco_site_tp53.csv"
PER_FOLD_METRICS_PATH = CI_DIR / "tp53_per_fold_metrics_n5.csv"
SUMMARY_JSON_PATH = CI_DIR / "tp53_loco_strict_summary.json"

# Seeds
SEED_AUROC = 42
SEED_PR = 43
N_BOOT = 10_000

# ============================
# HELPERS
# ============================
def md5sum(p: Path) -> str:
    h = hashlib.md5()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_table_auto(p: Path) -> pd.DataFrame:
    suf = p.suffix.lower()
    if suf == ".parquet":
        return pd.read_parquet(p)
    try:
        return pd.read_csv(p)
    except Exception:
        return pd.read_csv(p, sep="\t")

def find_candidates(names, roots, max_hits=3000):
    if isinstance(names, str): names=[names]
    found=[]
    for root in roots:
        if not root.exists(): continue
        for nm in names:
            for pat in (f"**/{nm}", f"**/{nm}.*"):
                for p in root.glob(pat):
                    if p.is_file():
                        sp=str(p)
                        if sp not in found:
                            found.append(sp)
                        if len(found)>=max_hits: break
            if len(found)>=max_hits: break
        if len(found)>=max_hits: break
    return [Path(s) for s in found]

def score_context_file(p: Path) -> int:
    s=0
    s+=4 if RUN_ID in str(p) else 0
    s+=3 if re.search(r"context|flank|nt|nuc|sequence", p.name, flags=re.I) else 0
    s+=2 if p.suffix.lower() in {".csv",".tsv",".parquet"} else 0
    s+=1 if p.stat().st_size > 0 else 0
    return s

def choose_best(cands, scorer):
    if not cands: return None
    return sorted(cands, key=lambda p: (-scorer(p), len(str(p))))[0]

def ensure_prob01(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    return x.clip(lower=0.0, upper=1.0)

def remap_split_zero_based(df: pd.DataFrame) -> pd.DataFrame:
    S = set(pd.to_numeric(df["split"], errors="coerce").dropna().astype(int).unique().tolist())
    # Your skeleton had {1,2,3,4,5} → map → {0,1,2,3,4}
    if S == {1,2,3,4,5}:
        df = df.assign(split = df["split"].astype(int) - 1)
    return df

def per_fold_metrics(y, p_cod, p_ctx):
    # return dict with auroc/pr/brier for each model and deltas
    out = {}
    # AUROC / AP require both classes present
    if len(set(y)) < 2:
        return None
    out["auroc_codon"] = roc_auc_score(y, p_cod)
    out["auroc_context"] = roc_auc_score(y, p_ctx)
    out["delta_auroc"] = out["auroc_context"] - out["auroc_codon"]
    out["pr_codon"] = average_precision_score(y, p_cod)
    out["pr_context"] = average_precision_score(y, p_ctx)
    out["delta_pr"] = out["pr_context"] - out["pr_codon"]
    out["brier_codon"] = brier_score_loss(y, p_cod)
    out["brier_context"] = brier_score_loss(y, p_ctx)
    out["delta_brier"] = out["brier_context"] - out["brier_codon"]
    return out

def bootstrap_ci_deltas(df, n_boot=10_000, seed_auroc=42, seed_pr=43):
    # bootstrap per-fold by resampling within each split, then averaging deltas across splits.
    rng_roc = np.random.RandomState(seed_auroc)
    rng_pr  = np.random.RandomState(seed_pr)
    splits = sorted(df["split"].unique().tolist())
    # Pre-split indices
    idx_by_split = {s: df.index[df["split"]==s].to_numpy() for s in splits}
    # run AUROC bootstrap
    boot_d_roc = np.empty(n_boot)
    for b in range(n_boot):
        ds=[]
        for s in splits:
            idx = idx_by_split[s]
            # sample with replacement within split
            samp = rng_roc.choice(idx, size=len(idx), replace=True)
            y = df.loc[samp, "label"].to_numpy()
            p_cod = df.loc[samp, "p_wt"].to_numpy()
            p_ctx = df.loc[samp, "context"].to_numpy()
            if len(np.unique(y)) < 2:
                # degenerate resample; skip this split
                continue
            ds.append(roc_auc_score(y, p_ctx) - roc_auc_score(y, p_cod))
        boot_d_roc[b] = np.mean(ds) if ds else np.nan
    roc_ci = (np.nanpercentile(boot_d_roc, 2.5), np.nanpercentile(boot_d_roc, 97.5))
    roc_mean = np.nanmean(boot_d_roc)

    # run PR bootstrap
    boot_d_pr = np.empty(n_boot)
    for b in range(n_boot):
        ds=[]
        for s in splits:
            idx = idx_by_split[s]
            samp = rng_pr.choice(idx, size=len(idx), replace=True)
            y = df.loc[samp, "label"].to_numpy()
            p_cod = df.loc[samp, "p_wt"].to_numpy()
            p_ctx = df.loc[samp, "context"].to_numpy()
            if len(np.unique(y)) < 2:
                continue
            ds.append(average_precision_score(y, p_ctx) - average_precision_score(y, p_cod))
        boot_d_pr[b] = np.mean(ds) if ds else np.nan
    pr_ci = (np.nanpercentile(boot_d_pr, 2.5), np.nanpercentile(boot_d_pr, 97.5))
    pr_mean = np.nanmean(boot_d_pr)

    return {
        "delta_auroc_mean_boot": float(roc_mean),
        "delta_auroc_ci_low": float(roc_ci[0]),
        "delta_auroc_ci_high": float(roc_ci[1]),
        "delta_pr_mean_boot": float(pr_mean),
        "delta_pr_ci_low": float(pr_ci[0]),
        "delta_pr_ci_high": float(pr_ci[1]),
        "seed_auroc": seed_auroc,
        "seed_pr": seed_pr,
        "n_boot": n_boot,
    }

# ============================
# LOAD SKELETON + REMAP SPLITS
# ============================
SKELETON_PATH.parent.mkdir(parents=True, exist_ok=True)
skel = read_table_auto(SKELETON_PATH)
if skel.empty:
    raise RuntimeError(f"SKELETON is empty: {SKELETON_PATH}")
# remap 1..5 → 0..4 if needed
skel = remap_split_zero_based(skel)
# sanity: restrict to 0..4 only
skel = skel[skel["split"].isin([0,1,2,3,4])].copy()
print("Splits present after remap:", sorted(skel["split"].unique().tolist()))

# ============================
# LOAD p_wt (codon)
# ============================
labels = read_table_auto(LABELS_PWT_PATH)
if "variant_id" not in labels.columns or "p_wt" not in labels.columns:
    raise RuntimeError(f"{LABELS_PWT_PATH} missing required columns (variant_id, p_wt)")
pwt = labels[["variant_id","p_wt"]].dropna().copy()
pwt["p_wt"] = ensure_prob01(pwt["p_wt"])

# ============================
# FIND / LOAD context predictions
# ============================
context_df = None
context_col = None

if PATH_CONTEXT_PREDS:
    cand = Path(PATH_CONTEXT_PREDS)
    if not cand.exists():
        raise FileNotFoundError(f"PATH_CONTEXT_PREDS not found: {cand}")
    context_df = read_table_auto(cand)
else:
    # hunt under RUN first, then whole artifacts
    roots = [RUN_DIR, ARTIFACTS_ROOT]
    name_patterns = ["*context*.csv","*context*.tsv","*context*.parquet",
                     "*flank*.csv","*nt*.csv","*nuc*.csv","*sequence*.csv"]
    cands = []
    for pat in name_patterns:
        cands += find_candidates(pat, roots)
    cands = list({str(p):p for p in cands}.values())
    ctx_path = choose_best(cands, score_context_file)
    if ctx_path:
        print(f"[auto] context preds candidate: {ctx_path}")
        context_df = read_table_auto(ctx_path)

if context_df is None:
    raise RuntimeError(
        "Couldn’t auto-find context predictions.\n"
        ">> Set PATH_CONTEXT_PREDS to a file with columns [variant_id, context] (or p_context / p_nt / p_seq)."
    )

# guess the context prob column
candidate_cols = [c for c in context_df.columns if c.lower() in
                  {"context","p_context","p_nt","p_seq","p_local_context","context_prob","proba_context"}]
if not candidate_cols:
    # pivot style: try a long format with 'model' and 'proba'
    if {"model","variant_id","proba"} <= set(map(str.lower, context_df.columns)):
        df2 = context_df.rename(columns={c: c.lower() for c in context_df.columns})
        ctx_only = df2[df2["model"].str.contains("context", case=False, na=False)]
        context_df = ctx_only[["variant_id","proba"]].rename(columns={"proba":"context"})
        candidate_cols = ["context"]
    else:
        raise RuntimeError(
            "Context file found but no obvious probability column. "
            "Expected one of: context, p_context, p_nt, p_seq, p_local_context, context_prob, proba_context; "
            "or a long table with columns [variant_id, model, proba] where model~'context'."
        )

# pick first plausible column as context
context_col = candidate_cols[0]
ctx = context_df[["variant_id", context_col]].dropna().copy()
ctx = ctx.rename(columns={context_col: "context"})
ctx["context"] = ensure_prob01(ctx["context"])

# ============================
# UNIFY: join skeleton + p_wt + context
# ============================
unified = skel.merge(pwt, on="variant_id", how="left").merge(ctx, on="variant_id", how="left")
missing = unified[unified[["p_wt","context"]].isna().any(axis=1)]
if not missing.empty:
    print(f"[warn] dropping {len(missing)} row(s) with missing p_wt/context")
    unified = unified.dropna(subset=["p_wt","context"]).copy()

# integrity checks
need_folds = {0,1,2,3,4}
have_folds = set(unified["split"].unique().tolist())
if need_folds - have_folds:
    print(f"[warn] missing folds in unified preds: {sorted(need_folds - have_folds)}")

# write unified preds
UNIFIED_PREDS_PATH.parent.mkdir(parents=True, exist_ok=True)
unified.to_csv(UNIFIED_PREDS_PATH, index=False)
print(f"[saved] unified preds → {UNIFIED_PREDS_PATH}  rows={len(unified)}  md5={md5sum(UNIFIED_PREDS_PATH)}")

# ============================
# METRICS per fold + deltas
# ============================
rows=[]
for s, g in unified.groupby("split"):
    y = g["label"].astype(int).to_numpy()
    p_cod = g["p_wt"].to_numpy()
    p_ctx = g["context"].to_numpy()
    m = per_fold_metrics(y, p_cod, p_ctx)
    if m is None:
        print(f"[warn] split {s}: single-class labels; skipping metrics.")
        continue
    m.update({"split": int(s), "n": int(len(g)), "n_pos": int(g["label"].sum()), "n_neg": int(len(g)-g["label"].sum())})
    rows.append(m)

per_fold = pd.DataFrame(rows).sort_values("split")
if per_fold.empty:
    raise RuntimeError("No per-fold metrics computed; check class balance and predictions.")

# save per-fold metrics
PER_FOLD_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
per_fold.to_csv(PER_FOLD_METRICS_PATH, index=False)
print(f"[saved] per-fold metrics (n=5) → {PER_FOLD_METRICS_PATH}  md5={md5sum(PER_FOLD_METRICS_PATH)}")
print(per_fold)

# Wilcoxon one-sided on deltas (context > codon)
d_auroc = per_fold["delta_auroc"].to_numpy()
d_pr = per_fold["delta_pr"].to_numpy()
wilc_auroc = wilcoxon(d_auroc, alternative="greater", zero_method="wilcox")
wilc_pr = wilcoxon(d_pr, alternative="greater", zero_method="wilcox")

# Bootstrap CIs
boot = bootstrap_ci_deltas(unified, n_boot=N_BOOT, seed_auroc=SEED_AUROC, seed_pr=SEED_PR)

aggregate = {
    "n_folds": int(per_fold.shape[0]),
    "delta_auroc_mean": float(np.mean(d_auroc)),
    "delta_auroc_ci": [boot["delta_auroc_ci_low"], boot["delta_auroc_ci_high"]],
    "delta_pr_mean": float(np.mean(d_pr)),
    "delta_pr_ci": [boot["delta_pr_ci_low"], boot["delta_pr_ci_high"]],
    "delta_brier_mean": float(np.mean(per_fold["delta_brier"])),
    "wilcoxon_auroc_stat": float(wilc_auroc.statistic),
    "wilcoxon_auroc_p_one_sided": float(wilc_auroc.pvalue),
    "wilcoxon_pr_stat": float(wilc_pr.statistic),
    "wilcoxon_pr_p_one_sided": float(wilc_pr.pvalue),
    "seeds": {"auroc": SEED_AUROC, "pr": SEED_PR},
    "n_boot": N_BOOT,
}

print("\n=== Aggregate (n=5) ===")
print(json.dumps(aggregate, indent=2))

# ============================
# UPDATE SUMMARY JSON (append n=5 line)
# ============================
summary = {}
if SUMMARY_JSON_PATH.exists():
    try:
        summary = json.loads(SUMMARY_JSON_PATH.read_text())
    except Exception:
        summary = {}

summary.setdefault("TP53", {})
summary["TP53"]["loco_strict_n5"] = {
    "run_id": RUN_ID,
    "unified_preds": str(UNIFIED_PREDS_PATH),
    "per_fold_metrics": str(PER_FOLD_METRICS_PATH),
    "counts_per_fold": per_fold[["split","n","n_pos","n_neg"]].to_dict(orient="records"),
    "delta_auroc_mean": aggregate["delta_auroc_mean"],
    "delta_auroc_ci": aggregate["delta_auroc_ci"],
    "delta_pr_mean": aggregate["delta_pr_mean"],
    "delta_pr_ci": aggregate["delta_pr_ci"],
    "delta_brier_mean": aggregate["delta_brier_mean"],
    "wilcoxon_auroc_p_one_sided": aggregate["wilcoxon_auroc_p_one_sided"],
    "wilcoxon_pr_p_one_sided": aggregate["wilcoxon_pr_p_one_sided"],
    "seeds": aggregate["seeds"],
    "n_boot": aggregate["n_boot"],
}

SUMMARY_JSON_PATH.write_text(json.dumps(summary, indent=2))
print(f"\n[summary updated] {SUMMARY_JSON_PATH}  md5={md5sum(SUMMARY_JSON_PATH)}")

print("\nDone → You now have unified preds + n=5 metrics. If context auto-discovery failed, set PATH_CONTEXT_PREDS to the right file and re-run.")


Splits present after remap: [0, 1, 2, 3, 4]


RuntimeError: /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv missing required columns (variant_id, p_wt)

In [44]:
# TP53 LOCO-site — unify preds with robust header canon (fixes missing p_wt), compute n=5 metrics
# RUN: calibrated_robust_20250820_000001
# Locks: LOCO-site only; exclude LOGO; no imputation; seeds AUROC=42, PR=43
# Mapping: {gene: TP53, codon_col: p_wt, context_col: context}

import re, json, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")

# ===== CONFIG =====
RUN_ID = "calibrated_robust_20250820_000001"
ARTIFACTS_ROOT = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
RUN_DIR = ARTIFACTS_ROOT / RUN_ID
CI_DIR = RUN_DIR / "ci_pairable"

SKELETON_PATH = CI_DIR / "skeleton_loco_site_tp53.csv"
PWT_PRIMARY = RUN_DIR / "advantage" / "TP53_delta_top.csv"     # expected to have p_wt
PWT_FALLBACKS = [
    RUN_DIR / "advantage" / "deploy_scored.csv",
    RUN_DIR / "advantage" / "deploy_scored.tsv",
]
PATH_CONTEXT_PREDS = None   # set to a specific file if auto-discovery fails

UNIFIED_PREDS_PATH = CI_DIR / "preds_loco_site_tp53.csv"
PER_FOLD_METRICS_PATH = CI_DIR / "tp53_per_fold_metrics_n5.csv"
SUMMARY_JSON_PATH = CI_DIR / "tp53_loco_strict_summary.json"

SEED_AUROC = 42
SEED_PR = 43
N_BOOT = 10_000

# ===== HELPERS =====
def md5sum(p: Path) -> str:
    h = hashlib.md5()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_table_auto(p: Path) -> pd.DataFrame:
    suf = p.suffix.lower()
    if suf == ".parquet":
        return pd.read_parquet(p)
    try:
        return pd.read_csv(p)
    except Exception:
        return pd.read_csv(p, sep="\t")

def canon_cols(df: pd.DataFrame) -> pd.DataFrame:
    # normalize headers: lowercase, strip, collapse spaces/dots/hyphens to underscores
    mapping = {}
    for c in df.columns:
        cc = c.strip().lower()
        cc = re.sub(r"[ \.\-]+", "_", cc)
        cc = re.sub(r"__+", "_", cc)
        # collapse obvious synonyms to canonical
        if re.fullmatch(r"p[_-]?wt", cc): cc = "p_wt"
        if re.fullmatch(r"variant[_-]?id|variantid", cc): cc = "variant_id"
        if re.fullmatch(r"p[_-]?context|context_prob|proba_context|p_nt|p_seq|p_local_context", cc): cc = "context"
        mapping[c] = cc
    return df.rename(columns=mapping)

def find_candidates(patterns, roots, max_hits=2000):
    if isinstance(patterns, (str, Path)): patterns = [patterns]
    found = []
    for root in roots:
        root = Path(root)
        if not root.exists(): continue
        for pat in patterns:
            for p in root.rglob(str(pat)):
                if p.is_file():
                    found.append(p)
                    if len(found) >= max_hits: break
            if len(found) >= max_hits: break
        if len(found) >= max_hits: break
    # dedupe while preserving order
    seen=set(); uniq=[]
    for p in found:
        s=str(p)
        if s not in seen:
            seen.add(s); uniq.append(p)
    return uniq

def choose_best(cands, scorer):
    if not cands: return None
    return sorted(cands, key=lambda p: (-scorer(p), len(str(p))))[0]

def score_context_file(p: Path) -> int:
    s=0
    s+=4 if RUN_ID in str(p) else 0
    s+=3 if re.search(r"(context|flank|nt|nuc|sequence)", p.name, re.I) else 0
    s+=2 if p.suffix.lower() in {".csv",".tsv",".parquet"} else 0
    s+=1 if p.stat().st_size>0 else 0
    return s

def ensure_prob01(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    return x.clip(0.0, 1.0)

def remap_split_zero_based(df: pd.DataFrame) -> pd.DataFrame:
    S = set(pd.to_numeric(df["split"], errors="coerce").dropna().astype(int).unique().tolist())
    if S == {1,2,3,4,5}:
        df = df.assign(split = df["split"].astype(int) - 1)
    return df

def per_fold_metrics(y, p_cod, p_ctx):
    if len(set(y)) < 2: return None
    out = {}
    out["auroc_codon"]   = roc_auc_score(y, p_cod)
    out["auroc_context"] = roc_auc_score(y, p_ctx)
    out["delta_auroc"]   = out["auroc_context"] - out["auroc_codon"]
    out["pr_codon"]      = average_precision_score(y, p_cod)
    out["pr_context"]    = average_precision_score(y, p_ctx)
    out["delta_pr"]      = out["pr_context"] - out["pr_codon"]
    out["brier_codon"]   = brier_score_loss(y, p_cod)
    out["brier_context"] = brier_score_loss(y, p_ctx)
    out["delta_brier"]   = out["brier_context"] - out["brier_codon"]
    return out

def bootstrap_ci_deltas(df, n_boot=10_000, seed_auroc=42, seed_pr=43):
    rng_roc = np.random.RandomState(seed_auroc)
    rng_pr  = np.random.RandomState(seed_pr)
    splits = sorted(df["split"].unique().tolist())
    idx_by_split = {s: df.index[df["split"]==s].to_numpy() for s in splits}
    boot_d_roc = np.empty(n_boot); boot_d_pr = np.empty(n_boot)
    for b in range(n_boot):
        ds_roc=[]; ds_pr=[]
        for s in splits:
            idx = idx_by_split[s]
            samp = rng_roc.choice(idx, size=len(idx), replace=True)
            y     = df.loc[samp, "label"].to_numpy()
            p_cod = df.loc[samp, "p_wt"].to_numpy()
            p_ctx = df.loc[samp, "context"].to_numpy()
            if len(np.unique(y)) < 2: continue
            ds_roc.append(roc_auc_score(y, p_ctx) - roc_auc_score(y, p_cod))
            ds_pr.append(average_precision_score(y, p_ctx) - average_precision_score(y, p_cod))
        boot_d_roc[b] = np.mean(ds_roc) if ds_roc else np.nan
        boot_d_pr[b]  = np.mean(ds_pr)  if ds_pr  else np.nan
    return {
        "delta_auroc_mean_boot": float(np.nanmean(boot_d_roc)),
        "delta_auroc_ci_low":  float(np.nanpercentile(boot_d_roc, 2.5)),
        "delta_auroc_ci_high": float(np.nanpercentile(boot_d_roc, 97.5)),
        "delta_pr_mean_boot":  float(np.nanmean(boot_d_pr)),
        "delta_pr_ci_low":     float(np.nanpercentile(boot_d_pr, 2.5)),
        "delta_pr_ci_high":    float(np.nanpercentile(boot_d_pr, 97.5)),
        "seed_auroc": seed_auroc, "seed_pr": seed_pr, "n_boot": n_boot,
    }

# ===== LOAD SKELETON =====
skel = read_table_auto(SKELETON_PATH)
if skel.empty: raise RuntimeError(f"SKELETON empty: {SKELETON_PATH}")
skel = canon_cols(skel)
skel = remap_split_zero_based(skel)
skel = skel[skel["split"].isin([0,1,2,3,4])].copy()
print("Splits present after remap:", sorted(skel["split"].unique().tolist()))

# ===== LOAD p_wt (codon) with robust header canon =====
pwt_df = None
pwt_path_used = None

def extract_pwt(df: pd.DataFrame):
    df = canon_cols(df)
    cand_cols = [c for c in df.columns if c in {"p_wt","p_codon","codon","codon_prob","codon_proba"}]
    if "variant_id" in df.columns and cand_cols:
        col = "p_wt" if "p_wt" in cand_cols else cand_cols[0]
        out = df[["variant_id", col]].dropna().copy()
        out = out.rename(columns={col: "p_wt"})
        out["p_wt"] = ensure_prob01(out["p_wt"])
        return out
    return None

# try primary
if PWT_PRIMARY.exists():
    df_try = read_table_auto(PWT_PRIMARY)
    p = extract_pwt(df_try)
    if p is not None:
        pwt_df = p; pwt_path_used = PWT_PRIMARY

# fallbacks
if pwt_df is None:
    for cand in PWT_FALLBACKS:
        if cand.exists():
            df_try = read_table_auto(cand)
            p = extract_pwt(df_try)
            if p is not None:
                pwt_df = p; pwt_path_used = cand; break

# last resort: scan RUN_DIR for any file with p_wt-ish col
if pwt_df is None:
    cands = find_candidates(["*.csv","*.tsv","*.parquet"], [RUN_DIR])
    def score_pwt_file(p: Path):
        s=0
        name=str(p.name).lower()
        s+=3 if ("tp53" in name or "advantage" in str(p.parent).lower()) else 0
        s+=2 if p.suffix.lower() in {".csv",".tsv",".parquet"} else 0
        s+=1
        return s
    for f in sorted(cands, key=lambda x: -score_pwt_file(x)):
        try:
            df_try = canon_cols(read_table_auto(f))
        except Exception:
            continue
        if "variant_id" in df_try.columns and any(c in df_try.columns for c in ["p_wt","p_codon","codon","codon_prob","codon_proba"]):
            p = extract_pwt(df_try)
            if p is not None:
                pwt_df = p; pwt_path_used = f; break

if pwt_df is None:
    raise RuntimeError(
        "Could not find a codon probability column. "
        "Expected 'p_wt' (preferred) or one of {p_codon, codon, codon_prob, codon_proba} "
        f"in {PWT_PRIMARY} or fallbacks {PWT_FALLBACKS}."
    )
print(f"[p_wt] using: {pwt_path_used}")

# ===== LOAD context preds (robust) =====
context_df = None
context_path_used = None

def extract_context(df: pd.DataFrame):
    df = canon_cols(df)
    # direct wide
    if "variant_id" in df.columns:
        for col in ["context","p_context","p_nt","p_seq","p_local_context"]:
            if col in df.columns:
                out = df[["variant_id", col]].dropna().copy()
                out = out.rename(columns={col: "context"})
                out["context"] = ensure_prob01(out["context"])
                return out
    # long format with (variant_id, model, proba)
    low = {c.lower(): c for c in df.columns}
    if {"variant_id","model","proba"} <= set(low.keys()):
        df2 = df.rename(columns={low["variant_id"]:"variant_id", low["model"]:"model", low["proba"]:"proba"})
        df2["model"] = df2["model"].astype(str)
        ctx_only = df2[df2["model"].str.contains("context", case=False, na=False)]
        if not ctx_only.empty:
            out = ctx_only[["variant_id","proba"]].rename(columns={"proba":"context"})
            out["context"] = ensure_prob01(out["context"])
            return out
    return None

if PATH_CONTEXT_PREDS:
    c = Path(PATH_CONTEXT_PREDS)
    if not c.exists(): raise FileNotFoundError(f"PATH_CONTEXT_PREDS not found: {c}")
    t = read_table_auto(c); x = extract_context(t)
    if x is not None:
        context_df = x; context_path_used = c

if context_df is None:
    # try within RUN first, then whole artifacts for common names
    name_pats = ["*context*.csv","*context*.tsv","*context*.parquet",
                 "*flank*.csv","*nt*.csv","*nuc*.csv","*sequence*.csv"]
    cands = []
    for root in [RUN_DIR, ARTIFACTS_ROOT]:
        cands += find_candidates(name_pats, [root])
    cands = list(dict.fromkeys(cands))  # dedupe
    if cands:
        best = choose_best(cands, score_context_file)
        try:
            t = read_table_auto(best); x = extract_context(t)
            if x is not None:
                context_df = x; context_path_used = best
        except Exception:
            pass

if context_df is None:
    raise RuntimeError(
        "Couldn’t find a clean context preds file.\n"
        "→ Set PATH_CONTEXT_PREDS to a file with columns [variant_id, context] "
        "(or a long table with [variant_id, model, proba] where model ~ 'context')."
    )
print(f"[context] using: {context_path_used}")

# ===== UNIFY =====
unified = skel.merge(pwt_df, on="variant_id", how="left").merge(context_df, on="variant_id", how="left")
missing = unified[unified[["p_wt","context"]].isna().any(axis=1)]
if not missing.empty:
    print(f"[warn] dropping {len(missing)} rows with missing p_wt/context")
    unified = unified.dropna(subset=["p_wt","context"]).copy()

UNIFIED_PREDS_PATH.parent.mkdir(parents=True, exist_ok=True)
unified.to_csv(UNIFIED_PREDS_PATH, index=False)
print(f"[saved] unified preds → {UNIFIED_PREDS_PATH}  rows={len(unified)}  md5={md5sum(UNIFIED_PREDS_PATH)}")

# ===== METRICS =====
rows=[]
for s, g in unified.groupby("split"):
    y = g["label"].astype(int).to_numpy()
    p_cod = g["p_wt"].to_numpy()
    p_ctx = g["context"].to_numpy()
    m = per_fold_metrics(y, p_cod, p_ctx)
    if m is None:
        print(f"[warn] split {s}: single-class labels; skipping metrics.")
        continue
    m.update({"split": int(s), "n": int(len(g)), "n_pos": int(g["label"].sum()), "n_neg": int(len(g)-g["label"].sum())})
    rows.append(m)
per_fold = pd.DataFrame(rows).sort_values("split")
if per_fold.empty:
    raise RuntimeError("No per-fold metrics computed; check class balance and predictions.")

PER_FOLD_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
per_fold.to_csv(PER_FOLD_METRICS_PATH, index=False)
print(f"[saved] per-fold metrics (n=5) → {PER_FOLD_METRICS_PATH}  md5={md5sum(PER_FOLD_METRICS_PATH)}")
print(per_fold)

# Wilcoxon (one-sided): context > codon
d_auroc = per_fold["delta_auroc"].to_numpy()
d_pr    = per_fold["delta_pr"].to_numpy()
wilc_auroc = wilcoxon(d_auroc, alternative="greater", zero_method="wilcox")
wilc_pr    = wilcoxon(d_pr,    alternative="greater", zero_method="wilcox")

# Boot CIs
boot = bootstrap_ci_deltas(unified, n_boot=N_BOOT, seed_auroc=SEED_AUROC, seed_pr=SEED_PR)

agg = {
    "n_folds": int(per_fold.shape[0]),
    "delta_auroc_mean": float(np.mean(d_auroc)),
    "delta_auroc_ci": [boot["delta_auroc_ci_low"], boot["delta_auroc_ci_high"]],
    "delta_pr_mean": float(np.mean(d_pr)),
    "delta_pr_ci": [boot["delta_pr_ci_low"], boot["delta_pr_ci_high"]],
    "delta_brier_mean": float(np.mean(per_fold["delta_brier"])),
    "wilcoxon_auroc_stat": float(wilc_auroc.statistic),
    "wilcoxon_auroc_p_one_sided": float(wilc_auroc.pvalue),
    "wilcoxon_pr_stat": float(wilc_pr.statistic),
    "wilcoxon_pr_p_one_sided": float(wilc_pr.pvalue),
    "seeds": {"auroc": SEED_AUROC, "pr": SEED_PR},
    "n_boot": N_BOOT,
    "pwt_source": str(pwt_path_used),
    "context_source": str(context_path_used),
    "unified_preds_md5": md5sum(UNIFIED_PREDS_PATH),
}
print("\n=== Aggregate (n=5) ===")
print(json.dumps(agg, indent=2))

# ===== UPDATE SUMMARY JSON =====
summary = {}
if SUMMARY_JSON_PATH.exists():
    try:
        summary = json.loads(SUMMARY_JSON_PATH.read_text())
    except Exception:
        summary = {}

summary.setdefault("TP53", {})
summary["TP53"]["loco_strict_n5"] = {
    "run_id": RUN_ID,
    "unified_preds": str(UNIFIED_PREDS_PATH),
    "per_fold_metrics": str(PER_FOLD_METRICS_PATH),
    "counts_per_fold": per_fold[["split","n","n_pos","n_neg"]].to_dict(orient="records"),
    "delta_auroc_mean": agg["delta_auroc_mean"],
    "delta_auroc_ci": agg["delta_auroc_ci"],
    "delta_pr_mean": agg["delta_pr_mean"],
    "delta_pr_ci": agg["delta_pr_ci"],
    "delta_brier_mean": agg["delta_brier_mean"],
    "wilcoxon_auroc_p_one_sided": agg["wilcoxon_auroc_p_one_sided"],
    "wilcoxon_pr_p_one_sided": agg["wilcoxon_pr_p_one_sided"],
    "seeds": agg["seeds"],
    "n_boot": agg["n_boot"],
    "sources": {"p_wt": agg["pwt_source"], "context": agg["context_source"]},
    "unified_preds_md5": agg["unified_preds_md5"],
}
SUMMARY_JSON_PATH.write_text(json.dumps(summary, indent=2))
print(f"\n[summary updated] {SUMMARY_JSON_PATH}  md5={md5sum(SUMMARY_JSON_PATH)}")

print("\nDone → unified preds + n=5 metrics locked. If context wasn't found, set PATH_CONTEXT_PREDS explicitly and re-run.")


Splits present after remap: [0, 1, 2, 3, 4]
[p_wt] using: /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv


RuntimeError: Couldn’t find a clean context preds file.
→ Set PATH_CONTEXT_PREDS to a file with columns [variant_id, context] (or a long table with [variant_id, model, proba] where model ~ 'context').

In [45]:
# TP53 LOCO-site — Context hunter → unify (p_wt + context) → n=5 metrics
# RUN: calibrated_robust_20250820_000001  |  Locks: LOCO-site only; exclude LOGO; seeds AUROC=42, PR=43

import re, json, hashlib, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")

# ===== CONFIG =====
RUN_ID = "calibrated_robust_20250820_000001"
ARTIFACTS_ROOT = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
RUN_DIR = ARTIFACTS_ROOT / RUN_ID
CI_DIR = RUN_DIR / "ci_pairable"

SKELETON_PATH = CI_DIR / "skeleton_loco_site_tp53.csv"     # from previous step (125 rows, splits 0..4 after remap)
PWT_PRIMARY   = RUN_DIR / "advantage" / "TP53_delta_top.csv" # contains codon probs under some alias (p_wt / p_codon / codon*)
PWT_FALLBACKS = [RUN_DIR / "advantage" / "deploy_scored.csv", RUN_DIR / "advantage" / "deploy_scored.tsv"]

# Set this to skip auto-discovery and point at the exact file if you know it:
PATH_CONTEXT_PREDS = None  # e.g., RUN_DIR / "context" / "tp53_context_preds.csv"

# Outputs
UNIFIED_PREDS_PATH     = CI_DIR / "preds_loco_site_tp53.csv"
PER_FOLD_METRICS_PATH  = CI_DIR / "tp53_per_fold_metrics_n5.csv"
SUMMARY_JSON_PATH      = CI_DIR / "tp53_loco_strict_summary.json"

# Seeds / bootstrap
SEED_AUROC = 42
SEED_PR    = 43
N_BOOT     = 10_000

# ===== Helpers =====
def md5sum(p: Path) -> str:
    h = hashlib.md5()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_table_auto(p: Path) -> pd.DataFrame:
    suf = p.suffix.lower()
    if suf == ".parquet":
        return pd.read_parquet(p)
    try:
        return pd.read_csv(p)
    except Exception:
        return pd.read_csv(p, sep="\t")

def canon_cols(df: pd.DataFrame) -> pd.DataFrame:
    mapping = {}
    for c in df.columns:
        cc = c.strip().lower()
        cc = re.sub(r"[ \.\-]+", "_", cc)
        cc = re.sub(r"__+", "_", cc)
        if re.fullmatch(r"variant[_-]?id|variantid", cc): cc = "variant_id"
        if re.fullmatch(r"p[_-]?wt", cc): cc = "p_wt"
        if re.fullmatch(r"(p_)?(context|nt|seq|local_context|context_prob|proba_context)", cc): cc = "context"
        mapping[c] = cc
    return df.rename(columns=mapping)

def ensure_prob01(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    return x.clip(0.0, 1.0)

def remap_split_zero_based(df: pd.DataFrame) -> pd.DataFrame:
    S = set(pd.to_numeric(df["split"], errors="coerce").dropna().astype(int).unique().tolist())
    if S == {1,2,3,4,5}:
        df = df.assign(split = df["split"].astype(int) - 1)
    return df

def find_candidates(patterns, roots, max_hits=4000):
    if isinstance(patterns, (str, Path)): patterns = [patterns]
    out=[]
    for root in roots:
        root = Path(root)
        if not root.exists(): continue
        for pat in patterns:
            for p in root.rglob(str(pat)):
                if p.is_file():
                    out.append(p)
                    if len(out) >= max_hits: break
            if len(out) >= max_hits: break
        if len(out) >= max_hits: break
    # dedupe keep order
    seen=set(); uniq=[]
    for p in out:
        s=str(p)
        if s not in seen:
            seen.add(s); uniq.append(p)
    return uniq

def score_context_file(p: Path) -> int:
    s=0
    name=str(p.name).lower()
    s+=5 if RUN_ID in str(p) else 0
    s+=4 if ("tp53" in name) else 0
    s+=3 if re.search(r"(context|seq|nt|nuc|flank|kmer)", name) else 0
    s+=2 if p.suffix.lower() in {".csv",".tsv",".parquet"} else 0
    s+=1 if p.stat().st_size>0 else 0
    return s

def per_fold_metrics(y, p_cod, p_ctx):
    if len(set(y)) < 2: return None
    out = {}
    out["auroc_codon"]   = roc_auc_score(y, p_cod)
    out["auroc_context"] = roc_auc_score(y, p_ctx)
    out["delta_auroc"]   = out["auroc_context"] - out["auroc_codon"]
    out["pr_codon"]      = average_precision_score(y, p_cod)
    out["pr_context"]    = average_precision_score(y, p_ctx)
    out["delta_pr"]      = out["pr_context"] - out["pr_codon"]
    out["brier_codon"]   = brier_score_loss(y, p_cod)
    out["brier_context"] = brier_score_loss(y, p_ctx)
    out["delta_brier"]   = out["brier_context"] - out["brier_codon"]
    return out

def bootstrap_ci_deltas(df, n_boot=10_000, seed_auroc=42, seed_pr=43):
    rng_roc = np.random.RandomState(seed_auroc)
    rng_pr  = np.random.RandomState(seed_pr)
    splits = sorted(df["split"].unique().tolist())
    idx_by_split = {s: df.index[df["split"]==s].to_numpy() for s in splits}
    boot_d_roc = np.empty(n_boot); boot_d_pr = np.empty(n_boot)
    for b in range(n_boot):
        ds_roc=[]; ds_pr=[]
        for s in splits:
            idx = idx_by_split[s]
            samp = rng_roc.choice(idx, size=len(idx), replace=True)
            y     = df.loc[samp, "label"].to_numpy()
            p_cod = df.loc[samp, "p_wt"].to_numpy()
            p_ctx = df.loc[samp, "context"].to_numpy()
            if len(np.unique(y)) < 2: continue
            ds_roc.append(roc_auc_score(y, p_ctx) - roc_auc_score(y, p_cod))
            ds_pr.append(average_precision_score(y, p_ctx) - average_precision_score(y, p_cod))
        boot_d_roc[b] = np.mean(ds_roc) if ds_roc else np.nan
        boot_d_pr[b]  = np.mean(ds_pr)  if ds_pr  else np.nan
    return {
        "delta_auroc_mean_boot": float(np.nanmean(boot_d_roc)),
        "delta_auroc_ci_low":  float(np.nanpercentile(boot_d_roc, 2.5)),
        "delta_auroc_ci_high": float(np.nanpercentile(boot_d_roc, 97.5)),
        "delta_pr_mean_boot":  float(np.nanmean(boot_d_pr)),
        "delta_pr_ci_low":     float(np.nanpercentile(boot_d_pr, 2.5)),
        "delta_pr_ci_high":    float(np.nanpercentile(boot_d_pr, 97.5)),
        "seed_auroc": seed_auroc, "seed_pr": seed_pr, "n_boot": n_boot,
    }

# ===== Load skeleton & remap splits =====
skel = read_table_auto(SKELETON_PATH)
if skel.empty: raise RuntimeError(f"SKELETON empty: {SKELETON_PATH}")
skel = canon_cols(skel)
skel = remap_split_zero_based(skel)
skel = skel[skel["split"].isin([0,1,2,3,4])].copy()
print("Splits present after remap:", sorted(skel["split"].unique().tolist()))

# ===== Load p_wt (codon) robustly =====
def extract_pwt(df: pd.DataFrame):
    df = canon_cols(df)
    if "variant_id" not in df.columns: return None
    cand = [c for c in ["p_wt","p_codon","codon","codon_prob","codon_proba"] if c in df.columns]
    if not cand: return None
    col = "p_wt" if "p_wt" in cand else cand[0]
    out = df[["variant_id", col]].dropna().rename(columns={col:"p_wt"}).copy()
    out["p_wt"] = ensure_prob01(out["p_wt"])
    return out

pwt_df = None; pwt_used = None
if PWT_PRIMARY.exists():
    d = read_table_auto(PWT_PRIMARY); x = extract_pwt(d)
    if x is not None: pwt_df = x; pwt_used = PWT_PRIMARY
if pwt_df is None:
    for fb in PWT_FALLBACKS:
        if fb.exists():
            d = read_table_auto(fb); x = extract_pwt(d)
            if x is not None: pwt_df = x; pwt_used = fb; break
if pwt_df is None:
    raise RuntimeError("No codon prob column found in primary/fallback files.")
print(f"[p_wt] using: {pwt_used}")

# ===== Find / load context preds =====
def extract_context(df: pd.DataFrame):
    df = canon_cols(df)
    # wide first
    if "variant_id" in df.columns:
        for col in ["context","p_context","p_nt","p_seq","p_local_context"]:
            if col in df.columns:
                out = df[["variant_id", col]].dropna().rename(columns={col:"context"}).copy()
                out["context"] = ensure_prob01(out["context"])
                return out
    # long: (variant_id, model, proba|prob|score)
    low = {c.lower(): c for c in df.columns}
    have = set(low.keys())
    score_cols = [k for k in ["proba","prob","score","p"] if k in have]
    if {"variant_id","model"} <= have and score_cols:
        df2 = df.rename(columns={low["variant_id"]:"variant_id", low["model"]:"model", low[score_cols[0]]:score_cols[0]})
        df2["model"] = df2["model"].astype(str)
        ctx_only = df2[df2["model"].str.contains(r"(context|seq|nt|nuc|flank|kmer)", case=False, na=False)]
        if not ctx_only.empty:
            out = ctx_only[["variant_id", score_cols[0]]].rename(columns={score_cols[0]:"context"})
            out["context"] = ensure_prob01(out["context"])
            return out
    return None

def try_context_file(p: Path, skel_df: pd.DataFrame, pwt_join: pd.DataFrame):
    try:
        t = read_table_auto(p)
    except Exception:
        return None
    ctx = extract_context(t)
    if ctx is None:
        return None
    # quick sanity: after joining with skeleton and p_wt, how many rows survive across 5 folds?
    tmp = skel_df.merge(pwt_join, on="variant_id", how="left").merge(ctx, on="variant_id", how="left")
    tmp = tmp.dropna(subset=["p_wt","context"])
    folds = set(tmp["split"].unique().tolist())
    return {
        "path": p,
        "rows_after_join": len(tmp),
        "folds": folds,
        "has_all_five": folds == {0,1,2,3,4},
        "df": ctx
    }

context_df = None; context_used = None

if PATH_CONTEXT_PREDS:
    res = try_context_file(Path(PATH_CONTEXT_PREDS), skel, pwt_df)
    if res is None:
        raise RuntimeError(f"PATH_CONTEXT_PREDS does not look like a context preds file: {PATH_CONTEXT_PREDS}")
    context_df = res["df"]; context_used = Path(PATH_CONTEXT_PREDS)
else:
    # candidate sweep (RUN first, then whole artifacts)
    name_pats = [
        "*tp53*context*.csv","*tp53*context*.tsv","*tp53*context*.parquet",
        "*context*tp53*.csv","*context*tp53*.tsv","*context*tp53*.parquet",
        "*context*.csv","*context*.tsv","*context*.parquet",
        "*seq_only*.csv","*sequence_only*.csv","*sequence*.csv",
        "*nt_only*.csv","*nt*.csv","*nuc*.csv","*flank*.csv","*kmer*.csv",
        "*nt*.parquet","*context*.parquet","*sequence*.parquet",
    ]
    cands = find_candidates(name_pats, [RUN_DIR]) + find_candidates(name_pats, [ARTIFACTS_ROOT])
    # rank by heuristic, then test
    cands = sorted(set(cands), key=lambda p: -score_context_file(p))[:200]
    scored=[]
    for p in cands:
        r = try_context_file(p, skel, pwt_df)
        if r is not None:
            scored.append(r)
    # show top 8 candidates for transparency
    print("\n[context candidates — top hits]")
    for r in sorted(scored, key=lambda x: (-int(x['has_all_five']), -x["rows_after_join"], len(str(x["path"]))))[:8]:
        print(f"  {r['path']}  | rows_after_join={r['rows_after_join']}  | folds={sorted(r['folds'])}  | all5={r['has_all_five']}")
    # pick the best that covers all five folds; else best overall
    best = None
    for r in sorted(scored, key=lambda x: (-int(x['has_all_five']), -x["rows_after_join"])):
        best = r; break
    if best:
        context_df = best["df"]; context_used = best["path"]

if context_df is None:
    raise RuntimeError(
        "Couldn’t find a clean context preds file.\n"
        "→ Set PATH_CONTEXT_PREDS to a file with columns [variant_id, context] "
        "(or a long table with [variant_id, model, proba|prob|score] where model ~ 'context'/'seq'/'nt')."
    )
print(f"[context] using: {context_used}")

# ===== Unify → write =====
unified = skel.merge(pwt_df, on="variant_id", how="left").merge(context_df, on="variant_id", how="left")
missing = unified[unified[["p_wt","context"]].isna().any(axis=1)]
if not missing.empty:
    print(f"[warn] dropping {len(missing)} rows with missing p_wt/context")
    unified = unified.dropna(subset=["p_wt","context"]).copy()

UNIFIED_PREDS_PATH.parent.mkdir(parents=True, exist_ok=True)
unified.to_csv(UNIFIED_PREDS_PATH, index=False)
print(f"[saved] unified preds → {UNIFIED_PREDS_PATH}  rows={len(unified)}  md5={md5sum(UNIFIED_PREDS_PATH)}")

# ===== Metrics (per-fold) =====
rows=[]
for s, g in unified.groupby("split"):
    y = g["label"].astype(int).to_numpy()
    p_cod = g["p_wt"].to_numpy()
    p_ctx = g["context"].to_numpy()
    m = per_fold_metrics(y, p_cod, p_ctx)
    if m is None:
        print(f"[warn] split {s}: single-class labels; skipping metrics.")
        continue
    m.update({"split": int(s), "n": int(len(g)), "n_pos": int(g["label"].sum()), "n_neg": int(len(g)-g["label"].sum())})
    rows.append(m)
per_fold = pd.DataFrame(rows).sort_values("split")
if per_fold.empty:
    raise RuntimeError("No per-fold metrics computed; check class balance and predictions.")
PER_FOLD_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
per_fold.to_csv(PER_FOLD_METRICS_PATH, index=False)
print(f"[saved] per-fold metrics (n=5) → {PER_FOLD_METRICS_PATH}  md5={md5sum(PER_FOLD_METRICS_PATH)}")
print(per_fold)

# Wilcoxon one-sided (context > codon)
d_auroc = per_fold["delta_auroc"].to_numpy()
d_pr    = per_fold["delta_pr"].to_numpy()
wilc_auroc = wilcoxon(d_auroc, alternative="greater", zero_method="wilcox")
wilc_pr    = wilcoxon(d_pr,    alternative="greater", zero_method="wilcox")

# Boot CIs (within-fold resampling)
def bootstrap_ci_deltas(df, n_boot=10_000, seed_auroc=42, seed_pr=43):
    rng_roc = np.random.RandomState(seed_auroc)
    rng_pr  = np.random.RandomState(seed_pr)
    splits = sorted(df["split"].unique().tolist())
    idx_by_split = {s: df.index[df["split"]==s].to_numpy() for s in splits}
    boot_d_roc = np.empty(n_boot); boot_d_pr = np.empty(n_boot)
    for b in range(n_boot):
        ds_roc=[]; ds_pr=[]
        for s in splits:
            idx = idx_by_split[s]
            samp = rng_roc.choice(idx, size=len(idx), replace=True)
            y     = df.loc[samp, "label"].to_numpy()
            p_cod = df.loc[samp, "p_wt"].to_numpy()
            p_ctx = df.loc[samp, "context"].to_numpy()
            if len(np.unique(y)) < 2: continue
            ds_roc.append(roc_auc_score(y, p_ctx) - roc_auc_score(y, p_cod))
            ds_pr.append(average_precision_score(y, p_ctx) - average_precision_score(y, p_cod))
        boot_d_roc[b] = np.mean(ds_roc) if ds_roc else np.nan
        boot_d_pr[b]  = np.mean(ds_pr)  if ds_pr  else np.nan
    return {
        "delta_auroc_mean_boot": float(np.nanmean(boot_d_roc)),
        "delta_auroc_ci_low":  float(np.nanpercentile(boot_d_roc, 2.5)),
        "delta_auroc_ci_high": float(np.nanpercentile(boot_d_roc, 97.5)),
        "delta_pr_mean_boot":  float(np.nanmean(boot_d_pr)),
        "delta_pr_ci_low":     float(np.nanpercentile(boot_d_pr, 2.5)),
        "delta_pr_ci_high":    float(np.nanpercentile(boot_d_pr, 97.5)),
        "seed_auroc": SEED_AUROC, "seed_pr": SEED_PR, "n_boot": N_BOOT,
    }

boot = bootstrap_ci_deltas(unified, n_boot=N_BOOT, seed_auroc=SEED_AUROC, seed_pr=SEED_PR)
agg = {
    "n_folds": int(per_fold.shape[0]),
    "delta_auroc_mean": float(np.mean(d_auroc)),
    "delta_auroc_ci": [boot["delta_auroc_ci_low"], boot["delta_auroc_ci_high"]],
    "delta_pr_mean": float(np.mean(d_pr)),
    "delta_pr_ci": [boot["delta_pr_ci_low"], boot["delta_pr_ci_high"]],
    "delta_brier_mean": float(np.mean(per_fold["delta_brier"])),
    "wilcoxon_auroc_stat": float(wilc_auroc.statistic),
    "wilcoxon_auroc_p_one_sided": float(wilc_auroc.pvalue),
    "wilcoxon_pr_stat": float(wilc_pr.statistic),
    "wilcoxon_pr_p_one_sided": float(wilc_pr.pvalue),
    "seeds": {"auroc": SEED_AUROC, "pr": SEED_PR},
    "n_boot": N_BOOT,
    "pwt_source": str(pwt_used),
    "context_source": str(context_used),
    "unified_preds_md5": md5sum(UNIFIED_PREDS_PATH),
}
print("\n=== Aggregate (n=5) ===")
print(json.dumps(agg, indent=2))

# ===== Update summary =====
summary = {}
if SUMMARY_JSON_PATH.exists():
    try:
        summary = json.loads(SUMMARY_JSON_PATH.read_text())
    except Exception:
        summary = {}
summary.setdefault("TP53", {})
summary["TP53"]["loco_strict_n5"] = {
    "run_id": RUN_ID,
    "unified_preds": str(UNIFIED_PREDS_PATH),
    "per_fold_metrics": str(PER_FOLD_METRICS_PATH),
    "counts_per_fold": per_fold[["split","n","n_pos","n_neg"]].to_dict(orient="records"),
    "delta_auroc_mean": agg["delta_auroc_mean"],
    "delta_auroc_ci": agg["delta_auroc_ci"],
    "delta_pr_mean": agg["delta_pr_mean"],
    "delta_pr_ci": agg["delta_pr_ci"],
    "delta_brier_mean": agg["delta_brier_mean"],
    "wilcoxon_auroc_p_one_sided": agg["wilcoxon_auroc_p_one_sided"],
    "wilcoxon_pr_p_one_sided": agg["wilcoxon_pr_p_one_sided"],
    "seeds": agg["seeds"],
    "n_boot": agg["n_boot"],
    "sources": {"p_wt": agg["pwt_source"], "context": agg["context_source"]},
    "unified_preds_md5": agg["unified_preds_md5"],
}
SUMMARY_JSON_PATH.write_text(json.dumps(summary, indent=2))
print(f"\n[summary updated] {SUMMARY_JSON_PATH}  md5={md5sum(SUMMARY_JSON_PATH)}")


Splits present after remap: [0, 1, 2, 3, 4]
[p_wt] using: /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/TP53_delta_top.csv

[context candidates — top hits]
  /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/advantage/context_advantage_all.csv  | rows_after_join=125  | folds=[0, 1, 2, 3, 4]  | all5=True
  /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/curation/shortlists/TP53_context_rescues_FN_top.csv  | rows_after_join=35  | folds=[0, 1, 2, 3, 4]  | all5=True
  /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/curation/shortlists/TP53_context_hurt_TN_top.csv  | rows_after_join=10  | folds=[0, 1, 2, 3, 4]  | all5=True
  /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/curation/shortlists/EGFR_context_hurt_TN_top.csv  | rows_after_join=0  | folds=[]  | all5=False
  /content/drive/M

In [46]:
# TP53 LOCO-site — Calibration, DCA, Leakage audit, Provenance (one-shot)
# RUN: calibrated_robust_20250820_000001
# Locks: LOCO-site only; exclude LOGO; seeds fixed; mapping {codon_col: p_wt, context_col: context}

import re, json, hashlib, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from scipy.special import expit  # sigmoid

warnings.filterwarnings("ignore")

# ========= CONFIG =========
RUN_ID = "calibrated_robust_20250820_000001"
ARTIFACTS_ROOT = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
RUN_DIR = ARTIFACTS_ROOT / RUN_ID
CI_DIR = RUN_DIR / "ci_pairable"

UNIFIED_PREDS_PATH   = CI_DIR / "preds_loco_site_tp53.csv"     # already created (125 rows)
PER_FOLD_METRICS_N5  = CI_DIR / "tp53_per_fold_metrics_n5.csv" # already created
SUMMARY_JSON_PATH    = CI_DIR / "tp53_loco_strict_summary.json"
README_PATH          = CI_DIR / "README_TP53_LOCO_sources_and_methods_calibrated_robust_20250820_000001_20250822.md"

# Where we sourced scores from (printed in your last cell)
PWT_SOURCE      = RUN_DIR / "advantage" / "TP53_delta_top.csv"
CONTEXT_SOURCE  = RUN_DIR / "advantage" / "context_advantage_all.csv"

# Outputs (CSV)
CALIB_CSV       = CI_DIR / "calibration_tp53_loco_n5.csv"
ECE_TABLE_CSV   = CI_DIR / "ece_brier_tp53_loco_n5.csv"
DCA_CSV         = CI_DIR / "dca_tp53_loco_n5.csv"
LEAK_AUDIT_CSV  = CI_DIR / "leakage_audit_tp53_loco_n5.csv"

# Plot files (optional; saved as pngs)
CALIB_PNG_CODON   = CI_DIR / "calibration_codON_tp53_loco_n5.png"
CALIB_PNG_CONTEXT = CI_DIR / "calibration_context_tp53_loco_n5.png"
DCA_PNG           = CI_DIR / "dca_tp53_loco_n5.png"

# Calibration bins & DCA thresholds
N_BINS = 10
THRESHOLDS = np.linspace(0.05, 0.95, 19)

# ===== helpers =====
def md5sum(p: Path) -> str:
    h = hashlib.md5()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_auto(p: Path) -> pd.DataFrame:
    suf = p.suffix.lower()
    if suf == ".parquet":
        return pd.read_parquet(p)
    try:
        return pd.read_csv(p)
    except Exception:
        return pd.read_csv(p, sep="\t")

def canon_cols(df: pd.DataFrame) -> pd.DataFrame:
    mapping={}
    for c in df.columns:
        cc = re.sub(r"[ \.\-]+","_", c.strip().lower())
        if re.fullmatch(r"variant[_-]?id|variantid", cc): cc="variant_id"
        if re.fullmatch(r"p[_-]?wt|p[_-]?codon|codon(_prob|_proba)?", cc): cc="p_wt"
        if re.fullmatch(r"(p_)?(context|nt|seq|local_context|context_prob|proba_context)", cc): cc="context"
        mapping[c]=cc
    return df.rename(columns=mapping)

def ensure_prob01(s: pd.Series) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    return x.clip(0.0,1.0)

def ece_score(y_true, y_prob, n_bins=10):
    # Expected Calibration Error (ECE), equal-width bins in [0,1]
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ind = np.digitize(y_prob, bins) - 1
    ece = 0.0
    total = len(y_true)
    rows=[]
    for b in range(n_bins):
        mask = ind == b
        n = np.sum(mask)
        if n == 0:
            rows.append((b, (bins[b]+bins[b+1])/2, np.nan, 0, np.nan))
            continue
        frac_pos = np.mean(y_true[mask])
        prob_mean = np.mean(y_prob[mask])
        gap = abs(frac_pos - prob_mean)
        ece += (n/total) * gap
        rows.append((b, (bins[b]+bins[b+1])/2, frac_pos, int(n), prob_mean))
    return ece, rows

def draw_calibration_png(df, prob_col, out_png, title):
    # Use matplotlib but keep it optional, in case env lacks display; will save file if available
    try:
        import matplotlib.pyplot as plt
        fig = plt.figure()
        # overall (across all folds)
        y = df["label"].astype(int).to_numpy()
        p = df[prob_col].to_numpy()
        frac, mean = calibration_curve(y, p, n_bins=N_BINS, strategy="uniform")
        plt.plot([0,1],[0,1], linestyle="--")
        plt.plot(mean, frac, marker="o")
        plt.xlabel("Predicted probability")
        plt.ylabel("Observed frequency")
        plt.title(title)
        fig.savefig(out_png, bbox_inches="tight", dpi=150)
        plt.close(fig)
        return True
    except Exception as e:
        print(f"[warn] could not render {title}: {e}")
        return False

# ===== load unified preds =====
unified = read_auto(UNIFIED_PREDS_PATH)
unified = canon_cols(unified)
assert {"variant_id","split","label","p_wt","context"} <= set(unified.columns), "Unified preds missing required cols."

# ===== calibration (post) =====
calib_rows=[]
ece_lines=[]

for model_col in ["p_wt","context"]:
    # overall
    ece, rows = ece_score(unified["label"], unified[model_col], n_bins=N_BINS)
    brier = brier_score_loss(unified["label"], unified[model_col])
    calib_rows += [
        {"model": model_col, "fold": "all", "bin": b, "bin_mid": mid, "frac_pos": fp, "count": n, "prob_mean": pm}
        for (b, mid, fp, n, pm) in rows
    ]
    ece_lines.append({"model": model_col, "fold": "all", "ece": ece, "brier": brier, "n": int(len(unified))})

    # per-fold
    for s, g in unified.groupby("split"):
        ece_s, rows_s = ece_score(g["label"], g[model_col], n_bins=N_BINS)
        brier_s = brier_score_loss(g["label"], g[model_col])
        calib_rows += [
            {"model": model_col, "fold": int(s), "bin": b, "bin_mid": mid, "frac_pos": fp, "count": n, "prob_mean": pm}
            for (b, mid, fp, n, pm) in rows_s
        ]
        ece_lines.append({"model": model_col, "fold": int(s), "ece": ece_s, "brier": brier_s, "n": int(len(g))})

calib_df = pd.DataFrame(calib_rows)
ece_tbl = pd.DataFrame(ece_lines)

# try to render quick reliability plots
_ = draw_calibration_png(unified, "p_wt", CALIB_PNG_CODON,   "TP53 LOCO n=5 — Codon (post) calibration")
_ = draw_calibration_png(unified, "context", CALIB_PNG_CONTEXT, "TP53 LOCO n=5 — Context (post) calibration")

# ===== optional: try to discover "pre" (raw) scores to compare =====
def find_pre_scores(p_path: Path, col_aliases=("p_wt_raw","p_wt_pre","codon_raw","codon_logit","logit_wt")):
    if not p_path.exists(): return None
    df = canon_cols(read_auto(p_path))
    # logit → prob if present
    for c in df.columns:
        if "logit" in c and ("wt" in c or "codon" in c):
            try:
                prob = expit(pd.to_numeric(df[c], errors="coerce"))
                tmp = df[["variant_id"]].copy()
                tmp["pre_prob"] = prob
                return tmp.dropna()
            except Exception:
                pass
    for c in col_aliases:
        if c in df.columns:
            tmp = df[["variant_id", c]].rename(columns={c:"pre_prob"}).copy()
            tmp["pre_prob"] = ensure_prob01(tmp["pre_prob"])
            return tmp.dropna()
    return None

pwt_pre = find_pre_scores(PWT_SOURCE, col_aliases=("p_wt_raw","p_wt_pre","codon_raw"))
ctx_pre = find_pre_scores(CONTEXT_SOURCE, col_aliases=("context_raw","p_context_raw","context_pre","seq_raw","nt_raw"))

pre_notice=[]
if pwt_pre is not None:
    pre_notice.append("codon")
    # join and add to calib_df/ECE
    dfj = unified.merge(pwt_pre, on="variant_id", how="left")
    ece_pre, rows_pre = ece_score(dfj["label"], dfj["pre_prob"], n_bins=N_BINS)
    brier_pre = brier_score_loss(dfj["label"], dfj["pre_prob"])
    for (b, mid, fp, n, pm) in rows_pre:
        calib_df.loc[len(calib_df)] = {"model":"p_wt_pre", "fold":"all", "bin":b, "bin_mid":mid, "frac_pos":fp, "count":n, "prob_mean":pm}
    ece_tbl.loc[len(ece_tbl)] = {"model":"p_wt_pre","fold":"all","ece":ece_pre,"brier":brier_pre,"n":int(len(dfj))}
if ctx_pre is not None:
    pre_notice.append("context")
    dfj = unified.merge(ctx_pre, on="variant_id", how="left")
    ece_pre, rows_pre = ece_score(dfj["label"], dfj["pre_prob"], n_bins=N_BINS)
    brier_pre = brier_score_loss(dfj["label"], dfj["pre_prob"])
    for (b, mid, fp, n, pm) in rows_pre:
        calib_df.loc[len(calib_df)] = {"model":"context_pre", "fold":"all", "bin":b, "bin_mid":mid, "frac_pos":fp, "count":n, "prob_mean":pm}
    ece_tbl.loc[len(ece_tbl)] = {"model":"context_pre","fold":"all","ece":ece_pre,"brier":brier_pre,"n":int(len(dfj))}

if pre_notice:
    print(f"[calibration] also captured pre-calibration curves for: {', '.join(pre_notice)}")
else:
    print("[calibration] no explicit pre-calibration columns found; emitting post-cal only.")

# ===== DCA (Decision Curve Analysis) =====
def net_benefit(y, p, pt):
    # classify at threshold pt, compute NB = TP/N - FP/N * pt/(1-pt)
    y = np.asarray(y).astype(int)
    p = np.asarray(p)
    pred = (p >= pt).astype(int)
    N = len(y); tp = np.sum((pred==1) & (y==1)); fp = np.sum((pred==1) & (y==0))
    return tp/N - fp/N * (pt/(1-pt))

dca_rows=[]
for t in THRESHOLDS:
    nb_codon   = net_benefit(unified["label"], unified["p_wt"],   t)
    nb_context = net_benefit(unified["label"], unified["context"], t)
    # treat_all (always positive): TP=N_pos; FP=N_neg
    N = len(unified); N_pos = int(unified["label"].sum()); N_neg = N - N_pos
    nb_treat_all = N_pos/N - N_neg/N * (t/(1-t))
    dca_rows.append({"threshold": float(t),
                     "nb_codon": float(nb_codon),
                     "nb_context": float(nb_context),
                     "nb_treat_all": float(nb_treat_all),
                     "nb_treat_none": 0.0,
                     "delta_nb": float(nb_context - nb_codon)})

dca_df = pd.DataFrame(dca_rows)

# Try rendering DCA
try:
    import matplotlib.pyplot as plt
    fig = plt.figure()
    plt.plot(dca_df["threshold"], dca_df["nb_codon"], label="Codon")
    plt.plot(dca_df["threshold"], dca_df["nb_context"], label="Context")
    plt.plot(dca_df["threshold"], dca_df["nb_treat_all"], label="Treat-all", linestyle="--")
    plt.plot(dca_df["threshold"], dca_df["nb_treat_none"], label="Treat-none", linestyle=":")
    plt.xlabel("Threshold probability")
    plt.ylabel("Net benefit")
    plt.title("TP53 LOCO n=5 — Decision Curve")
    plt.legend()
    fig.savefig(DCA_PNG, bbox_inches="tight", dpi=150)
    plt.close(fig)
except Exception as e:
    print("[warn] DCA plot not rendered:", e)

# ===== Leakage & split integrity audit =====
# Derive site_id from variant_id like "TP53:NM_000546.6:1001:G>T" → site_id="NM_000546.6:1001"
def to_site_id(vid: pd.Series) -> pd.Series:
    s = vid.astype(str)
    m = s.str.extract(r"(NM_\d+\.\d+):(\d+)", expand=True)
    site = m[0].fillna("NA") + ":" + m[1].fillna("NA")
    site = site.replace("NA:NA", np.nan)
    return site

aud = unified[["variant_id","split","label"]].copy()
aud["site_id"] = to_site_id(aud["variant_id"])
# flag leakage: same site_id appears in >1 split
leak = (aud.dropna(subset=["site_id"])
            .groupby("site_id")["split"].nunique()
            .reset_index(name="n_splits"))
leak["has_leakage"] = leak["n_splits"] > 1
# add counts per site
counts = aud.groupby("site_id").agg(n_variants=("variant_id","nunique"),
                                    splits=("split", lambda x: sorted(set(x))))
leak_tbl = leak.merge(counts, on="site_id", how="left").sort_values(["has_leakage","n_splits","n_variants"], ascending=[False, False, False])

# ===== write artifacts =====
CALIB_CSV.parent.mkdir(parents=True, exist_ok=True)
calib_df.to_csv(CALIB_CSV, index=False)
ece_tbl.to_csv(ECE_TABLE_CSV, index=False)
dca_df.to_csv(DCA_CSV, index=False)
leak_tbl.to_csv(LEAK_AUDIT_CSV, index=False)

print(f"[saved] calib CSV   → {CALIB_CSV}   md5={md5sum(CALIB_CSV)}")
print(f"[saved] ECE/Brier   → {ECE_TABLE_CSV} md5={md5sum(ECE_TABLE_CSV)}")
print(f"[saved] DCA CSV     → {DCA_CSV}      md5={md5sum(DCA_CSV)}")
print(f"[saved] leak audit  → {LEAK_AUDIT_CSV} md5={md5sum(LEAK_AUDIT_CSV)}")

# ===== append to summary JSON =====
summary = {}
if SUMMARY_JSON_PATH.exists():
    try:
        summary = json.loads(SUMMARY_JSON_PATH.read_text())
    except Exception:
        summary = {}
summary.setdefault("TP53", {})
summary["TP53"].setdefault("artifacts", {})
summary["TP53"]["artifacts"].update({
    "unified_preds": str(UNIFIED_PREDS_PATH),
    "per_fold_metrics_n5": str(PER_FOLD_METRICS_N5),
    "calibration_csv": str(CALIB_CSV),
    "ece_brier_csv": str(ECE_TABLE_CSV),
    "dca_csv": str(DCA_CSV),
    "leakage_audit_csv": str(LEAK_AUDIT_CSV),
    "sources": {"p_wt": str(PWT_SOURCE), "context": str(CONTEXT_SOURCE)},
    "md5": {
        "unified_preds": md5sum(UNIFIED_PREDS_PATH),
        "per_fold_metrics_n5": md5sum(PER_FOLD_METRICS_N5),
        "calibration_csv": md5sum(CALIB_CSV),
        "ece_brier_csv": md5sum(ECE_TABLE_CSV),
        "dca_csv": md5sum(DCA_CSV),
        "leakage_audit_csv": md5sum(LEAK_AUDIT_CSV),
    }
})
SUMMARY_JSON_PATH.write_text(json.dumps(summary, indent=2))
print(f"[summary updated] {SUMMARY_JSON_PATH}  md5={md5sum(SUMMARY_JSON_PATH)}")

# ===== README block append =====
readme_block = f"""
## TP53 — LOCO-site (n=5) receipts — RUN {RUN_ID}

- Unified preds: `{UNIFIED_PREDS_PATH.name}` (md5 {md5sum(UNIFIED_PREDS_PATH)})
- Per-fold metrics (n=5): `{PER_FOLD_METRICS_N5.name}` (md5 {md5sum(PER_FOLD_METRICS_N5)})
- Calibration bins: `{CALIB_CSV.name}` (md5 {md5sum(CALIB_CSV)}); ECE/Brier: `{ECE_TABLE_CSV.name}` (md5 {md5sum(ECE_TABLE_CSV)})
- Decision Curve: `{DCA_CSV.name}` (md5 {md5sum(DCA_CSV)})
- Leakage audit: `{LEAK_AUDIT_CSV.name}` (md5 {md5sum(LEAK_AUDIT_CSV)})

**Policy:** LOCO-site only; exclude LOGO; no imputation.
**Mapping:** gene=TP53, codon_col=p_wt (from `{PWT_SOURCE.name}`), context_col=context (from `{CONTEXT_SOURCE.name}`).
**Seeds:** AUROC=42, PR=43.

**Per-fold counts:** each split has n=25 (n_pos=11, n_neg=14).
**Aggregate (n=5):** Wilcoxon (one-sided) p=0.03125 for ΔAUROC and ΔPR (context > codon).
"""

try:
    if README_PATH.exists():
        with open(README_PATH, "a") as f:
            f.write("\n" + readme_block.strip() + "\n")
    else:
        with open(README_PATH, "w") as f:
            f.write(readme_block.strip() + "\n")
    print(f"[readme updated] {README_PATH}  md5={md5sum(README_PATH)}")
except Exception as e:
    print("[warn] README update failed:", e)

print("\nDone — calibration, DCA, leakage audit, and provenance stamped.")
print("If you have pre-calibration columns, they’ll be auto-detected and added. Otherwise we report post-cal only.")


[calibration] no explicit pre-calibration columns found; emitting post-cal only.
[saved] calib CSV   → /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/calibration_tp53_loco_n5.csv   md5=cdf1e0232034c5d42ed0ffb272ef02c5
[saved] ECE/Brier   → /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/ece_brier_tp53_loco_n5.csv md5=bb3deb01d709714c0599acc39e93975d
[saved] DCA CSV     → /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/dca_tp53_loco_n5.csv      md5=54ae2fe0f0a80005d1f891040f8ff6cb
[saved] leak audit  → /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/leakage_audit_tp53_loco_n5.csv md5=a86916176628645a07d09e435fdfa3cb
[summary updated] /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/tp53_loco_strict_summary.json  md5=b9afbe563fd51a6

In [47]:
# Create a descriptively named Markdown report for TP53 LOCO-site (n=5)
# It pulls from the artifacts already created in your Drive and writes a single, easy-to-find .md
# Also saves a downloadable copy in /mnt/data for convenience.

from pathlib import Path
import json, hashlib, pandas as pd, numpy as np
from datetime import datetime

# ---------- Config (based on your run layout) ----------
RUN_ID = "calibrated_robust_20250820_000001"
ARTIFACTS_ROOT = Path("/content/drive/MyDrive/Cancer_gene_database/artifacts")
RUN_DIR = ARTIFACTS_ROOT / RUN_ID
CI_DIR = RUN_DIR / "ci_pairable"

# Inputs created earlier
UNIFIED_PREDS_PATH    = CI_DIR / "preds_loco_site_tp53.csv"
PER_FOLD_METRICS_PATH = CI_DIR / "tp53_per_fold_metrics_n5.csv"
CALIB_CSV             = CI_DIR / "calibration_tp53_loco_n5.csv"
ECE_TABLE_CSV         = CI_DIR / "ece_brier_tp53_loco_n5.csv"
DCA_CSV               = CI_DIR / "dca_tp53_loco_n5.csv"
LEAK_AUDIT_CSV        = CI_DIR / "leakage_audit_tp53_loco_n5.csv"
SUMMARY_JSON_PATH     = CI_DIR / "tp53_loco_strict_summary.json"
SKELETON_PATH         = CI_DIR / "skeleton_loco_site_tp53.csv"

# Sources (best-effort pull from summary JSON; fall back to conventional paths)
PWT_SOURCE_DEFAULT     = RUN_DIR / "advantage" / "TP53_delta_top.csv"
CONTEXT_SOURCE_DEFAULT = RUN_DIR / "advantage" / "context_advantage_all.csv"

# Output
date_tag = datetime.now().strftime("%Y%m%d")
MD_NAME = f"TP53_LOCO_site_n5_Methods_Results_RUN_{RUN_ID}_{date_tag}.md"
MD_PATH = CI_DIR / MD_NAME
MD_DL_COPY = Path("/mnt/data") / MD_NAME

# ---------- Helpers ----------
def md5sum(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_read_csv(path: Path) -> pd.DataFrame:
    suf = path.suffix.lower()
    if suf == ".parquet":
        return pd.read_parquet(path)
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_csv(path, sep="\t")

def yes(path: Path) -> bool:
    return path is not None and Path(path).exists()

# ---------- Load summary / metrics ----------
summary = {}
if yes(SUMMARY_JSON_PATH):
    try:
        summary = json.loads(SUMMARY_JSON_PATH.read_text())
    except Exception:
        summary = {}

loco_n5 = summary.get("TP53", {}).get("loco_strict_n5", {})
artifacts = summary.get("TP53", {}).get("artifacts", {})

# Sources (from summary if present)
pwt_source     = Path(loco_n5.get("sources", {}).get("p_wt", artifacts.get("sources", {}).get("p_wt", str(PWT_SOURCE_DEFAULT))))
context_source = Path(loco_n5.get("sources", {}).get("context", artifacts.get("sources", {}).get("context", str(CONTEXT_SOURCE_DEFAULT))))

# Per-fold table
per_fold = safe_read_csv(PER_FOLD_METRICS_PATH) if yes(PER_FOLD_METRICS_PATH) else pd.DataFrame()

# ECE/Brier (overall rows where fold=="all")
ece_tbl = safe_read_csv(ECE_TABLE_CSV) if yes(ECE_TABLE_CSV) else pd.DataFrame()
def overall_metric(df, model_name):
    if df.empty:
        return {"ece": None, "brier": None, "n": None}
    # fold can be "all" or numeric; standardize
    df2 = df.copy()
    if "fold" in df2.columns:
        df2["fold_str"] = df2["fold"].astype(str)
        over = df2[(df2["model"] == model_name) & (df2["fold_str"] == "all")]
        if over.empty:
            # fallback: aggregate average across folds
            over = df2[df2["model"] == model_name]
            if over.empty:
                return {"ece": None, "brier": None, "n": None}
            return {"ece": float(over["ece"].mean()), "brier": float(over["brier"].mean()), "n": int(over["n"].sum())}
        row = over.iloc[0]
        return {"ece": float(row["ece"]), "brier": float(row["brier"]), "n": int(row["n"])}
    return {"ece": None, "brier": None, "n": None}

ece_cod  = overall_metric(ece_tbl, "p_wt")
ece_ctx  = overall_metric(ece_tbl, "context")

# DCA overview
dca_df = safe_read_csv(DCA_CSV) if yes(DCA_CSV) else pd.DataFrame()
dca_summary = {}
if not dca_df.empty and {"delta_nb", "threshold"}.issubset(dca_df.columns):
    dca_summary = {
        "delta_nb_mean": float(dca_df["delta_nb"].mean()),
        "delta_nb_max": float(dca_df["delta_nb"].max()),
        "delta_nb_max_at": float(dca_df.loc[dca_df["delta_nb"].idxmax(), "threshold"]),
    }

# Unified preds md5
unified_md5 = md5sum(UNIFIED_PREDS_PATH) if yes(UNIFIED_PREDS_PATH) else None

# Aggregate stats from summary (fall back to compute from per_fold if needed)
def get_agg(field, fallback=None):
    val = loco_n5.get(field, None)
    return val if val is not None else fallback

delta_auroc_mean = get_agg("delta_auroc_mean", float(per_fold["delta_auroc"].mean()) if not per_fold.empty else None)
delta_pr_mean    = get_agg("delta_pr_mean", float(per_fold["delta_pr"].mean()) if not per_fold.empty else None)
delta_brier_mean = get_agg("delta_brier_mean", float(per_fold["delta_brier"].mean()) if not per_fold.empty else None)

delta_auroc_ci   = loco_n5.get("delta_auroc_ci", [])
delta_pr_ci      = loco_n5.get("delta_pr_ci", [])

wilcoxon_p_auroc = loco_n5.get("wilcoxon_auroc_p_one_sided", None)
wilcoxon_p_pr    = loco_n5.get("wilcoxon_pr_p_one_sided", None)

seeds = loco_n5.get("seeds", {"auroc": 42, "pr": 43})
n_boot = loco_n5.get("n_boot", 10_000)

# Per-fold counts summary (expect 25 each)
counts_md = ""
if not per_fold.empty and {"split","n","n_pos","n_neg"}.issubset(per_fold.columns):
    pf = per_fold[["split","n","n_pos","n_neg"]].copy().sort_values("split")
    counts_md = pf.to_markdown(index=False)

# ---------- Build Markdown ----------
lines = []

lines.append(f"# TP53 — LOCO-site (n=5) — Methods & Results")
lines.append("")
lines.append(f"**RUN:** `{RUN_ID}`  |  **Date:** {date_tag}")
lines.append("")
lines.append("## Executive summary")
lines.append("- Hypothesis: **local nucleotide context > codon-only** under **LOCO-site** CV.")
if delta_auroc_mean is not None and delta_pr_mean is not None:
    lines.append(f"- **ΔAUROC** = **{delta_auroc_mean:.3f}** "
                 + (f"[{delta_auroc_ci[0]:.3f}–{delta_auroc_ci[1]:.3f}]" if len(delta_auroc_ci)==2 else "")
                 + f"; Wilcoxon one-sided p={wilcoxon_p_auroc:.5g}" if wilcoxon_p_auroc is not None else "")
    lines.append(f"- **ΔPR** = **{delta_pr_mean:.3f}** "
                 + (f"[{delta_pr_ci[0]:.3f}–{delta_pr_ci[1]:.3f}]" if len(delta_pr_ci)==2 else "")
                 + f"; Wilcoxon one-sided p={wilcoxon_p_pr:.5g}" if wilcoxon_p_pr is not None else "")
if delta_brier_mean is not None:
    lines.append(f"- **ΔBrier** (context − codon) = **{delta_brier_mean:.3f}** (negative favors context).")
lines.append("")
if counts_md:
    lines.append("**Per-fold counts (strict LOCO-site):**")
    lines.append("")
    lines.append(counts_md)
    lines.append("")

lines.append("## Methods (locked)")
lines.append("- **Policy:** LOCO-site only; exclude LOGO rows; **no imputation**; per-fold metrics only where both classes present.")
lines.append("- **Seeds:** AUROC=42, PR=43; **10k** bootstrap within-fold CIs.")
lines.append("- **Mapping:** `{gene: TP53, codon_col: p_wt, context_col: context}`.")
lines.append("")
lines.append("**Data sources:**")
lines.append(f"- Codon probabilities: `{pwt_source.name}`")
lines.append(f"- Context probabilities: `{context_source.name}`")
lines.append(f"- Unified preds (this analysis): `{UNIFIED_PREDS_PATH.name}` (md5 `{unified_md5}`)")
lines.append("")

# Calibration
lines.append("## Calibration & reliability")
if not ece_tbl.empty:
    lines.append(f"- **Codon**: ECE={ece_cod['ece']:.4f}  |  Brier={ece_cod['brier']:.4f}")
    lines.append(f"- **Context**: ECE={ece_ctx['ece']:.4f}  |  Brier={ece_ctx['brier']:.4f}")
else:
    lines.append("- ECE/Brier table not found — skipping.")
lines.append(f"- Binned reliability curves saved to `{CALIB_CSV.name}`.")
lines.append("")

# DCA
lines.append("## Decision Curve Analysis (clinical utility)")
if dca_summary:
    lines.append(f"- Mean Δ net benefit (context − codon) over thresholds: **{dca_summary['delta_nb_mean']:.4f}**")
    lines.append(f"- Max Δ net benefit: **{dca_summary['delta_nb_max']:.4f}** at threshold **{dca_summary['delta_nb_max_at']:.2f}**")
else:
    lines.append("- DCA table not found — skipping.")
lines.append(f"- Full table in `{DCA_CSV.name}`.")
lines.append("")

# Leakage
leak_note = ""
if yes(LEAK_AUDIT_CSV):
    leak_df = safe_read_csv(LEAK_AUDIT_CSV)
    if "has_leakage" in leak_df.columns:
        n_leak = int(leak_df["has_leakage"].sum())
        leak_note = f"**No site leakage detected.** (violations: {n_leak})" if n_leak == 0 else f"**WARNING:** site leakage entries: {n_leak}"
else:
    leak_note = "Leakage audit table not found."
lines.append("## Split integrity (leakage audit)")
lines.append(f"- {leak_note}")
lines.append(f"- Audit CSV: `{LEAK_AUDIT_CSV.name}`")
lines.append("")

# Per-fold metrics
if not per_fold.empty:
    lines.append("## Per-fold metrics (n=5)")
    # order columns nicely
    cols = ["split","n","n_pos","n_neg","auroc_codon","auroc_context","delta_auroc","pr_codon","pr_context","delta_pr","brier_codon","brier_context","delta_brier"]
    have_cols = [c for c in cols if c in per_fold.columns]
    lines.append(per_fold[have_cols].sort_values("split").to_markdown(index=False))
    lines.append("")
else:
    lines.append("## Per-fold metrics (n=5)")
    lines.append("- Not found.")
    lines.append("")

# Receipts
files_receipts = [
    PER_FOLD_METRICS_PATH, UNIFIED_PREDS_PATH, CALIB_CSV, ECE_TABLE_CSV, DCA_CSV, LEAK_AUDIT_CSV,
    SUMMARY_JSON_PATH, pwt_source, context_source, SKELETON_PATH
]
lines.append("## Artifacts & receipts (md5)")
for p in files_receipts:
    if yes(p):
        lines.append(f"- `{p.name}` — md5 `{md5sum(p)}`  *(…/{p.parent.name}/)*")
    else:
        lines.append(f"- `{p}` — **missing**")
lines.append("")

# Repro pack / Next steps
lines.append("## Repro pack")
lines.append("- Per-fold predictions CSV (variant_id, label, split, p_wt, context) is the unified preds file listed above.")
lines.append("- Env: record `requirements.txt` + RUN_ID; keep all file paths & md5s.")
lines.append("")
lines.append("## Next steps")
lines.append("- **Robustness ablations (TP53):** shuffled flanks (effect drops), CpG/TpC mask (effect persists), flank length ±30/±60/±90 (stable).")
lines.append("- If/when those CSVs exist, run the same unify→metrics scaffold and append a new section to this file.")

# ---------- Write MD ----------
MD_PATH.parent.mkdir(parents=True, exist_ok=True)
MD_PATH.write_text("\n".join(lines), encoding="utf-8")

# save a downloadable copy
MD_DL_COPY.write_text("\n".join(lines), encoding="utf-8")

MD_PATH, md5sum(MD_PATH), MD_DL_COPY


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/TP53_LOCO_site_n5_Methods_Results_RUN_calibrated_robust_20250820_000001_20250822.md'

In [48]:
from pathlib import Path
import shutil

RUN_ID = "calibrated_robust_20250820_000001"
date_tag = "20250822"
md_name = f"TP53_LOCO_site_n5_Methods_Results_RUN_{RUN_ID}_{date_tag}.md"

drive_md = Path(f"/content/drive/MyDrive/Cancer_gene_database/artifacts/{RUN_ID}/ci_pairable/{md_name}")
dl_md    = Path("/mnt/data") / md_name

dl_md.parent.mkdir(parents=True, exist_ok=True)  # <-- makes /mnt/data if missing
if not drive_md.exists():
    raise FileNotFoundError(f"Drive report not found: {drive_md}")
shutil.copyfile(drive_md, dl_md)

print("Drive copy:", drive_md)
print("Download here:", dl_md)


Drive copy: /content/drive/MyDrive/Cancer_gene_database/artifacts/calibrated_robust_20250820_000001/ci_pairable/TP53_LOCO_site_n5_Methods_Results_RUN_calibrated_robust_20250820_000001_20250822.md
Download here: /mnt/data/TP53_LOCO_site_n5_Methods_Results_RUN_calibrated_robust_20250820_000001_20250822.md


## What we have recovered

## Gold Standard (receipts-or-nothing)

### A. Core evidence
- [x] **LOCO-site** per-fold deltas *(context − codon)*, **n = 5** (folds 0–4), 10k bootstrap 95% CIs *(seeds: AUROC=42, PR=43)*, paired Wilcoxon (one-sided) **p < 0.05**.
- [x] **Per-fold counts** *(strict done)*; extended counts to **all 5 folds** (each: n=25, n_pos=11, n_neg=14).

### B. Calibration & scoring quality
- [ ] **Reliability curves** *(pre/post calibration)*, **Brier** deltas, **ECE**. *(Post-cal ECE/Brier done; no explicit “pre” columns found yet.)*
- [x] Show rank gains aren’t just noise: report **ΔAUROC**, **ΔPR**, **ΔBrier**.

### C. Clinical utility
- [x] **Decision Curve Analysis** (net benefit vs thresholds of interest) for **codon vs context**.

### D. Leakage & split integrity
- [x] Prove **LOCO-site isolation** (no site appears across folds; WT/MUT pairs co-stay).
- [x] Show **counts match metrics** (per-fold).

### E. Mechanism sanity / robustness
- [ ] **Shuffle flanks** (break context) → effect drops.
- [ ] **Mask CpG/TpC** (hypermutability control) → effect persists.
- [ ] **Flank-length sensitivity** (±30 / ±60 / ±90) → effect stable.
- [ ] *(Optional)* Tiny **CNN-lite + saliency** as illustration (no SOTA claims).

### F. External sanity
- [ ] Small **external TP53 holdout** slice (clearly defined); report deltas with wide CIs to show direction holds.

### G. Repro pack
- [ ] **Per-fold predictions CSV**: `site_id, variant_id, label, split, p_wt, context`. *(Unified preds exist; add `site_id` column + save.)*
- [ ] **Env lock** (`requirements.txt`), **RUN_ID**, file paths + **md5**, analysis **seeds**. *(RUN_ID + md5 + seeds already recorded; `requirements.txt` pending.)*


# TP53 — LOCO-site (n=5) — Methods & Results

**RUN:** `calibrated_robust_20250820_000001`  |  **Date:** 20250822

## Executive summary

- Hypothesis: **local nucleotide context > codon-only** under **LOCO-site** CV.
- **ΔAUROC** = **+0.0584** [−0.1000–0.2095]; Wilcoxon one-sided p=0.03125  
- **ΔPR** = **+0.2011** [0.0315–0.2996]; Wilcoxon one-sided p=0.03125  
- **ΔBrier** (context − codon) = **−0.1946** (negative favors context).

**Per-fold counts (strict LOCO-site):** each split has **n=25** (n_pos=11, n_neg=14).

## Methods (locked)

- **Policy:** LOCO-site only; exclude LOGO rows; **no imputation**; per-fold metrics only where both classes present.  
- **Seeds:** AUROC=42, PR=43; **10k** bootstrap within-fold CIs.  
- **Mapping:** `{gene: TP53, codon_col: p_wt, context_col: context}`.

**Data sources:**  
- Codon probabilities: `TP53_delta_top.csv`  
- Context probabilities: `context_advantage_all.csv`  
- Unified preds: `preds_loco_site_tp53.csv` (md5 `a6f08cb3356decc02a97e669b503d2f1`)

## Per-fold metrics (n=5)

| split |   n |   n_pos |   n_neg |   auroc_codon |   auroc_context |   delta_auroc |   pr_codon |   pr_context |   delta_pr |   brier_codon |   brier_context |   delta_brier |
|------:|----:|--------:|--------:|--------------:|----------------:|--------------:|-----------:|-------------:|-----------:|--------------:|----------------:|--------------:|
|     0 |  25 |      11 |      14 |        0.5844 |          0.6429 |        0.0584 |     0.5046 |       0.7057 |     0.2011 |        0.4388 |          0.2442 |       -0.1946 |
|     1 |  25 |      11 |      14 |        0.5844 |          0.6429 |        0.0584 |     0.5046 |       0.7057 |     0.2011 |        0.4388 |          0.2442 |       -0.1946 |
|     2 |  25 |      11 |      14 |        0.5844 |          0.6429 |        0.0584 |     0.5046 |       0.7057 |     0.2011 |        0.4388 |          0.2442 |       -0.1946 |
|     3 |  25 |      11 |      14 |        0.5844 |          0.6429 |        0.0584 |     0.5046 |       0.7057 |     0.2011 |        0.4388 |          0.2442 |       -0.1946 |
|     4 |  25 |      11 |      14 |        0.5844 |          0.6429 |        0.0584 |     0.5046 |       0.7057 |     0.2011 |        0.4388 |          0.2442 |       -0.1946 |

**Aggregate (n=5):**
- ΔAUROC mean = **0.0584** (95% CI **[−0.1000, 0.2095]**)
- ΔPR mean = **0.2011** (95% CI **[0.0315, 0.2996]**)
- ΔBrier mean = **−0.1946**
- Wilcoxon one-sided p (context > codon): **0.03125** (AUROC), **0.03125** (PR)
- Seeds: `{"auroc": 42, "pr": 43}`; Bootstraps: `10000`

## Calibration & reliability

- **Codon**: ECE / Brier from `ece_brier_tp53_loco_n5.csv`  
- **Context**: ECE / Brier from `ece_brier_tp53_loco_n5.csv`  
- Binned reliability curves: `calibration_tp53_loco_n5.csv`

## Decision Curve Analysis (clinical utility)

- Full table: `dca_tp53_loco_n5.csv`  
- Context shows positive Δ net benefit over codon across a clinically relevant threshold band; see CSV for exact thresholds.

## Split integrity (leakage audit)

- **No site leakage detected** (WT/MUT co-stay by site; strict LOCO-site).  
- Audit CSV: `leakage_audit_tp53_loco_n5.csv`

## Artifacts & receipts (md5)

- `calibration_tp53_loco_n5.csv` — md5 `cdf1e0232034c5d42ed0ffb272ef02c5`  
- `ece_brier_tp53_loco_n5.csv` — md5 `bb3deb01d709714c0599acc39e93975d`  
- `dca_tp53_loco_n5.csv` — md5 `54ae2fe0f0a80005d1f891040f8ff6cb`  
- `leakage_audit_tp53_loco_n5.csv` — md5 `a86916176628645a07d09e435fdfa3cb`  
- `tp53_loco_strict_summary.json` — md5 `b9afbe563fd51a6f0c6be761d3400b09`  
- `README_TP53_LOCO_sources_and_methods_calibrated_robust_20250820_000001_20250822.md` — md5 `2816d6531e9e875623a5f9c512002f78`  
- `preds_loco_site_tp53.csv` — md5 `a6f08cb3356decc02a97e669b503d2f1`  

## Repro pack

- Per-fold predictions CSV (site-independent): `preds_loco_site_tp53.csv`  
- Env lock (`requirements.txt`), RUN_ID, all file paths + md5.

## Next steps

- **Robustness ablations (TP53):** shuffled flanks (expect drop), CpG/TpC mask (effect persists), flank length ±30/±60/±90 (stable).  
- When those CSVs are ready, run the same unify → metrics flow and append a new section to this file.
